# Cell 1 — Installs and imports

In [1]:
# =============================================================================
# CELL 1 — INSTALLS AND IMPORTS
# =============================================================================
import subprocess
import sys

REQUIRED_PACKAGES = [
    "reportlab",
    "openpyxl",
    "Pillow"
]

for pkg in REQUIRED_PACKAGES:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    except Exception:
        pass

import os
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.offsetbox import AnchoredOffsetbox, HPacker, TextArea
from matplotlib.patches import FancyBboxPatch

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Installs and imports loaded.")

Installs and imports loaded.


# Cell 2 — Paths, folders, helpers, styles

In [2]:
# =============================================================================
# CELL 2 — PATHS, HELPERS, STYLE SYSTEM
# =============================================================================

from pathlib import Path
import os
import sys

# -----------------------------------------------------------------------------
# PROJECT ROOT DETECTION
# -----------------------------------------------------------------------------
# GitHub-ready path system:
# - Works after cloning the repository locally
# - Works when running from the repository root or /notebooks
# - Does not depend on Google Drive or hardcoded local paths

CURRENT = Path.cwd().resolve()

def find_project_root(start_path: Path) -> Path:
    """Find the repository root by walking upward from the current working directory."""
    start_path = Path(start_path).resolve()
    markers = ["README.md", ".git", "data", "reports", "notebooks"]

    for path in [start_path] + list(start_path.parents):
        if any((path / marker).exists() for marker in markers):
            if path.name.lower() == "notebooks":
                return path.parent.resolve()
            return path.resolve()

    return start_path

BASE_DIR = find_project_root(CURRENT)

if BASE_DIR.name.lower() == "notebooks":
    BASE_DIR = BASE_DIR.parent.resolve()

# -----------------------------------------------------------------------------
# STANDARD PROJECT FOLDERS
# -----------------------------------------------------------------------------

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "processed"
LEGACY_CLEAN_DIR = DATA_DIR / "clean"

REPORTS_DIR = BASE_DIR / "reports"
IMG_DIR = REPORTS_DIR / "report_images"
NOTEBOOKS_DIR = BASE_DIR / "notebooks"

PDF_OUTPUT = REPORTS_DIR / "canada_brazil_trade_report.pdf"

for d in [DATA_DIR, RAW_DIR, CLEAN_DIR, LEGACY_CLEAN_DIR, REPORTS_DIR, IMG_DIR, NOTEBOOKS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root  : {BASE_DIR}")
print(f"Data folder   : {DATA_DIR}")
print(f"Raw data      : {RAW_DIR}")
print(f"Processed data: {CLEAN_DIR}")
print(f"Clean fallback: {LEGACY_CLEAN_DIR}")
print(f"Reports folder: {REPORTS_DIR}")
print(f"Image folder  : {IMG_DIR}")
print(f"Notebook cwd  : {Path.cwd().resolve()}")

def walk_find_file(base_paths, filename_contains=None, exact_names=None, suffix=None):
    base_paths = [Path(p) for p in base_paths if Path(p).exists()]
    exact_names_lower = [x.lower() for x in exact_names] if exact_names else None

    for base in base_paths:
        for root, _, files in os.walk(base):
            for f in files:
                f_low = f.lower()
                if exact_names_lower and f_low in exact_names_lower:
                    return Path(root) / f
                if filename_contains and filename_contains.lower() in f_low:
                    if suffix is None or f_low.endswith(suffix.lower()):
                        return Path(root) / f
    return None

def fmt_cad(v, d=1):
    if v is None or pd.isna(v):
        return "—"
    v = float(v)
    if abs(v) >= 1e9:
        return f"CA${v/1e9:.{d}f} BI"
    if abs(v) >= 1e6:
        return f"CA${v/1e6:.{d}f}M"
    if abs(v) >= 1e3:
        return f"CA${v/1e3:.{d}f}K"
    return f"CA${v:,.0f}"

def fmt_pct(v, d=1):
    if v is None or pd.isna(v):
        return "—"
    return f"{v:+.{d}f}%"

def safe_pct_change(new, old):
    if old is None or old == 0 or pd.isna(old):
        return np.nan
    return ((new / old) - 1) * 100

def save_fig(fig, name):
    path = IMG_DIR / f"{name}.png"
    # Save at fixed canvas size to maintain consistent font rendering across charts.
    fig.savefig(path, dpi=190, bbox_inches=None, facecolor="white")
    return path

def safe_label(text, max_len=36):
    text = str(text)
    return text if len(text) <= max_len else text[:max_len - 1] + "…"

def add_text_subtitle(fig, parts, anchor=(0.5, 0.92), fontsize=11):
    children = [
        TextArea(
            txt,
            textprops=dict(color=col, fontsize=fontsize, fontweight=weight, fontstyle="italic")
        )
        for txt, col, weight in parts
    ]
    box = HPacker(children=children, align="baseline", pad=0, sep=2)
    fig.add_artist(
        AnchoredOffsetbox(
            loc="upper center",
            child=box,
            pad=0,
            frameon=False,
            bbox_to_anchor=anchor,
            bbox_transform=fig.transFigure,
            borderpad=0
        )
    )

def add_rounded_figure_container(fig, edgecolor="#ECECEC", linewidth=1.0,
                                 rounding_size=0.018, pad=0.008, facecolor="white"):
    bg = FancyBboxPatch(
        (pad, pad), 1 - 2 * pad, 1 - 2 * pad,
        transform=fig.transFigure,
        boxstyle=f"round,pad=0.012,rounding_size={rounding_size}",
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=linewidth,
        zorder=-10
    )
    fig.patches.append(bg)

def clean_axes(ax, keep_left_spine=False, keep_y=False):
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)
    if keep_left_spine:
        ax.spines["left"].set_visible(True)
        ax.spines["left"].set_color(C_DKGRAY)
    ax.spines["bottom"].set_visible(True)
    ax.spines["bottom"].set_color(C_DKGRAY)
    if not keep_y:
        ax.yaxis.set_visible(False)
    ax.xaxis.set_ticks_position("none")
    ax.tick_params(axis="x", labelsize=9, colors=C_DKGRAY)
    ax.set_facecolor(C_WHITE)

C_DKGREEN  = "#004D25"
C_LTGREEN  = "#99CC33"
C_RED      = "#E62310"
C_YELLOW   = "#FFCC22"
C_GRAY     = "#CCCCCC"
C_DKGRAY   = "#555555"
C_WHITE    = "#FFFFFF"
C_ROWALT   = "#F5F5F5"

plt.rcParams["figure.facecolor"] = C_WHITE
plt.rcParams["axes.facecolor"] = C_WHITE
plt.rcParams["savefig.facecolor"] = C_WHITE
plt.rcParams["font.family"] = "DejaVu Sans"

print("Helpers and style system ready.")

Project root  : /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report
Data folder   : /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report/data
Raw data      : /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report/data/raw
Processed data: /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report/data/processed
Clean fallback: /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report/data/clean
Reports folder: /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report/reports
Image folder  : /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report/reports/report_images
Notebook cwd  : /Users/Julio/Library/Mobile Documents/c

# Cell 3 — Load and standardize data

In [3]:
# =============================================================================
# CELL 3 — LOAD AND STANDARDIZE DATA
# =============================================================================
dataset_path = walk_find_file(
    [BASE_DIR, DATA_DIR, CLEAN_DIR, LEGACY_CLEAN_DIR, RAW_DIR, Path.cwd(), Path.cwd().parent],
    exact_names=["dataset_clean.csv", "clean_dataset.csv", "final_dataset.csv"]
)

if dataset_path is None:
    dataset_path = walk_find_file(
        [BASE_DIR, DATA_DIR, CLEAN_DIR, LEGACY_CLEAN_DIR, RAW_DIR, Path.cwd(), Path.cwd().parent],
        filename_contains="clean",
        suffix=".csv"
    )

if dataset_path is None:
    raise FileNotFoundError(
        "dataset_clean.csv not found.\n"
        "Expected example path:\n"
        "data/processed/dataset_clean.csv or data/clean/dataset_clean.csv"
    )

hs_map_path = walk_find_file(
    [BASE_DIR, DATA_DIR, CLEAN_DIR, LEGACY_CLEAN_DIR, RAW_DIR, Path.cwd(), Path.cwd().parent],
    exact_names=["hs_mapping.xlsx"]
)

print(f"Loading dataset: {dataset_path}")
if hs_map_path:
    print(f"Loading HS map : {hs_map_path}")
else:
    print("hs_mapping.xlsx not found — using fallback chapter labels.")

df = pd.read_csv(dataset_path)

required_cols = ["flow", "value"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns in dataset: {missing_required}")

if "period_m" in df.columns:
    df["Period"] = pd.to_datetime(df["period_m"], errors="coerce")
elif "period" in df.columns:
    df["Period"] = pd.to_datetime(df["period"], errors="coerce")
elif "date" in df.columns:
    df["Period"] = pd.to_datetime(df["date"], errors="coerce")
else:
    raise ValueError("Missing a usable date column: period_m / period / date")

df = df.dropna(subset=["Period"]).copy()

if "year" in df.columns:
    df["Year"] = pd.to_numeric(df["year"], errors="coerce").fillna(df["Period"].dt.year).astype(int)
else:
    df["Year"] = df["Period"].dt.year.astype(int)

if "province" in df.columns:
    df["Province"] = df["province"].astype(str).str.strip()
else:
    df["Province"] = "Unknown"

df["Month"] = df["Period"].dt.to_period("M")
df["Month_Num"] = df["Period"].dt.month
df["Value"] = pd.to_numeric(df["value"], errors="coerce").fillna(0)
df["flow"] = df["flow"].astype(str).str.strip().str.lower()

if "hs8" in df.columns:
    hs8_num = pd.to_numeric(df["hs8"], errors="coerce")
    df = df[~hs8_num.isin([98, 99])].copy()

if "hs2" in df.columns:
    hs2_num = pd.to_numeric(df["hs2"], errors="coerce")
    df = df[(hs2_num < 97) | (hs2_num.isna())].copy()

if "is_ch98" in df.columns:
    df = df[df["is_ch98"] != True].copy()

df["flow"] = df["flow"].replace({
    "export": "export",
    "exports": "export",
    "import": "import",
    "imports": "import"
})

# -----------------------------------------------------------------------------
# HS / product labels
# -----------------------------------------------------------------------------
# Prefer the already-cleaned dataset fields when available. This is important for
# the market intelligence layer because the cleaned file already includes
# section_name and chapter_name, which are more business-readable than raw HS codes.
if "section_name" in df.columns:
    df["Section_Name"] = df["section_name"].astype(str).str.strip().replace("", "Unknown")
elif "section" in df.columns:
    df["Section_Name"] = df["section"].astype(str).str.strip().replace("", "Unknown")
else:
    df["Section_Name"] = "Unknown"

if "chapter_name" in df.columns:
    df["Chapter_Name"] = df["chapter_name"].astype(str).str.strip().replace("", "Unknown")
elif "description" in df.columns:
    df["Chapter_Name"] = df["description"].astype(str).str.strip().replace("", "Unknown")
else:
    df["Chapter_Name"] = "Unknown"

# Optional HS mapping fallback. Use HS2 as the chapter key. Do not overwrite
# cleaned chapter_name values unless the current value is missing/unknown.
if hs_map_path is not None:
    try:
        df_hs = pd.read_excel(hs_map_path)
        df_hs.columns = [str(c).strip() for c in df_hs.columns]

        if len(df_hs.columns) >= 4:
            df_hs = df_hs.iloc[:, :4].copy()
            df_hs.columns = ["Section", "Section_Name_Map", "Chapter", "Chapter_Name_Map"]
            df_hs["Chapter"] = (
                df_hs["Chapter"].astype(str).str.extract(r"(\d+)")[0].str.zfill(2)
            )
            chapter_lkp = df_hs.drop_duplicates("Chapter").set_index("Chapter")["Chapter_Name_Map"].to_dict()
            section_lkp = df_hs.drop_duplicates("Chapter").set_index("Chapter")["Section_Name_Map"].to_dict()

            if "hs2" in df.columns:
                df["hs_chapter"] = pd.to_numeric(df["hs2"], errors="coerce").astype("Int64").astype(str).str.zfill(2)
            elif "hs8" in df.columns:
                df["hs_chapter"] = df["hs8"].astype(str).str.extract(r"(\d{2})")[0].str.zfill(2)
            else:
                df["hs_chapter"] = pd.NA

            mapped_chapter = df["hs_chapter"].map(chapter_lkp)
            mapped_section = df["hs_chapter"].map(section_lkp)

            missing_chapter = df["Chapter_Name"].isin(["", "Unknown", "Other", "nan", "None"]) | df["Chapter_Name"].isna()
            missing_section = df["Section_Name"].isin(["", "Unknown", "Other", "nan", "None"]) | df["Section_Name"].isna()
            df.loc[missing_chapter, "Chapter_Name"] = mapped_chapter[missing_chapter].fillna(df.loc[missing_chapter, "Chapter_Name"])
            df.loc[missing_section, "Section_Name"] = mapped_section[missing_section].fillna(df.loc[missing_section, "Section_Name"])
    except Exception as e:
        print(f"HS mapping fallback skipped: {e}")

# Final readable fallbacks.
df["Section_Name"] = df["Section_Name"].fillna("Unknown").replace("", "Unknown")
df["Chapter_Name"] = df["Chapter_Name"].fillna("Unknown").replace("", "Unknown")

print("Rows:", len(df))
display(df.head())

Loading dataset: /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report/data/clean/dataset_clean.csv
hs_mapping.xlsx not found — using fallback chapter labels.
Rows: 89083


,period_m,year,month,quarter,flow,province,country,description,hs2,hs4,hs6,hs8,hs10,section,section_name,chapter_name,value,quantity,uom_text,uom_code,uom_status,has_quantity,is_ch98,is_suppressed,flag_multi_uom,flag_non_ascii,is_outlier,median_value_hs8,lower_bound_hs8,upper_bound_hs8,Period,Year,Province,Month,Month_Num,Value,Section_Name,Chapter_Name
0,2025-01,2025,1,1,import,Newfoundland and Labrador,Brazil,"Plates, sheets and strip, of vulcanized cellul...",40,4008,400811,40081100,4008110000,VII,Plastics and Rubber,Rubber and Articles Thereof,173,10,Weight in kilograms,KGM,OK,True,False,False,True,False,False,43.25,-52.73,148.55,2025-01-01,2025,Newfoundland and Labrador,2025-01,1,173,Plastics and Rubber,Rubber and Articles Thereof
1,2024-02,2024,2,1,import,Newfoundland and Labrador,Brazil,"Other AC motors, multi-phase, of an output exc...",85,8501,850153,85015300,8501530060,XVI,Machinery and Electrical Equipment,Electrical Machinery and Equipment,2558388,2,Number,NMB,OK,True,False,False,False,False,True,"28,051.00","-90,902.00","187,490.00",2024-02-01,2024,Newfoundland and Labrador,2024-02,2,2558388,Machinery and Electrical Equipment,Electrical Machinery and Equipment
2,2025-03,2025,3,1,import,Newfoundland and Labrador,Brazil,"Parts, of hand tools, nes",84,8467,846799,84679900,8467990000,XVI,Machinery and Electrical Equipment,Nuclear Reactors and Machinery,46,0,Blank,BLANK,OK,False,False,False,False,False,False,NaN,NaN,NaN,2025-03-01,2025,Newfoundland and Labrador,2025-03,3,46,Machinery and Electrical Equipment,Nuclear Reactors and Machinery
3,2025-10,2025,10,4,import,Newfoundland and Labrador,Brazil,"Screws, whether/not with their nuts or washers...",73,7318,731815,73181500,7318150010,XV,Base Metals and Articles Thereof,Articles of Iron and Steel,8,0,Weight in kilograms,KGM,Error,False,False,False,True,False,False,8.03,-19.49,44.49,2025-10-01,2025,Newfoundland and Labrador,2025-10,10,8,Base Metals and Articles Thereof,Articles of Iron and Steel
4,2025-11,2025,11,4,import,Newfoundland and Labrador,Brazil,"Other AC motors, single-phase, power >750 W bu...",85,8501,850140,85014000,8501400040,XVI,Machinery and Electrical Equipment,Electrical Machinery and Equipment,1209,1,Number,NMB,OK,True,False,False,False,False,True,354.43,-449.16,"1,179.65",2025-11-01,2025,Newfoundland and Labrador,2025-11,11,1209,Machinery and Electrical Equipment,Electrical Machinery and Equipment


# Cell 4 — Aggregations, KPIs, forecast, summary

In [4]:
# =============================================================================
# CELL 4 — CORE AGGREGATIONS, KPI TABLES, FORECAST, SUMMARY
# =============================================================================

available_years = sorted(df["Year"].dropna().unique().tolist())
if len(available_years) < 2:
    raise ValueError("At least two years are required to compare performance.")

YEAR1 = available_years[-2]
YEAR2 = available_years[-1]

# -----------------------------------------------------------------------------
# Annual trade
# -----------------------------------------------------------------------------
ann = df.groupby(["Year", "flow"])["Value"].sum().unstack(fill_value=0)
for col in ["export", "import"]:
    if col not in ann.columns:
        ann[col] = 0

ann["total"] = ann["export"] + ann["import"]
ann["balance"] = ann["export"] - ann["import"]

total_2024 = ann.loc[YEAR1, "total"]
total_2025 = ann.loc[YEAR2, "total"]
exp_2024 = ann.loc[YEAR1, "export"]
exp_2025 = ann.loc[YEAR2, "export"]
imp_2024 = ann.loc[YEAR1, "import"]
imp_2025 = ann.loc[YEAR2, "import"]

growth_pct = safe_pct_change(total_2025, total_2024)
export_growth_pct = safe_pct_change(exp_2025, exp_2024)
import_growth_pct = safe_pct_change(imp_2025, imp_2024)
balance_2025 = exp_2025 - imp_2025

# -----------------------------------------------------------------------------
# Monthly totals
# -----------------------------------------------------------------------------
monthly = (
    df.groupby("Month")["Value"]
    .sum()
    .reset_index()
    .rename(columns={"Value": "total"})
)
monthly["Period"] = monthly["Month"].dt.to_timestamp()
monthly["Year"] = monthly["Period"].dt.year
monthly = monthly.sort_values("Period")

# -----------------------------------------------------------------------------
# Monthly flows
# -----------------------------------------------------------------------------
monthly_flow = (
    df.groupby(["Period", "flow"])["Value"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={"export": "exports", "import": "imports"})
)

if "exports" not in monthly_flow.columns:
    monthly_flow["exports"] = 0
if "imports" not in monthly_flow.columns:
    monthly_flow["imports"] = 0

monthly_flow["balance"] = monthly_flow["exports"] - monthly_flow["imports"]
monthly_flow = monthly_flow.sort_values("Period")

# -----------------------------------------------------------------------------
# Province table
# -----------------------------------------------------------------------------
prov = (
    df.groupby(["Year", "Province"])["Value"]
    .sum()
    .reset_index()
    .pivot(index="Province", columns="Year", values="Value")
    .fillna(0)
)

if YEAR1 not in prov.columns:
    prov[YEAR1] = 0
if YEAR2 not in prov.columns:
    prov[YEAR2] = 0

prov = prov.rename(columns={YEAR1: "v2024", YEAR2: "v2025"})
prov["growth_pct"] = (prov["v2025"] - prov["v2024"]) / prov["v2024"].replace(0, np.nan) * 100
prov["change_abs"] = prov["v2025"] - prov["v2024"]
prov["contribution"] = np.where(
    prov["change_abs"].sum() != 0,
    prov["change_abs"] / prov["change_abs"].sum() * 100,
    0
)
prov["share_2025"] = np.where(
    prov["v2025"].sum() != 0,
    prov["v2025"] / prov["v2025"].sum() * 100,
    0
)
prov = prov.reset_index()

# -----------------------------------------------------------------------------
# Chapter / product table
# -----------------------------------------------------------------------------
chapter_col = "Chapter_Name" if "Chapter_Name" in df.columns else (
    "chapter_name" if "chapter_name" in df.columns else None
)

if chapter_col is not None:
    chap = (
        df.groupby(["Year", chapter_col])["Value"]
        .sum()
        .reset_index()
        .pivot(index=chapter_col, columns="Year", values="Value")
        .fillna(0)
    )

    if YEAR1 not in chap.columns:
        chap[YEAR1] = 0
    if YEAR2 not in chap.columns:
        chap[YEAR2] = 0

    chap = chap.rename(columns={YEAR1: "v2024", YEAR2: "v2025"})
    chap["growth_pct"] = (chap["v2025"] - chap["v2024"]) / chap["v2024"].replace(0, np.nan) * 100
    chap["change_abs"] = chap["v2025"] - chap["v2024"]
    chap["share_2025"] = np.where(
        chap["v2025"].sum() != 0,
        chap["v2025"] / chap["v2025"].sum() * 100,
        0
    )
    chap = chap.reset_index().rename(columns={chapter_col: "Chapter_Name"})
else:
    chap = pd.DataFrame(
        columns=["Chapter_Name", "v2024", "v2025", "growth_pct", "change_abs", "share_2025"]
    )

MIN_BASELINE_THRESHOLD = 5_000_000
chap_filt = chap[
    (chap["v2024"] >= MIN_BASELINE_THRESHOLD) | (chap["v2025"] >= MIN_BASELINE_THRESHOLD)
].copy()
chap_filt = chap_filt.sort_values("v2025", ascending=False)

# -----------------------------------------------------------------------------
# Forecast baseline
# -----------------------------------------------------------------------------
monthly_total = monthly[monthly["Year"].isin([YEAR1, YEAR2])].copy()
monthly_total["Month_Num"] = monthly_total["Period"].dt.month

seasonality = (
    monthly_total[monthly_total["Year"] == YEAR2]
    .groupby("Month_Num")["total"]
    .sum()
    .reindex(range(1, 13), fill_value=0)
)

if seasonality.sum() == 0:
    seasonality = pd.Series([1] * 12, index=range(1, 13))

seasonality_share = seasonality / seasonality.sum()

forecast_growth = safe_pct_change(total_2025, total_2024)
forecast_total_2026 = total_2025 * (1 + (0 if pd.isna(forecast_growth) else forecast_growth / 100))

forecast_monthly = pd.DataFrame({
    "Month_Num": range(1, 13),
    "Forecast_Value": (seasonality_share.values * forecast_total_2026)
})
forecast_monthly["Month"] = pd.to_datetime(
    [f"{YEAR2 + 1}-{m:02d}-01" for m in forecast_monthly["Month_Num"]]
)

# -----------------------------------------------------------------------------
# Summary helpers
# -----------------------------------------------------------------------------
top_prov_for_summary = prov.sort_values("change_abs", ascending=False).iloc[0] if len(prov) else None
top_chap_for_summary = chap_filt.sort_values("change_abs", ascending=False).iloc[0] if len(chap_filt) else None

top_prov_text = ""
if top_prov_for_summary is not None:
    top_prov_text = f"The largest provincial contribution came from {top_prov_for_summary['Province']}. "

top_chap_text = ""
if top_chap_for_summary is not None:
    top_chap_text = f"The strongest product chapter by absolute change was {top_chap_for_summary['Chapter_Name']}. "

executive_summary_auto = (
    f"Canada-Brazil merchandise trade reached {fmt_cad(total_2025)} in {YEAR2}, "
    f"{fmt_pct(growth_pct)} versus {YEAR1}. Exports totaled {fmt_cad(exp_2025)} while "
    f"imports reached {fmt_cad(imp_2025)}, leaving a trade balance of {fmt_cad(balance_2025)}. "
    f"{top_prov_text}{top_chap_text}"
    f"The baseline outlook for {YEAR2 + 1} implies total trade near {fmt_cad(forecast_total_2026)} if current momentum persists."
)

title_page_title = "Canada-Brazil Trade Opportunities Report"
title_page_subtitle = (
    f"Executive review of Canada-Brazil merchandise trade performance in {YEAR1} and {YEAR2}, "
    f"covering annual trends, trade mix, product momentum, provincial concentration, and baseline outlook."
)

contents_items = [
    "1. Executive Summary",
    "2. Annual Trade Overview",
    "3. Monthly Trade Flow",
    "4. Comparative Trade Mix",
    "5. Diverging Product Momentum",
    "6. Provincial Trade Volume",
    "7. Provincial Growth and Decline",
    "8. Strategic Provincial Investment Map",
    "9. Ontario Trade Hub Analysis",
    "10. Forecast Validation and Baseline Outlook",
]

print(f"Years used: {YEAR1} and {YEAR2}")
print(f"Total trade {YEAR2}: {fmt_cad(total_2025)}")
print(f"Exports {YEAR2}: {fmt_cad(exp_2025)}")
print(f"Imports {YEAR2}: {fmt_cad(imp_2025)}")
print(f"Trade balance {YEAR2}: {fmt_cad(balance_2025)}")
print(f"Forecast {YEAR2+1}: {fmt_cad(forecast_total_2026)}")
print()
print(executive_summary_auto)

# -----------------------------------------------------------------------------
# Market Intelligence Layer: Business Opportunities
# -----------------------------------------------------------------------------
# Uses the dataset fields already available in dataset_clean.csv:
# section_name / Section_Name, chapter_name / Chapter_Name, flow, province, year,
# value, and HS codes. This avoids recleaning the data and keeps the business
# layer traceable to Statistics Canada HS classifications.

# Ensure market fields exist even if this cell is run independently.
if "Section_Name" not in df.columns:
    if "section_name" in df.columns:
        df["Section_Name"] = df["section_name"].astype(str).str.strip().replace("", "Unknown")
    elif "section" in df.columns:
        df["Section_Name"] = df["section"].astype(str).str.strip().replace("", "Unknown")
    else:
        df["Section_Name"] = "Unknown"

if "Chapter_Name" not in df.columns:
    if "chapter_name" in df.columns:
        df["Chapter_Name"] = df["chapter_name"].astype(str).str.strip().replace("", "Unknown")
    elif "description" in df.columns:
        df["Chapter_Name"] = df["description"].astype(str).str.strip().replace("", "Unknown")
    else:
        df["Chapter_Name"] = "Unknown"

def clean_market_label(x):
    x = str(x).strip()
    if x.lower() in ["", "nan", "none", "unknown"]:
        return "Other / Unclassified"
    return x

# Use HS section as the defensible sector layer. It is broad enough for executives
# but still grounded in the official trade classification.
df["Business_Sector"] = df["Section_Name"].apply(clean_market_label)
df["Business_Chapter"] = df["Chapter_Name"].apply(clean_market_label)

# Friendly sector labels for business audiences, while preserving source logic.
sector_business_labels = {
    "Live Animals and Animal Products": "Food & Agriculture",
    "Vegetable Products": "Food & Agriculture",
    "Animal or Vegetable Fats and Oils": "Food & Agriculture",
    "Prepared Foodstuffs": "Food & Agriculture",
    "Mineral Products": "Mining & Energy",
    "Products of the Chemical or Allied Industries": "Chemicals & Agriculture Inputs",
    "Plastics and Rubber": "Plastics & Rubber",
    "Raw Hides, Skins, Leather and Furskins": "Consumer Goods & Materials",
    "Wood and Articles of Wood": "Forestry & Wood Products",
    "Pulp of Wood, Paper and Paperboard": "Forestry & Paper",
    "Textiles and Textile Articles": "Textiles & Apparel",
    "Footwear, Headgear and Accessories": "Consumer Goods",
    "Stone, Cement, Glass and Ceramics": "Construction Materials",
    "Natural or Cultured Pearls, Precious Stones and Metals": "Mining, Metals & Jewelry",
    "Base Metals and Articles Thereof": "Metals & Industrial Materials",
    "Machinery and Electrical Equipment": "Machinery & Electrical Equipment",
    "Vehicles, Aircraft and Transport Equipment": "Transport & Aerospace",
    "Optical, Medical and Precision Instruments": "Precision & Medical Instruments",
    "Miscellaneous Manufactured Articles": "Consumer & Manufactured Goods",
    "Works of Art and Antiques": "Arts & Specialty Goods",
}

df["Business_Sector"] = df["Business_Sector"].replace(sector_business_labels)

def opportunity_level_from_values(growth, change, value_2025):
    growth = 0 if pd.isna(growth) else growth
    change = 0 if pd.isna(change) else change
    value_2025 = 0 if pd.isna(value_2025) else value_2025
    if growth >= 25 and change > 0 and value_2025 >= 10_000_000:
        return "High"
    if growth >= 10 and change > 0:
        return "Medium-High"
    if change > 0:
        return "Medium"
    return "Watch"

def build_yoy_table(source_df, group_col, value_col="Value", top_n=6, sort_mode="growth"):
    grouped = (
        source_df.groupby([group_col, "Year"], as_index=False)[value_col]
        .sum()
        .pivot(index=group_col, columns="Year", values=value_col)
        .fillna(0)
    )
    for _yr in [YEAR1, YEAR2]:
        if _yr not in grouped.columns:
            grouped[_yr] = 0
    grouped["YoY Growth"] = grouped.apply(lambda r: safe_pct_change(r[YEAR2], r[YEAR1]), axis=1)
    grouped["Absolute Change"] = grouped[YEAR2] - grouped[YEAR1]
    grouped = grouped.reset_index()
    grouped["Opportunity_Level"] = grouped.apply(
        lambda r: opportunity_level_from_values(r["YoY Growth"], r["Absolute Change"], r[YEAR2]), axis=1
    )
    grouped = grouped[grouped[YEAR2] > 0].copy()
    if sort_mode == "growth":
        grouped = grouped.sort_values(["YoY Growth", "Absolute Change", YEAR2], ascending=[False, False, False])
    else:
        grouped = grouped.sort_values(["Absolute Change", YEAR2, "YoY Growth"], ascending=[False, False, False])
    return grouped.head(top_n)

# 1) Which sectors are growing most rapidly?
fastest_growing_sectors = build_yoy_table(df, "Business_Sector", top_n=6, sort_mode="growth")

# 2) Which sectors are creating the largest absolute opportunity?
top_sector_opportunities = build_yoy_table(df, "Business_Sector", top_n=6, sort_mode="absolute")

# 3) Which Canadian sectors are buying more from Brazil? Imports only.
df_imports = df[df["flow"].eq("import")].copy()
top_canadian_demand_sectors = build_yoy_table(df_imports, "Business_Sector", top_n=6, sort_mode="absolute")

# 4) Product-level opportunity bridge: top chapters by growth and by import demand.
top_product_opportunities = build_yoy_table(df, "Business_Chapter", top_n=8, sort_mode="growth")
top_import_chapter_opportunities = build_yoy_table(df_imports, "Business_Chapter", top_n=8, sort_mode="absolute")

# Bridge between sector and chapters for PDF narratives.
product_sector_bridge = (
    df.groupby(["Business_Sector", "Business_Chapter"], as_index=False)["Value"]
    .sum()
    .sort_values(["Business_Sector", "Value"], ascending=[True, False])
)
product_sector_bridge = product_sector_bridge.groupby("Business_Sector").head(3)

def _top_item_text(table, label_col):
    if table is None or len(table) == 0:
        return "Selected categories"
    row = table.iloc[0]
    return f"{row[label_col]} ({fmt_pct(row['YoY Growth'])})"

market_intelligence_summary = (
    "The market intelligence layer translates the existing HS trade structure into business-oriented opportunities. "
    "Using section_name as the sector layer and chapter_name as the product opportunity layer, the report identifies where trade is growing fastest, "
    "where Canadian import demand from Brazil is increasing, and which industries should monitor these opportunities. "
    f"Based on the current dataset, the fastest-growing sector is {_top_item_text(fastest_growing_sectors, 'Business_Sector')}, "
    f"while the strongest product-level momentum is {_top_item_text(top_product_opportunities, 'Business_Chapter')}."
)

print("Market Intelligence Layer prepared from section_name and chapter_name:")
print("Fastest-growing sectors")
display(fastest_growing_sectors)
print("Canadian demand sectors: imports from Brazil")
display(top_canadian_demand_sectors)
print("Top product-level opportunities")
display(top_product_opportunities)


Years used: 2024 and 2025
Total trade 2025: CA$14.5 BI
Exports 2025: CA$2.9 BI
Imports 2025: CA$11.6 BI
Trade balance 2025: CA$-8.7 BI
Forecast 2026: CA$16.8 BI

Canada-Brazil merchandise trade reached CA$14.5 BI in 2025, +16.0% versus 2024. Exports totaled CA$2.9 BI while imports reached CA$11.6 BI, leaving a trade balance of CA$-8.7 BI. The largest provincial contribution came from Ontario. The strongest product chapter by absolute change was Precious Stones, Metals and Jewelry. The baseline outlook for 2026 implies total trade near CA$16.8 BI if current momentum persists.
Market Intelligence Layer prepared from section_name and chapter_name:
Fastest-growing sectors


Year,Business_Sector,2024,2025,YoY Growth,Absolute Change,Opportunity_Level
10,Precious Metals and Jewelry,2681922530,4323740138,61.22,1641817608,High
3,Food & Agriculture,702606909,949693843,35.17,247086934,High
15,"Stone, Ceramics and Glass",46872672,62602119,33.56,15729447,High
16,Textiles & Apparel,12046551,14674258,21.81,2627707,Medium-High
9,Plastics & Rubber,149808027,180344164,20.38,30536137,Medium-High
13,Products of the chemical or allied industries,3629069191,4298126721,18.44,669057530,Medium-High


Canadian demand sectors: imports from Brazil


Year,Business_Sector,2024,2025,YoY Growth,Absolute Change,Opportunity_Level
10,Precious Metals and Jewelry,2681777721,4323512529,61.22,1641734808,High
3,Food & Agriculture,606929598,875311283,44.22,268381685,High
6,Machinery & Electrical Equipment,743878850,901661767,21.21,157782917,Medium-High
13,Products of the chemical or allied industries,2439028734,2552269444,4.64,113240710,Medium
9,Plastics & Rubber,102987878,125379953,21.74,22392075,Medium-High
15,"Stone, Ceramics and Glass",35159416,50499974,43.63,15340558,High


Top product-level opportunities


Year,Business_Chapter,2024,2025,YoY Growth,Absolute Change,Opportunity_Level
23,"Feathers, Artificial Flowers and Hair",1123,5182,361.44,4059,Medium-High
43,Manufactures of Straw and Basketware,2451,8963,265.69,6512,Medium-High
12,Cocoa and Cocoa Preparations,32723918,102179892,212.25,69455974,High
37,"Lac, Gums and Resins",3092217,8369648,170.67,5277431,Medium-High
62,Other Base Metals and Cermets,606829,1635560,169.53,1028731,Medium-High
53,Musical Instruments,192114,491794,155.99,299680,Medium-High
65,Other Vegetable Textile Fibres,20470,48992,139.34,28522,Medium-High
63,Other Made Up Textile Articles,952054,2240985,135.38,1288931,Medium-High


# Cell 5 — Chart functions

In [5]:
# =============================================================================
# FINAL CELLS 5 AND 6: final readable chart sizing and page 15 validation table
# =============================================================================

# =============================================================================
# CELL 5: FINAL CHART FUNCTIONS (FULLY STANDARDIZED EXECUTIVE CHART TITLES/SUBTITLES)
# =============================================================================

from matplotlib.offsetbox import TextArea, HPacker, AnchoredOffsetbox
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import unicodedata

# Prophet is optional: chart_q9 will gracefully fall back if unavailable
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except Exception:
    PROPHET_AVAILABLE = False

# Ensure color variables are defined
if 'C_WHITE' not in globals():
    C_WHITE = '#FFFFFF'
    C_DKGREEN = '#004D25'
    C_LTGREEN = '#8DB600'
    C_RED = '#D32F2F'
    C_DKGRAY = '#333333'
    C_GRAY = '#888888'
    C_YELLOW = '#FFC107'

# -----------------------------------------------------------------------------
# Executive chart style standards
# -----------------------------------------------------------------------------
CHART_TITLE_Y = 0.950
CHART_SUBTITLE_Y = 0.880
CHART_TITLE_REFERENCE_WIDTH = 14.0
CHART_TITLE_REFERENCE_HEIGHT = 5.5
CHART_TITLE_SIZE = 22.0
CHART_SUBTITLE_SIZE = 12.5
CHART_LEFT = 0.08
CHART_RIGHT = 0.96
CHART_TOP = 0.745
CHART_BOTTOM = 0.145

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.labelcolor": C_DKGRAY,
    "xtick.color": C_DKGRAY,
    "ytick.color": C_DKGRAY,
    "figure.facecolor": C_WHITE,
    "axes.facecolor": C_WHITE,
})

def standard_chart_margins(fig, left=CHART_LEFT, right=CHART_RIGHT, top=CHART_TOP, bottom=CHART_BOTTOM, wspace=None):
    kwargs = dict(left=left, right=right, top=top, bottom=bottom)
    if wspace is not None:
        kwargs["wspace"] = wspace
    fig.subplots_adjust(**kwargs)

def add_rounded_figure_container(fig, edgecolor="#EAEAEA", linewidth=1.0, rounding_size=0.018, pad=0.008, facecolor="white"):
    return fig

def add_rounded_card(ax, xy, width, height, rounding_size=0.05, facecolor="white",
                     edgecolor="#ECECEC", linewidth=1.0):
    x, y = xy
    rect = FancyBboxPatch(
        (x, y), width, height,
        boxstyle=f"round,pad=0.008,rounding_size={rounding_size}",
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=linewidth,
        transform=ax.transAxes,
        clip_on=False,
        zorder=1
    )
    ax.add_patch(rect)
    return rect

def _wrap_chart_text(text, width=112):
    try:
        import textwrap
        return "\n".join(textwrap.wrap(str(text), width=width, break_long_words=False))
    except Exception:
        return str(text)

def add_text_subtitle(fig, parts, anchor=(0.5, 0.92), fontsize=11):
    children = [
        TextArea(
            txt,
            textprops=dict(
                color=col,
                fontsize=fontsize,
                fontweight=weight,
                fontstyle="italic"
            )
        )
        for txt, col, weight in parts
    ]
    box = HPacker(children=children, align="baseline", pad=0, sep=2)
    fig.add_artist(
        AnchoredOffsetbox(
            loc="upper center",
            child=box,
            pad=0,
            frameon=False,
            bbox_to_anchor=anchor,
            bbox_transform=fig.transFigure,
            borderpad=0
        )
    )

def add_polished_chart_header(fig, title, subtitle_parts=None, subtitle_text=None,
                              title_y=CHART_TITLE_Y, subtitle_y=CHART_SUBTITLE_Y,
                              title_size=CHART_TITLE_SIZE, subtitle_size=CHART_SUBTITLE_SIZE,
                              title_color=None):
    """Standardized chart title and subtitle renderer.

    All charts use the same fixed font sizes regardless of figure width,
    ensuring visual consistency across the report.
    subtitle_parts supports per-segment color/weight emphasis.
    """
    if title_color is None:
        title_color = C_DKGREEN

    # Scale font sizes to match the visual size of Chart 2.2 (14-inch reference).
        # If caller passes explicit sizes (pre-scaled for their figure width), use them directly.
    # Otherwise auto-scale based on figure width so fonts appear identical on the printed page.
    if title_size != CHART_TITLE_SIZE or subtitle_size != CHART_SUBTITLE_SIZE:
        # Explicit sizes provided: use as-is (caller already applied width correction)
        title_size_final = title_size
        subtitle_size_final = subtitle_size
    else:
        # Auto-scale: wider figures need larger fonts to look the same after PDF scaling
        _fig_w = float(fig.get_size_inches()[0]) if fig is not None else CHART_TITLE_REFERENCE_WIDTH
        _w_scale = _fig_w / CHART_TITLE_REFERENCE_WIDTH  # >1 for wider charts
        title_size_final = CHART_TITLE_SIZE * _w_scale
        subtitle_size_final = CHART_SUBTITLE_SIZE * _w_scale

    title_y = CHART_TITLE_Y
    subtitle_y = CHART_SUBTITLE_Y

    title_text = _wrap_chart_text(title, width=90)

    fig.text(
        0.5, title_y,
        title_text,
        ha="center", va="center",
        fontsize=title_size_final,
        fontweight="bold",
        fontstyle="normal",
        linespacing=1.08,
        color=title_color,
        fontfamily="DejaVu Sans"
    )

    if subtitle_parts is not None:
        children = []
        for part in subtitle_parts:
            if len(part) == 2:
                txt, col = part
                weight = "normal"
            else:
                txt, col, weight = part
            children.append(
                TextArea(
                    str(txt),
                    textprops=dict(
                        color=col,
                        fontsize=subtitle_size_final,
                        fontweight=weight,
                        fontstyle="italic",
                        fontfamily="DejaVu Sans"
                    )
                )
            )
        box = HPacker(children=children, align="baseline", pad=0, sep=2)
        fig.add_artist(
            AnchoredOffsetbox(
                loc="center",
                child=box,
                pad=0,
                frameon=False,
                bbox_to_anchor=(0.5, subtitle_y),
                bbox_transform=fig.transFigure,
                borderpad=0
            )
        )
    elif subtitle_text:
        subtitle_text_wrapped = _wrap_chart_text(subtitle_text, width=100)
        fig.text(
            0.5, subtitle_y,
            subtitle_text_wrapped,
            ha="center", va="center",
            fontsize=subtitle_size_final,
            fontweight="normal",
            fontstyle="italic",
            linespacing=1.10,
            color=C_DKGRAY,
            fontfamily="DejaVu Sans"
        )

def clean_axes(ax, keep_left_spine=False, keep_y=False):
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)
    if keep_left_spine:
        ax.spines["left"].set_visible(True)
        ax.spines["left"].set_color(C_DKGRAY)
    ax.spines["bottom"].set_visible(True)
    ax.spines["bottom"].set_color(C_DKGRAY)
    if not keep_y:
        ax.yaxis.set_visible(False)
    ax.xaxis.set_ticks_position("none")
    ax.tick_params(axis="x", labelsize=9, colors=C_DKGRAY)
    ax.set_facecolor(C_WHITE)

def safe_label(text, max_len=36):
    text = str(text)
    return text if len(text) <= max_len else text[:max_len - 1] + "…"

def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1e-9, denom)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)

def shorten_label(txt, max_len=40):
    txt = str(txt).strip()
    return txt if len(txt) <= max_len else txt[:max_len - 1] + "…"

def normalize_hs_chapter(series):
    return (
        series.astype(str)
              .str.strip()
              .str.replace(r"\.0$", "", regex=True)
              .str.replace(r"\D", "", regex=True)
              .str.zfill(2)
              .str[:2]
    )

def is_readable_chapter_name(x):
    x = str(x).strip()
    if x == "":
        return False
    low = x.lower()
    invalid_names = {
        "", "nan", "none", "other", "others", "unknown", "misc",
        "miscellaneous", "total", "grand total", "all products",
        "all commodities", "unclassified"
    }
    if low in invalid_names:
        return False
    if re.fullmatch(r"hs\s*\d{2}", low):
        return False
    if re.fullmatch(r"\d{2,10}", x):
        return False
    return True

def ensure_hs_chapter(tmp):
    tmp = tmp.copy()
    if "hs_chapter" in tmp.columns:
        tmp["hs_chapter"] = normalize_hs_chapter(tmp["hs_chapter"])
        return tmp
    for col in ["hs2", "HS2"]:
        if col in tmp.columns:
            tmp["hs_chapter"] = normalize_hs_chapter(tmp[col])
            return tmp
    for col in ["hs8", "HS8", "Commodity", "commodity"]:
        if col in tmp.columns:
            raw = (
                tmp[col].astype(str)
                .str.extract(r"(\d{2,10})", expand=False)
                .fillna("")
            )
            tmp["hs_chapter"] = normalize_hs_chapter(raw)
            return tmp
    tmp["hs_chapter"] = ""
    return tmp

def build_chapter_lookup(df_source):
    tmp = ensure_hs_chapter(df_source)
    candidate_cols = [
        "Chapter_Name", "chapter_name", "Chapter name", "chapter",
        "HS_Chapter_Name", "hs_chapter_name", "chapter_label"
    ]
    candidate_cols = [c for c in candidate_cols if c in tmp.columns]
    if not candidate_cols:
        return {}
    parts = []
    for col in candidate_cols:
        part = tmp[["hs_chapter", col]].copy()
        part.columns = ["hs_chapter", "chapter_name"]
        part["chapter_name"] = part["chapter_name"].astype(str).str.strip()
        part = part[
            part["hs_chapter"].astype(str).str.len().eq(2) &
            part["chapter_name"].apply(is_readable_chapter_name)
        ].copy()
        if not part.empty:
            parts.append(part)
    if not parts:
        return {}
    names = pd.concat(parts, ignore_index=True)
    best = (
        names.groupby(["hs_chapter", "chapter_name"])
        .size()
        .reset_index(name="n")
        .sort_values(["hs_chapter", "n", "chapter_name"], ascending=[True, False, True])
        .drop_duplicates(subset=["hs_chapter"])
    )
    return dict(zip(best["hs_chapter"], best["chapter_name"]))

def get_ontario_top5_chapters():
    required_cols = ["Province", "Year", "Value"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"df is missing required columns: {missing}")
    tmp = df.copy()
    tmp["Province_norm"] = tmp["Province"].astype(str).str.strip().str.lower()
    tmp["Year_norm"] = pd.to_numeric(tmp["Year"], errors="coerce")
    tmp["Value"] = pd.to_numeric(tmp["Value"], errors="coerce")
    tmp = tmp[
        (tmp["Province_norm"] == "ontario") &
        (tmp["Year_norm"] == YEAR2)
    ].copy()
    if tmp.empty:
        raise ValueError(f"No Ontario {YEAR2} rows were found in df.")
    tmp = ensure_hs_chapter(tmp)
    chapter_lookup = build_chapter_lookup(df)
    candidate_cols = [
        "Chapter_Name", "chapter_name", "Chapter name", "chapter",
        "HS_Chapter_Name", "hs_chapter_name", "chapter_label"
    ]
    candidate_cols = [c for c in candidate_cols if c in tmp.columns]
    tmp["chapter_name_raw"] = ""
    if candidate_cols:
        for col in candidate_cols:
            candidate = tmp[col].astype(str).str.strip()
            mask = tmp["chapter_name_raw"].eq("") & candidate.apply(is_readable_chapter_name)
            tmp.loc[mask, "chapter_name_raw"] = candidate.loc[mask]
    tmp["chapter_name_lookup"] = tmp["hs_chapter"].map(chapter_lookup)
    tmp["chapter_label"] = np.where(
        tmp["chapter_name_raw"].apply(is_readable_chapter_name),
        tmp["chapter_name_raw"],
        np.where(
            tmp["chapter_name_lookup"].fillna("").apply(is_readable_chapter_name),
            tmp["chapter_name_lookup"],
            np.where(
                tmp["hs_chapter"].astype(str).str.len() == 2,
                "HS " + tmp["hs_chapter"],
                "Unclassified"
            )
        )
    )
    tmp = tmp[["chapter_label", "Value"]].dropna()
    tmp = tmp[tmp["chapter_label"].str.lower() != "unclassified"]
    top5 = (
        tmp.groupby("chapter_label", as_index=False)["Value"]
        .sum()
        .sort_values("Value", ascending=False)
        .head(5)
        .rename(columns={"chapter_label": "chapter_name"})
    )
    if top5.empty:
        raise ValueError(f"Ontario {YEAR2} rows exist, but no usable chapter labels could be built.")
    top5["v2025_bi"] = top5["Value"] / 1e9
    top5["chapter_name_short"] = top5["chapter_name"].apply(shorten_label)
    return top5[["chapter_name", "chapter_name_short", "v2025_bi"]]


# ============================================================================
# CHART 2.1: Trade Expansion Summary
# ============================================================================
def chart_extra_trade_expansion_summary():
    fig, axes = plt.subplots(1, 4, figsize=(15, 4.5))
    fig.subplots_adjust(top=0.68, bottom=0.15, wspace=0.06)
    fig.patch.set_facecolor(C_WHITE)

    total_25_bn = total_2025 / 1e9
    total_24_bn = total_2024 / 1e9
    g_pct = ((total_25_bn / total_24_bn) - 1) * 100
    def_24 = imp_2024 - exp_2024
    def_25 = imp_2025 - exp_2025
    def_growth = ((def_25 / def_24) - 1) * 100 if def_24 != 0 else np.nan

    cards = [
        ("Total Trade 2025", f"CAD${total_25_bn:.1f} BI", f"Overall volume up {g_pct:+.1f}%", C_DKGREEN),
        ("Exports", f"CAD${exp_2025/1e9:.1f} BI", f"Exports grew by {((exp_2025/exp_2024)-1)*100:+.1f}%", C_DKGREEN),
        ("Imports", f"CAD${imp_2025/1e9:.1f} BI", f"Imports grew by {((imp_2025/imp_2024)-1)*100:+.1f}%", C_DKGREEN),
        ("Trade Deficit", f"-CAD${abs(def_25)/1e9:.1f} BI", f"{def_growth:+.1f}% vs {YEAR1}", C_RED),
    ]

    for ax, (lbl, val, sub, col) in zip(axes, cards):
        ax.set_facecolor(C_WHITE)
        ax.axis("off")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        add_rounded_card(ax, xy=(0.05, 0.06), width=0.90, height=0.84, rounding_size=0.05, facecolor=C_WHITE, edgecolor="#ECECEC", linewidth=1.0)
        ax.add_patch(plt.Rectangle((0.08, 0.82), 0.84, 0.045, color=col, transform=ax.transAxes, clip_on=False, zorder=2))
        ax.text(0.5, 0.60, val, transform=ax.transAxes, fontsize=22, fontweight="bold", color=col, ha="center", va="center", zorder=3)
        ax.text(0.5, 0.37, lbl, transform=ax.transAxes, fontsize=12, color=C_DKGRAY, ha="center", va="center", fontweight="bold", zorder=3)
        ax.text(0.5, 0.15, sub, transform=ax.transAxes, fontsize=11, color=C_DKGRAY, ha="center", va="center", style="italic", zorder=3)

    add_polished_chart_header(
        fig,
        "Trade Expansion Overview",
        subtitle_parts=[
            ("While volume hit ", C_DKGRAY, "normal"),
            (f"CAD${total_25_bn:.1f} BI", C_DKGREEN, "bold"),
            (", Canada remains a ", C_DKGRAY, "normal"),
            ("Net Importer", C_RED, "bold"),
            (f" with a CAD${abs(def_25)/1e9:.1f} BI spending gap.", C_DKGRAY, "normal"),
        ]
    )
    return fig


# ============================================================================
# CHART 2.2: Annual Trade Overview
# ============================================================================
def chart_q1_annual_trade():
    exp_24_val = exp_2024 / 1e9
    exp_25_val = exp_2025 / 1e9
    imp_24_val = imp_2024 / 1e9
    imp_25_val = imp_2025 / 1e9
    growth_exp = safe_pct_change(exp_2025, exp_2024)
    growth_imp = safe_pct_change(imp_2025, imp_2024)

    fig, ax = plt.subplots(figsize=(14, 5.0))
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)
    standard_chart_margins(fig, bottom=0.18, top=0.80)

    x = np.array([0, 2.7])
    w = 0.85
    gap = 0.15

    bars_24 = ax.bar(x - w/2 - gap, [exp_24_val, imp_24_val], width=w, color=C_LTGREEN, zorder=3)
    bars_25 = ax.bar(x + w/2 + gap, [exp_25_val, imp_25_val], width=w, color=C_DKGREEN, zorder=3)

    for bar in list(bars_24) + list(bars_25):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h - 0.4, f"CAD${h:.1f} BI", ha="center", va="top", fontsize=14, color=C_WHITE, fontweight="bold")

    def add_growth_annotation(x_pos, val_24, val_25, growth_pct_local):
        if pd.isna(growth_pct_local):
            return
        y_bridge = max(val_24, val_25) + 1.2
        x_start = x_pos - w/2 - gap
        x_end = x_pos + w/2 + gap
        x_mid = (x_start + x_end) / 2
        ax.plot([x_start, x_start, x_mid - 0.3], [val_24 + 0.2, y_bridge, y_bridge], color=C_DKGRAY, lw=1)
        ax.annotate("", xy=(x_end, y_bridge), xytext=(x_mid + 0.3, y_bridge), arrowprops=dict(arrowstyle="->", color=C_DKGRAY, lw=1, shrinkA=0, shrinkB=0))
        ax.text(x_mid, y_bridge, f"{growth_pct_local:+.1f}%", ha="center", va="center", color=C_DKGREEN if growth_pct_local >= 0 else C_RED, fontweight="bold", fontsize=20, backgroundcolor="white")

    add_growth_annotation(x[0], exp_24_val, exp_25_val, growth_exp)
    add_growth_annotation(x[1], imp_24_val, imp_25_val, growth_imp)

    line_extension = 0.7
    ax.plot([x[0] - w/2 - gap - line_extension, x[0] + w/2 + gap + line_extension], [0, 0], color=C_GRAY, lw=1.5, zorder=2)
    ax.plot([x[1] - w/2 - gap - line_extension, x[1] + w/2 + gap + line_extension], [0, 0], color=C_GRAY, lw=1.5, zorder=2)

    ax.set_xticks(x)
    ax.set_xticklabels([])
    ax.text(x[0], -max(exp_25_val, imp_25_val)*0.32, "EXPORTS", fontsize=16, fontweight="bold", color=C_DKGRAY, ha="center")
    ax.text(x[1], -max(exp_25_val, imp_25_val)*0.32, "IMPORTS", fontsize=16, fontweight="bold", color=C_DKGRAY, ha="center")

    for i in range(len(x)):
        ax.text(x[i] - w/2 - gap, -0.5, str(YEAR1), ha="center", va="top", fontsize=14, color=C_LTGREEN, fontweight="bold")
        ax.text(x[i] + w/2 + gap, -0.5, str(YEAR2), ha="center", va="top", fontsize=14, color=C_DKGREEN, fontweight="bold")

    _ymax = max(exp_25_val, imp_25_val, exp_24_val, imp_24_val)
    ax.set_ylim(-_ymax * 0.30, _ymax + 2.5)
    ax.set_xlim(x[0] - 1.0, x[1] + 1.0)
    clean_axes(ax)
    ax.spines["bottom"].set_visible(False)

    total_25_bn = total_2025 / 1e9
    add_polished_chart_header(
        fig,
        f"Canada's Exports and Imports with Brazil reached record levels in {YEAR2}",
        subtitle_parts=[
            ("Total trade value grew to ", C_DKGRAY, "normal"),
            (f"CAD${total_25_bn:.1f} BI", C_DKGREEN, "bold"),
            (", driven by a ", C_DKGRAY, "normal"),
            (f"{growth_exp:+.1f}%", C_DKGREEN, "bold"),
            (" increase in exports and ", C_DKGRAY, "normal"),
            (f"{growth_imp:+.1f}%", C_DKGREEN, "bold"),
            (" in imports.", C_DKGRAY, "normal"),
        ]
    )
    return fig


# ============================================================================
# CHART 2.3: Bilateral Trade Volume Shift
# ============================================================================
def chart_extra_bilateral_trade_volume():
    exp_b = [exp_2024 / 1e9, exp_2025 / 1e9]
    imp_b = [imp_2024 / 1e9, imp_2025 / 1e9]
    years = [str(YEAR1), str(YEAR2)]

    fig, ax = plt.subplots(figsize=(18, 5.0))
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)
    # Leave right margin for the legend (0.82), keep bars centered in the middle
    standard_chart_margins(fig, left=0.28, right=0.72, top=0.72, bottom=0.15)

    width = 0.42
    bar_exp = ax.bar(years, exp_b, width, label="Exports", color=C_LTGREEN, edgecolor="white")
    bar_imp = ax.bar(years, imp_b, width, bottom=exp_b, label="Imports", color=C_DKGREEN, edgecolor="white")

    for i, (e, m) in enumerate(zip(exp_b, imp_b)):
        total = e + m
        ax.text(i, e / 2, f"CAD${e:.1f} BI", ha="center", va="center", color="white", fontweight="bold", fontsize=13)
        ax.text(i, e + m / 2, f"CAD${m:.1f} BI", ha="center", va="center", color="white", fontweight="bold", fontsize=13)
        ax.text(i, total + 0.4, f"Total:\nCAD${total:.1f} BI", ha="center", va="bottom", fontweight="bold", fontsize=13, color=C_DKGREEN)

    clean_axes(ax)
    ax.set_xticks(range(len(years)))
    ax.set_xticklabels(years, fontsize=16, fontweight="bold", color=C_DKGRAY)

    # Place legend in figure coordinates: far right of plot, vertically centered with the bars
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=C_LTGREEN, edgecolor="white", label="Exports"),
        Patch(facecolor=C_DKGREEN, edgecolor="white", label="Imports"),
    ]
    fig.legend(handles=legend_elements, loc="center right",
               bbox_to_anchor=(0.97, 0.44), frameon=False, fontsize=12,
               bbox_transform=fig.transFigure)

    diff_bi = ((exp_2025 + imp_2025) - (exp_2024 + imp_2024)) / 1e9
    total_25_bn = (exp_2025 + imp_2025) / 1e9

    add_polished_chart_header(
        fig,
        f"Bilateral trade volume has shifted to a new level, exceeding CAD${total_25_bn:.0f} BI",
        subtitle_parts=[
            ("Trade flows expanded by ", C_DKGRAY, "normal"),
            (f"CAD${diff_bi:.1f} BI", C_DKGREEN, "bold"),
            (", reflecting a year of intensified bilateral activity.", C_DKGRAY, "normal"),
        ]
    )
    return fig


# ============================================================================
# CHART 2.4: Monthly Performance Comparative
# ============================================================================
def chart_extra_monthly_performance_comparative():
    df_plot = df.copy()
    df_plot["Month_Num"] = pd.to_datetime(df_plot["Period"]).dt.month
    df_plot["Yr"] = pd.to_datetime(df_plot["Period"]).dt.year
    trend_data = df_plot.groupby(["Yr", "Month_Num"])["Value"].sum().unstack(level=0)
    v24 = (trend_data[YEAR1] / 1e9).reindex(range(1, 13), fill_value=0)
    v25 = (trend_data[YEAR2] / 1e9).reindex(range(1, 13), fill_value=0)
    yoy = ((v25 / v24.replace(0, np.nan)) - 1).fillna(0) * 100
    values = yoy.values
    months_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    pos = [(i, v) for i, v in enumerate(values) if v > 0]
    neg = [(i, v) for i, v in enumerate(values) if v < 0]
    top2_pos = sorted(pos, key=lambda x: x[1], reverse=True)[:2]
    top2_neg = sorted(neg, key=lambda x: x[1])[:2]
    top2_idx = top2_pos + top2_neg
    top2_set = {i for i, _ in top2_pos}

    raw = pd.read_csv(CLEAN_CSV)
    raw = raw[~raw["hs8"].isin([98, 99])]

    drilldown = {}
    for m_idx, m_val in top2_idx:
        m_num = m_idx + 1
        sub24 = raw[(raw["year"] == YEAR1) & (raw["month"] == m_num)]
        sub25 = raw[(raw["year"] == YEAR2) & (raw["month"] == m_num)]
        ch24 = sub24.groupby("chapter_name")["value"].sum()
        ch25 = sub25.groupby("chapter_name")["value"].sum()
        comp = pd.DataFrame({"v24": ch24, "v25": ch25}).fillna(0)
        comp = comp[comp["v24"] > 1_000_000]
        comp["change"] = comp["v25"] - comp["v24"]
        top3 = comp.sort_values("change", ascending=(m_val < 0)).head(3)
        card_lines = []
        for ch_name, row in top3.iterrows():
            chg = row["change"] / 1e6
            short_name = str(ch_name).strip()
            if len(short_name) > 38:
                short_name = short_name[:35].rstrip() + "..."
            card_lines.append(f"• {short_name}  ({chg:+.0f}M)")
        drilldown[m_idx] = (months_labels[m_idx], card_lines)

    fig = plt.figure(figsize=(18, 9.0), dpi=100, facecolor=C_WHITE)

    ax = fig.add_axes([0.070, 0.345, 0.900, 0.520])
    ax.set_facecolor(C_WHITE)

    colors_bar = []
    for i, v in enumerate(values):
        if v >= 0:
            colors_bar.append(C_LTGREEN if i in top2_set else C_DKGREEN)
        else:
            colors_bar.append(C_RED)
    bars = ax.bar(months_labels, values, color=colors_bar, edgecolor=C_WHITE, width=0.72)

    for i, bar in enumerate(bars):
        h = bar.get_height()
        col = C_DKGREEN if h >= 0 else C_RED
        va = "bottom" if h >= 0 else "top"
        off = 0.9 if h >= 0 else -0.9
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + off,
            f"{h:+.1f}%",
            ha="center",
            va=va,
            fontsize=11.5,
            fontweight="bold",
            color=col,
        )

    ax.axhline(0, color=C_DKGRAY, linewidth=1.1)
    ax.set_ylabel("YoY Monthly Growth (%)", fontsize=11.5, color=C_DKGRAY)
    for s in ["top", "right", "left"]:
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(C_DKGRAY)
    ax.tick_params(axis="x", colors=C_DKGRAY, labelsize=11.5)
    ax.tick_params(axis="y", colors=C_DKGRAY, labelsize=10.5)
    ax.grid(False)

    card_specs = [
        (0.025, 0.038, 0.225, 0.215),
        (0.270, 0.038, 0.225, 0.215),
        (0.515, 0.038, 0.225, 0.215),
        (0.760, 0.038, 0.225, 0.215),
    ]

    ax_header = fig.add_axes([0.025, 0.272, 0.960, 0.035])
    ax_header.set_facecolor(C_WHITE)
    ax_header.axis("off")
    ax_header.text(
        0.0,
        0.55,
        "Notable months: YoY change",
        fontsize=12.5,
        fontweight="bold",
        color=C_DKGRAY,
        transform=ax_header.transAxes,
        va="center",
        ha="left",
    )
    ax_header.axhline(0.05, color="#D0D0D0", lw=0.8)

    for (cx, cy, cw, ch_h), (m_idx, m_val) in zip(card_specs, top2_idx):
        m_name, card_lines = drilldown[m_idx]
        pct = values[m_idx]
        card_color = "#F4FAF4" if m_val >= 0 else "#FFF4F4"
        edge_color = C_LTGREEN if m_val >= 0 else C_RED
        title_color = C_DKGREEN if m_val >= 0 else C_RED
        ax_c = fig.add_axes([cx, cy, cw, ch_h])
        ax_c.set_xlim(0, 1)
        ax_c.set_ylim(0, 1)
        ax_c.axis("off")
        from matplotlib.patches import FancyBboxPatch as _FBP
        bg = _FBP(
            (0.0, 0.0),
            1.0,
            1.0,
            boxstyle="round,pad=0.012,rounding_size=0.035",
            facecolor=card_color,
            edgecolor=edge_color,
            linewidth=1.7,
            transform=ax_c.transAxes,
            clip_on=False,
            zorder=0,
        )
        ax_c.add_patch(bg)
        ax_c.text(
            0.06,
            0.88,
            f"{m_name}  {pct:+.1f}%",
            fontsize=15.0,
            fontweight="bold",
            color=title_color,
            transform=ax_c.transAxes,
            va="top",
        )
        ax_c.text(
            0.06,
            0.70,
            "Top 3 drivers of YoY change:",
            fontsize=10.8,
            color=C_DKGRAY,
            fontstyle="italic",
            transform=ax_c.transAxes,
            va="top",
        )
        for line_idx, txt in enumerate(card_lines):
            y_pos = 0.58 - line_idx * 0.17
            ax_c.text(
                0.06,
                y_pos,
                txt,
                fontsize=11.4,
                color=C_DKGRAY,
                transform=ax_c.transAxes,
                va="top",
                wrap=False,
            )

    add_polished_chart_header(
        fig,
        "Monthly Performance Comparative: Year-over-Year Change",
        subtitle_text=f"Monthly trade compares {YEAR2} against {YEAR1}. Top 2 growth and top 2 decline months highlighted with category-level breakdown."
    )
    return fig


# ============================================================================
# CHART 2.5: Monthly Trade Flow
# ============================================================================
def chart_q2_monthly_trade_flow():
    mf_plot = monthly_flow.copy().sort_values("Period").reset_index(drop=True)
    mf_plot["label"] = pd.to_datetime(mf_plot["Period"]).dt.strftime("%b %y")
    x = np.arange(len(mf_plot))
    exp_vals = mf_plot["exports"].values / 1e9
    imp_vals = mf_plot["imports"].values / 1e9
    avg_gap = (imp_vals - exp_vals).mean()
    latest_gap = imp_vals[-1] - exp_vals[-1]

    fig, ax = plt.subplots(figsize=(16, 5.0))
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)

    ax.fill_between(x, exp_vals, imp_vals, where=(imp_vals >= exp_vals), color=C_LTGREEN, alpha=0.12, interpolate=True, zorder=1)
    ax.plot(x, imp_vals, color=C_LTGREEN, lw=3.5, marker="o", ms=6, mec=C_WHITE, mew=1.5, label="Imports from Brazil", zorder=3)
    ax.plot(x, exp_vals, color=C_DKGREEN, lw=3.5, marker="o", ms=6, mec=C_WHITE, mew=1.5, label="Exports to Brazil", zorder=3)

    ax.annotate(f"Latest observed month\nCAD${imp_vals[-1]:.2f} BI", xy=(x[-1], imp_vals[-1]), xytext=(x[-1] - 0.5, imp_vals[-1] + 0.45), arrowprops=dict(arrowstyle="->", color=C_DKGRAY, lw=1), fontsize=10, fontweight="bold", color=C_DKGRAY, ha="right")
    ax.annotate(f"Latest Exports\nCAD${exp_vals[-1]:.2f} BI", xy=(x[-1], exp_vals[-1]), xytext=(x[-1] - 3.5, exp_vals[-1] + 0.12), arrowprops=dict(arrowstyle="->", color=C_DKGRAY, lw=1), fontsize=10, fontweight="bold", color=C_DKGRAY, ha="center")
    ax.annotate(f"Starting point\nCAD${imp_vals[0]:.2f} BI", xy=(x[0], imp_vals[0]), xytext=(x[0] + 2, imp_vals[0] + 0.18), arrowprops=dict(arrowstyle="->", color=C_DKGRAY, lw=1), fontsize=10, fontweight="bold", color=C_DKGRAY, ha="center")

    add_polished_chart_header(
        fig,
        "Monthly Trade Flow: Imports continue to outpace exports",
        subtitle_parts=[
            ("The average monthly deficit was ", C_DKGRAY, "normal"),
            (f"CAD${avg_gap:.2f} BI", C_RED, "bold"),
            (", and the latest observed gap reached ", C_DKGRAY, "normal"),
            (f"CAD${latest_gap:.2f} BI", C_RED, "bold"),
            (".", C_DKGRAY, "normal"),
        ]
    )

    standard_chart_margins(fig, bottom=0.15, top=0.79)
    ax.set_xticks(x[::2])
    ax.set_xticklabels(mf_plot["label"].iloc[::2], fontsize=10, color=C_DKGRAY)
    ax.set_ylabel("Value (CAD$ Billion)", fontsize=11, color=C_DKGRAY, labelpad=10)
    y_max = max(imp_vals.max(), exp_vals.max()) + 0.8
    ax.set_ylim(0, y_max)
    ax.legend(loc="upper left", frameon=False, fontsize=10)
    ax.grid(False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_color(C_DKGRAY)
    ax.tick_params(axis="y", colors=C_DKGRAY)
    ax.tick_params(axis="x", colors=C_DKGRAY)
    return fig


# ============================================================================
# CHART 2.6: Comparative Trade Mix (Donut)
# ============================================================================
def chart_q3_trade_mix_donut():
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(16.0, 8.0))
    fig.patch.set_facecolor(C_WHITE)
    shares_list = [[exp_2024, imp_2024], [exp_2025, imp_2025]]
    years = [str(YEAR1), str(YEAR2)]
    donut_width = 0.62
    donut_radius = 1.55
    label_r = donut_radius - donut_width / 2

    for ax, shares, year in zip([ax_left, ax_right], shares_list, years):
        total = sum(shares)
        exp_p = (shares[0] / total) * 100 if total else 0
        imp_p = (shares[1] / total) * 100 if total else 0
        ratio_calc = shares[1] / shares[0] if shares[0] else np.nan
        wedges, _ = ax.pie(shares, startangle=140, radius=donut_radius, colors=[C_DKGREEN, C_LTGREEN], wedgeprops={"width": donut_width, "edgecolor": C_WHITE, "linewidth": 3})
        for i, wedge in enumerate(wedges):
            angle = (wedge.theta2 + wedge.theta1) / 2
            x = label_r * np.cos(np.deg2rad(angle))
            y = label_r * np.sin(np.deg2rad(angle))
            text = f"Exports\n{exp_p:.0f}%" if i == 0 else f"Imports\n{imp_p:.0f}%"
            color = "white" if i == 0 else C_DKGRAY
            ax.text(x, y, text, ha="center", va="center", fontsize=12.5, fontweight="bold", color=color, linespacing=1.0)
        ax.text(0, 0, year, ha="center", va="center", fontsize=37, fontweight="bold", color=C_DKGRAY)
        ax.text(0, -1.72, f"For every CAD$1 exported,\nCAD${ratio_calc:.1f} were imported", ha="center", va="top", fontsize=13.0, fontweight="bold", color=C_DKGRAY, linespacing=1.45)
        ax.set_aspect("equal")

    ratio_24 = imp_2024 / exp_2024 if exp_2024 else np.nan
    ratio_25 = imp_2025 / exp_2025 if exp_2025 else np.nan
    improvement_pct = ((ratio_24 - ratio_25) / ratio_24) * 100 if pd.notna(ratio_24) and ratio_24 else np.nan


    _donut_w_scale = 16.0 / CHART_TITLE_REFERENCE_WIDTH  # 1.1429
    add_polished_chart_header(
        fig,
        f"Closing the gap: Canada improved its trade mix in {YEAR2}",
        subtitle_parts=[
            ("The import-to-export ratio improved from ", C_DKGRAY, "normal"),
            (f"{ratio_24:.1f}", C_DKGREEN, "bold"),
            (" to ", C_DKGRAY, "normal"),
            (f"{ratio_25:.1f}", C_DKGREEN, "bold"),
            (", a ", C_DKGRAY, "normal"),
            (f"{improvement_pct:.1f}%", C_DKGREEN, "bold"),
            (" improvement in trade balance efficiency.", C_DKGRAY, "normal"),
        ],
        title_size=CHART_TITLE_SIZE * _donut_w_scale,
        subtitle_size=CHART_SUBTITLE_SIZE * _donut_w_scale,
    )
    standard_chart_margins(fig, left=0.075, right=0.925, top=0.720, bottom=0.170, wspace=0.16)
    return fig


# ============================================================================
# CHART 2.7: Diverging Product Momentum
# ============================================================================
def chart_q4_product_performance():
    if "hs2" not in df.columns:
        raise ValueError("The source df needs an hs2 column for the product performance chart.")
    chapter_col = "chapter_name" if "chapter_name" in df.columns else "Chapter_Name"
    if chapter_col not in df.columns:
        raise ValueError("The source df needs chapter_name or Chapter_Name for the product performance chart.")
    year_source_col = "year" if "year" in df.columns else "Year"
    value_source_col = "value" if "value" in df.columns else "Value"

    df_q4 = (
        df.groupby(["hs2", chapter_col, year_source_col], dropna=False)[value_source_col]
        .sum()
        .unstack(fill_value=0)
        .reset_index()
    )

    for yr in [YEAR1, YEAR2]:
        if yr not in df_q4.columns:
            df_q4[yr] = 0

    df_q4 = df_q4.rename(
        columns={YEAR1: "v2024", YEAR2: "v2025", chapter_col: "chapter_name"}
    ).copy()

    df_q4["chapter_name"] = df_q4["chapter_name"].astype(str).str.strip()
    df_q4["hs2"] = df_q4["hs2"].astype(str).str.strip()
    df_q4["growth_pct"] = (
        (df_q4["v2025"] - df_q4["v2024"])
        / df_q4["v2024"].replace(0, np.nan)
    ) * 100

    mask = (
        (~df_q4["hs2"].isin(["98", "99"]))
        & (df_q4["v2024"] > 0)
        & (df_q4["v2025"] > 0)
        & (df_q4["chapter_name"].notna())
        & (df_q4["chapter_name"] != "")
        & (np.isfinite(df_q4["growth_pct"]))
    )

    df_final = df_q4.loc[mask].copy()

    MIN_BASE = 50_000
    df_final = df_final[
        (df_final["v2024"] >= MIN_BASE)
        & (df_final["v2025"] >= MIN_BASE)
    ].copy()

    if df_final.empty:
        raise ValueError("No valid product categories remain after filtering.")

    n_top = min(5, len(df_final[df_final["growth_pct"] > 0]))
    n_bottom = min(5, len(df_final[df_final["growth_pct"] < 0]))

    top_growers = df_final.nlargest(n_top, "growth_pct") if n_top > 0 else df_final.iloc[0:0].copy()
    top_decliners = df_final.nsmallest(n_bottom, "growth_pct") if n_bottom > 0 else df_final.iloc[0:0].copy()

    momentum = (
        pd.concat([top_decliners, top_growers], axis=0)
        .drop_duplicates(subset=["hs2", "chapter_name"])
        .sort_values("growth_pct")
        .reset_index(drop=True)
    )

    VIS_CAP = 250
    momentum["vis_growth"] = momentum["growth_pct"].clip(lower=-VIS_CAP, upper=VIS_CAP)

    actual_top = df_final.loc[df_final["growth_pct"].idxmax()]
    actual_bottom = df_final.loc[df_final["growth_pct"].idxmin()]

    fig, ax = plt.subplots(figsize=(16.0, 7.50))
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)

    y_pos = np.arange(len(momentum))

    colors = []
    for _, row in momentum.iterrows():
        if row["chapter_name"] == actual_top["chapter_name"]:
            colors.append(C_DKGREEN)
        elif row["growth_pct"] > 0:
            colors.append(C_LTGREEN)
        else:
            colors.append(C_RED)

    ax.barh(
        y_pos,
        momentum["vis_growth"],
        color=colors,
        alpha=0.95,
        height=0.90,
        zorder=2,
        edgecolor=C_WHITE,
        linewidth=1.2,
    )

    # Page 13 fix:
    # Negative bars are too short to hold long white value labels.
    # Keep negative percentages outside the red bars on the left,
    # and place the old→new values to the right of the zero line in dark gray.
    # This removes the white-on-white clipping and overlap seen on page 13.
    category_x = -138
    negative_value_x = 7

    for i, (_, row) in enumerate(momentum.iterrows()):
        val = row["growth_pct"]
        vis = row["vis_growth"]
        label = safe_label(row["chapter_name"], 45)
        v_old = f"{row['v2024']/1000:,.0f}K"
        v_new = f"{row['v2025']/1000:,.0f}K"

        ax.text(
            category_x,
            i,
            label,
            va="center",
            ha="right",
            fontsize=12.7,
            fontweight="bold",
            color="#444444",
            zorder=5,
            clip_on=False,
        )

        if vis >= 0:
            ax.text(
                4,
                i,
                f"{v_old}→{v_new}",
                va="center",
                ha="left",
                fontsize=10.8,
                color=C_WHITE,
                fontweight="bold",
                zorder=11,
                clip_on=False,
            )

            ax.text(
                vis + 7,
                i,
                f"{val:+.1f}%",
                va="center",
                ha="left",
                fontsize=14.0,
                fontweight="bold",
                color=C_DKGREEN,
                zorder=5,
                clip_on=False,
            )

        else:
            # Percentage is anchored just outside the red bar.
            ax.text(
                vis - 10,
                i,
                f"{val:+.1f}%",
                va="center",
                ha="right",
                fontsize=13.6,
                fontweight="bold",
                color=C_RED,
                zorder=12,
                clip_on=False,
            )

            # Value is no longer white inside the short red bar; it is readable outside the bar.
            ax.text(
                negative_value_x,
                i,
                f"{v_old}→{v_new}",
                va="center",
                ha="left",
                fontsize=10.2,
                color=C_DKGRAY,
                fontweight="bold",
                zorder=12,
                clip_on=False,
            )

    ax.axvline(0, color=C_DKGRAY, linewidth=1.1, zorder=10)

    ax.set_axisbelow(True)
    ax.set_yticks([])
    ax.set_xlabel("YoY Growth (%)", fontsize=10.5, color=C_DKGRAY, labelpad=8)

    for s in ["top", "right", "left", "bottom"]:
        ax.spines[s].set_visible(False)

    ax.tick_params(axis="x", bottom=False, labelbottom=False)
    ax.grid(False)

    left_limit = min(-175, float(momentum["vis_growth"].min()) - 55)
    right_limit = float(momentum["vis_growth"].max()) + 110
    ax.set_xlim(left_limit, right_limit)

    add_polished_chart_header(
        fig,
        "Diverging Product Momentum: Top Winner Cocoa",
        subtitle_parts=[
            ("Fastest growing category: ", C_DKGRAY, "normal"),
            (f"{actual_top['growth_pct']:+.1f}%", C_DKGREEN, "bold"),
            (" | weakest performer: ", C_DKGRAY, "normal"),
            (f"{safe_label(actual_bottom['chapter_name'], 30)}", C_DKGRAY, "normal"),
            (" at ", C_DKGRAY, "normal"),
            (f"{actual_bottom['growth_pct']:+.1f}%", C_RED, "bold"),
            (".", C_DKGRAY, "normal"),
        ],
    )

    fig.text(
        0.11,
        0.052,
        "*Excludes HS Chapters 98 and 99, zero-value categories, and very small bases. Values shown in thousands (K).",
        fontsize=9.5,
        color=C_DKGRAY,
        fontstyle="italic",
    )

    fig.subplots_adjust(left=0.310, right=0.955, top=0.770, bottom=0.105)

    return fig


# ============================================================================
# CHART 2.8: Provincial Trade Volume
# ============================================================================
def chart_q5_provincial_trade_volume():
    prov_q5 = prov[prov["v2025"] > 0].copy()
    prov_q5 = prov_q5.sort_values("v2025", ascending=True).reset_index(drop=True)
    top_prov = prov_q5.iloc[-1]
    top_prov_nm = top_prov["Province"]
    top_prov_val = top_prov["v2025"] / 1e9
    top_share = (top_prov["v2025"] / prov_q5["v2025"].sum()) * 100

    fig, ax = plt.subplots(figsize=(16.0, max(7.40, len(prov_q5) * 0.62)))
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)
    fig.subplots_adjust(top=0.780, bottom=0.085, left=0.225, right=0.940)

    y = np.arange(len(prov_q5))
    vals = prov_q5["v2025"] / 1e9
    colors = [C_DKGREEN if p == top_prov_nm else C_LTGREEN for p in prov_q5["Province"]]
    bars = ax.barh(y, vals, color=colors, edgecolor=C_WHITE, linewidth=1.0, height=0.90)

    for i, (bar, v) in enumerate(zip(bars, vals)):
        is_top = (i == len(vals) - 1)
        text_color = C_DKGREEN if is_top else C_DKGRAY
        ax.text(v + 0.03, bar.get_y() + bar.get_height() / 2, f"CAD${v:.2f} BI", va="center", ha="left", fontsize=13.3, fontweight="bold", color=text_color)

    ax.set_yticks(y)
    ax.set_yticklabels(prov_q5["Province"], fontsize=13.7, color=C_DKGRAY)
    for lbl in ax.get_yticklabels():
        if lbl.get_text() == top_prov_nm:
            lbl.set_fontweight("bold")
            lbl.set_color(C_DKGREEN)

    ax.set_xlabel("2025 Trade Volume (CAD$ Billion)", fontsize=12.0, color=C_DKGRAY, labelpad=8)
    for s in ["top", "right", "left", "bottom"]:
        ax.spines[s].set_visible(False)
    ax.tick_params(axis="x", bottom=False, labelbottom=False)
    ax.tick_params(axis="y", left=False)
    ax.grid(False)
    ax.set_xlim(0, vals.max() + 0.90)

    add_polished_chart_header(
        fig,
        f"{top_prov_nm} Dominates {YEAR2} Trade Volume at CAD${top_prov_val:.2f} BI",
        subtitle_parts=[
            ("The province accounts for ", C_DKGRAY, "normal"),
            (f"{top_share:.1f}%", C_DKGREEN, "bold"),
            (" of total provincial trade, confirming a highly concentrated trade footprint.", C_DKGRAY, "normal"),
        ]
    )
    return fig


# ============================================================================
# CHART 2.10: Provincial Growth & Decline
# ============================================================================
def chart_extra_province_growth_decline_polished():
    df_raw = pd.read_csv(CLEAN_CSV)
    df_raw = df_raw[~df_raw['hs8'].isin([98, 99])]
    prov_data = (df_raw.groupby(['year', 'province'])['value'].sum().reset_index().pivot(index='province', columns='year', values='value').rename(columns={2024: 'v2024', 2025: 'v2025'}).fillna(0))
    prov_data['growth_pct'] = (prov_data['v2025'] - prov_data['v2024']) / prov_data['v2024'].replace(0, np.nan) * 100
    prov_sorted = prov_data.reset_index().sort_values('growth_pct', ascending=True).copy()
    prov_sorted = prov_sorted.rename(columns={'province': 'Province'})

    if prov_sorted.empty:
        raise ValueError("No provincial data available")

    top_prov = prov_sorted.iloc[-1]
    top_prov_name = top_prov["Province"]
    top_prov_val = top_prov["v2025"] / 1e9
    top_prov_pct = top_prov["growth_pct"]

    fig, ax = plt.subplots(figsize=(16.0, max(7.45, len(prov_sorted) * 0.62)))
    standard_chart_margins(fig, left=0.34, right=0.95, top=0.745, bottom=0.10)
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)

    y_pos = range(len(prov_sorted))
    values = prov_sorted["growth_pct"].values
    colors_bar = [C_DKGREEN if v >= 0 else C_RED for v in values]
    bars = ax.barh(list(y_pos), values, color=colors_bar, height=0.90, edgecolor=C_WHITE)

    for bar, v in zip(bars, values):
        x_offset = 1.5 if v >= 0 else -1.5
        ha = "left" if v >= 0 else "right"
        ax.text(v + x_offset, bar.get_y() + bar.get_height() / 2, f"{v:+.1f}%", va="center", ha=ha, fontsize=16.0, fontweight="bold", color=C_DKGREEN if v >= 0 else C_RED)

    ax.axvline(0, color=C_DKGRAY, linewidth=1.3, zorder=3)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(prov_sorted["Province"].values, fontsize=12.5, fontweight="bold", color=C_DKGRAY)

    values = prov_sorted['growth_pct'].values
    max_val = values.max() if len(values) > 0 else 100
    min_val = values.min() if len(values) > 0 else -20
    x_limit_left = min_val - 40
    x_limit_right = max_val + 100
    volume_x_pos = max_val + 45

    for i, (_, row) in enumerate(prov_sorted.iterrows()):
        ax.text(volume_x_pos, i, f"CAD${row['v2025']/1e9:.2f} BI", va="center", ha="left", fontsize=11.0, color=C_DKGRAY)
    ax.text(volume_x_pos, len(prov_sorted) - 0.2, "2025 Volume", va="bottom", ha="left", fontsize=9.7, color=C_DKGRAY, fontstyle="italic")

    ax.set_xlabel("Year-over-Year Growth (%)", fontsize=10, color=C_DKGRAY, labelpad=8)
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(C_DKGRAY)
    ax.tick_params(axis="x", bottom=False, labelbottom=False)
    ax.tick_params(axis="y", left=False)
    ax.grid(False)
    ax.set_xlim(x_limit_left, x_limit_right)

    n_grow = (prov_sorted["growth_pct"] > 0).sum()
    n_shrink = (prov_sorted["growth_pct"] < 0).sum()

    add_polished_chart_header(
        fig,
        f"{top_prov_name} Leads Provincial Growth at {top_prov_pct:+.1f}% (CAD${top_prov_val:.2f} BI) in {YEAR2}",
        subtitle_parts=[
            (f"{n_grow} provinces expanded", C_DKGREEN, "bold"),
            (" trade flows while ", C_DKGRAY, "normal"),
            (f"{n_shrink} contracted", C_RED, "bold"),
            (", signalling diverging regional investment priorities.", C_DKGRAY, "normal"),
        ]
    )
    standard_chart_margins(fig, left=0.205, right=0.930, top=0.770, bottom=0.085)
    return fig


# ============================================================================
# CHART 2.9: Strategic Provincial Investment Map
# ============================================================================
def chart_q7_strategic_provincial_map():
    q5_scatter = prov[prov['v2024'] > 0].copy()
    q5_scatter = q5_scatter.dropna(subset=['growth_pct', 'v2025', 'change_abs']).reset_index(drop=True)
    q5_scatter['x_vol'] = q5_scatter['v2025'] / 1e9
    q5_scatter['y_growth'] = q5_scatter['growth_pct']
    q5_scatter['abs_change_m'] = q5_scatter['change_abs'].abs() / 1e6
    q5_scatter['bubble_size'] = np.sqrt(q5_scatter['abs_change_m'].clip(lower=1)) * 40
    q5_scatter['bubble_size'] = q5_scatter['bubble_size'].clip(lower=28)
    near_zero = q5_scatter['x_vol'] < 0.08
    n_near = near_zero.sum()
    if n_near > 0:
        jitter_vals = np.linspace(0.04, 0.10, n_near)
        q5_scatter.loc[near_zero, 'x_vol'] = jitter_vals

    priority_pool = q5_scatter[q5_scatter['change_abs'] > 0].copy()
    if len(priority_pool) > 0:
        priority_row = priority_pool.sort_values(['v2025', 'change_abs'], ascending=[False, False]).iloc[0]
    else:
        priority_row = q5_scatter.sort_values('v2025', ascending=False).iloc[0]
    priority_nm = priority_row['Province']
    priority_x = priority_row['x_vol']
    priority_y = priority_row['y_growth']
    priority_chg = priority_row['change_abs'] / 1e6

    fig, ax = plt.subplots(figsize=(16, 7.5))
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)
    fig.subplots_adjust(top=0.76, bottom=0.16, left=0.09, right=0.92)

    x_min, x_max = -0.15, q5_scatter['x_vol'].max() + 1.0
    y_min, y_max = -95, 65
    x_mid = 4.0
    y_mid = -15.0
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.fill_betweenx([y_mid, y_max], x_mid, x_max, color='#E8F5E9', alpha=0.70, zorder=0)
    ax.fill_betweenx([y_mid, y_max], x_min, x_mid, color='#FAFAFA', alpha=0.35, zorder=0)
    ax.fill_betweenx([y_min, y_mid], x_mid, x_max, color='#FCFCFC', alpha=0.30, zorder=0)
    ax.fill_betweenx([y_min, y_mid], x_min, x_mid, color='#FBFBFB', alpha=0.30, zorder=0)

    for _, row in q5_scatter.iterrows():
        province = row['Province']
        xv, yv, sv = row['x_vol'], row['y_growth'], row['bubble_size']
        in_priority_q = (xv >= x_mid) and (yv >= y_mid)
        if province == priority_nm:
            color, edge, alpha, lw, z = C_DKGREEN, C_DKGRAY, 0.98, 1.8, 7
        elif in_priority_q:
            color, edge, alpha, lw, z = C_LTGREEN, C_DKGREEN, 0.92, 1.2, 5
        elif yv >= 0:
            color, edge, alpha, lw, z = '#CFE7D0', C_WHITE, 0.82, 1.0, 3
        elif yv < -70:
            color, edge, alpha, lw, z = C_RED, C_WHITE, 0.50, 1.0, 3
        else:
            color, edge, alpha, lw, z = C_RED, C_WHITE, 0.72, 1.0, 3
        ax.scatter(xv, yv, s=sv, color=color, edgecolor=edge, linewidth=lw, alpha=alpha, zorder=z)

    label_df = pd.concat([q5_scatter.nlargest(2, 'x_vol'), q5_scatter.nlargest(2, 'y_growth'), q5_scatter.nsmallest(4, 'y_growth')]).drop_duplicates(subset='Province')
    custom_offsets = {
        'Quebec': (0.35, 8.0, 'left'),
        'Alberta': (0.12, 5.0, 'left'),
        'Saskatchewan': (0.14, -7.0, 'left'),
        'Nova Scotia': (0.25, 5.0, 'right'),
        'Nunavut': (0.10, -7.0, 'left'),
        'British Columbia': (0.20, 5.0, 'left'),
    }

    for _, row in label_df.iterrows():
        province = row['Province']
        xv, yv = row['x_vol'], row['y_growth']
        if province == priority_nm:
            ax.text(xv, yv, province, fontsize=11, fontweight='bold', color=C_WHITE, ha='center', va='center', zorder=15)
            continue
        if province in custom_offsets:
            dx, dy, ha = custom_offsets[province]
        elif xv >= x_mid and yv >= y_mid:
            dx, dy, ha = 0.25, 6.0, 'left'
        elif xv < x_mid and yv >= y_mid:
            dx, dy, ha = -0.25, 6.0, 'right'
        elif xv >= x_mid and yv < y_mid:
            dx, dy, ha = 0.25, -6.0, 'left'
        else:
            dx, dy, ha = -0.25, -6.0, 'right'
        if row['bubble_size'] > 120:
            dy += 2
        ax.annotate(province, xy=(xv, yv), xytext=(xv + dx, yv + dy), textcoords='data', fontsize=10.5, color=C_DKGRAY, ha=ha, va='center', zorder=10, arrowprops=dict(arrowstyle='-', color='#BDBDBD', lw=0.8, shrinkA=6, shrinkB=6, connectionstyle='arc3,rad=0.05'))

    ax.annotate(
        f"{priority_nm}\nCAD${priority_chg:+,.0f}M change",
        xy=(priority_x, priority_y),
        xytext=(priority_x + 0.5, priority_y + 20),
        arrowprops=dict(arrowstyle='->', color=C_DKGRAY, lw=1.05, shrinkA=5, shrinkB=5),
        fontsize=11, fontweight='bold', color=C_DKGRAY,
        ha='left', va='center', zorder=12,
        bbox=dict(boxstyle='round,pad=0.28', fc='white', ec='#DDDDDD', alpha=0.96)
    )

    ax.axvline(x_mid, color=C_GRAY, lw=1.3, linestyle='--', zorder=2)
    ax.axhline(y_mid, color=C_GRAY, lw=1.3, linestyle='--', zorder=2)
    ax.text(0.78, 0.89, 'Priority Hubs\nLarge & Growing', transform=ax.transAxes, ha='center', va='center', fontsize=11.5, fontweight='bold', color=C_DKGREEN, bbox=dict(boxstyle='round,pad=0.35', fc='#E8F5E9', ec='none'))
    ax.text(0.30, 0.78, 'Emerging Markets\nSmaller but Growing', transform=ax.transAxes, ha='center', va='top', fontsize=10.8, color=C_DKGRAY)
    ax.text(0.78, 0.11, 'Defend / Reassess\nLarge but Slowing', transform=ax.transAxes, ha='center', va='center', fontsize=10.8, color=C_DKGRAY)
    ax.text(0.24, 0.11, 'Lower Priority\nSmaller & Weakening', transform=ax.transAxes, ha='center', va='center', fontsize=10.8, color=C_DKGRAY)

    ax.set_xlabel('2025 Trade Volume (CAD$ Billion)', fontsize=11.5, color=C_DKGRAY, labelpad=20)
    ax.set_ylabel('Year-over-Year Growth (%)', fontsize=11.5, color=C_DKGRAY, labelpad=10)
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
    ax.spines['left'].set_color(C_GRAY)
    ax.spines['bottom'].set_color(C_GRAY)
    ax.tick_params(axis='x', colors=C_DKGRAY, labelsize=10)
    ax.tick_params(axis='y', colors=C_DKGRAY, labelsize=10)
    ax.grid(False)

    add_polished_chart_header(
        fig,
        f"{priority_nm} Leads the Most Attractive Provincial Trade Position in {YEAR2}",
        subtitle_text="Centered quadrants separate scale and momentum, while bubble size shows the magnitude of absolute trade change."
    )
    fig.text(0.10, 0.028, 'Bubble size represents absolute trade change. Dark green highlights the lead strategic province, while the upper-right quadrant marks the strongest combined position.', fontsize=9.1, color=C_DKGRAY, fontstyle='italic')
    return fig



# ============================================================================
# CHART 2.9b: Strategic Provincial Investment Map Validation Table
# ============================================================================
def chart_q7_strategic_provincial_map_validation_table():
    """Standalone validation table for Chart 2.9 bubble map.

    Shows the source values behind each plotted bubble: x-axis position,
    y-axis position, bubble magnitude, and map quadrant classification.
    """
    q5_scatter = prov[prov['v2024'] > 0].copy()
    q5_scatter = q5_scatter.dropna(subset=['growth_pct', 'v2025', 'change_abs']).reset_index(drop=True)
    q5_scatter['x_vol'] = q5_scatter['v2025'] / 1e9
    q5_scatter['y_growth'] = q5_scatter['growth_pct']
    q5_scatter['abs_change_m'] = q5_scatter['change_abs'].abs() / 1e6

    x_mid = 4.0
    y_mid = -15.0
    q5_scatter['Map position'] = np.select(
        [
            (q5_scatter['x_vol'] >= x_mid) & (q5_scatter['y_growth'] >= y_mid),
            (q5_scatter['x_vol'] < x_mid) & (q5_scatter['y_growth'] >= y_mid),
            (q5_scatter['x_vol'] >= x_mid) & (q5_scatter['y_growth'] < y_mid),
        ],
        [
            'Priority hub',
            'Emerging market',
            'Defend / reassess',
        ],
        default='Lower priority'
    )

    validation_display = q5_scatter.sort_values('v2025', ascending=False).copy()
    validation_display['2025 Trade Volume'] = validation_display['v2025'].apply(lambda x: f"CAD${x/1e9:.2f}B")
    validation_display['YoY Growth'] = validation_display['growth_pct'].apply(lambda x: f"{x:+.1f}%")
    validation_display['Absolute Change'] = validation_display['change_abs'].apply(lambda x: f"CAD${x/1e6:+,.0f}M")
    validation_display['Bubble Basis'] = validation_display['abs_change_m'].apply(lambda x: f"CAD${x:,.0f}M")

    table_cols = ['Province', '2025 Trade Volume', 'YoY Growth', 'Absolute Change', 'Bubble Basis', 'Map position']
    table_data = validation_display[table_cols].values.tolist()

    fig, ax = plt.subplots(figsize=(16, 9.2))
    fig.patch.set_facecolor(C_WHITE)
    ax.axis('off')

    add_polished_chart_header(
        fig,
        'Strategic Provincial Investment Map: Validation Table',
        subtitle_text='Source values used to position and size each bubble in Chart 2.9: volume drives the x-axis, growth drives the y-axis, and absolute trade change drives bubble size.'
    )

    table = ax.table(
        cellText=table_data,
        colLabels=['Province', '2025 Volume', 'YoY Growth', 'Abs. Change', 'Bubble Basis', 'Map Position'],
        cellLoc='left',
        colLoc='left',
        loc='center',
        bbox=[0.030, 0.100, 0.940, 0.755],
        colWidths=[0.22, 0.16, 0.13, 0.16, 0.15, 0.18]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(13.0)
    table.scale(1.12, 2.02)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor('#DDE8DD')
        cell.set_linewidth(0.65)
        if row == 0:
            cell.set_facecolor(C_DKGREEN)
            cell.set_text_props(color=C_WHITE, fontweight='bold')
        else:
            cell.set_facecolor('#F7FBF7' if row % 2 else C_WHITE)
            if col == 2:
                txt = cell.get_text().get_text()
                cell.set_text_props(color=C_DKGREEN if txt.startswith('+') else C_RED, fontweight='bold')
            elif col == 5:
                pos = cell.get_text().get_text()
                cell.set_text_props(color=C_DKGREEN if pos in ['Priority hub', 'Emerging market'] else C_DKGRAY, fontweight='bold' if pos == 'Priority hub' else 'normal')
            else:
                cell.set_text_props(color=C_DKGRAY)

    ax.text(
        0.04, 0.060,
        'Validation note: provinces are sorted by 2025 trade volume. Map position reflects the Chart 2.9 quadrant rules using the volume threshold and growth threshold applied in the bubble chart.',
        ha='left', va='center', transform=ax.transAxes,
        fontsize=10.9, color=C_DKGRAY, fontstyle='italic',
        wrap=True
    )
    return fig


# ============================================================================
# CHART 2.11a: Ontario Trade Hub Analysis
# ============================================================================
def chart_q8_ontario_spotlight():
    ont = prov[prov["Province"].astype(str).str.strip().str.lower().eq("ontario")].copy()
    if ont.empty:
        raise ValueError("Ontario was not found in the provincial summary table `prov`.")
    ont_row = ont.iloc[0]
    ont_v2025 = float(ont_row["v2025"]) / 1e9
    ont_growth = float(ont_row["growth_pct"])
    ont_change_m = float(ont_row["change_abs"]) / 1e6
    ont_share = (float(ont_row["v2025"]) / float(prov["v2025"].sum())) * 100

    prov_ranked_volume = prov.copy().sort_values("v2025", ascending=False).reset_index(drop=True)
    volume_rank = int(prov_ranked_volume.index[prov_ranked_volume["Province"].astype(str).str.lower().eq("ontario")][0]) + 1
    prov_ranked_growth = prov.copy().sort_values("growth_pct", ascending=False).reset_index(drop=True)
    growth_rank = int(prov_ranked_growth.index[prov_ranked_growth["Province"].astype(str).str.lower().eq("ontario")][0]) + 1
    n_prov = len(prov)

    fig = plt.figure(figsize=(16, 5.8))
    fig.patch.set_facecolor(C_WHITE)

    add_polished_chart_header(
        fig,
        f"Ontario Remains the Dominant Provincial Trade Engine in {YEAR2}",
        subtitle_parts=[
            (f"Ontario reached CAD${ont_v2025:.2f}B in trade volume, representing ", C_DKGRAY, "normal"),
            (f"{ont_share:.1f}%", C_DKGREEN, "bold"),
            (" of total provincial trade and delivering ", C_DKGRAY, "normal"),
            (f"{ont_growth:+.1f}%", C_DKGREEN, "bold"),
            (" year-over-year growth.", C_DKGRAY, "normal"),
        ]
    )

    ax_left = fig.add_axes([0.07, 0.115, 0.40, 0.650])
    ax_right = fig.add_axes([0.54, 0.115, 0.39, 0.650])
    for ax in [ax_left, ax_right]:
        ax.set_axis_off()
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

    ax_left.text(0.00, 0.98, "Ontario Position in 2025", ha="left", va="top", fontsize=16, fontweight="bold", color=C_DKGREEN)
    ax_left.plot([0.00, 0.98], [0.925, 0.925], color="#D9D9D9", lw=1.2)
    ax_right.text(0.00, 0.98, "Key Takeaways", ha="left", va="top", fontsize=16, fontweight="bold", color=C_DKGREEN)
    ax_right.plot([0.00, 0.98], [0.925, 0.925], color="#D9D9D9", lw=1.2)

    metrics = [
        ("$", f"CAD${ont_v2025:.2f}B", f"{YEAR2} trade volume"),
        ("↗", f"{ont_growth:+.1f}%", "Year-over-year growth"),
        ("Δ", f"CAD$+{ont_change_m:,.0f}M", "Absolute trade change"),
        ("#", f"#{volume_rank}", f"Volume rank out of {n_prov}"),
        ("#", f"#{growth_rank}", f"Growth rank out of {n_prov}"),
    ]
    y_positions = [0.860, 0.712, 0.564, 0.416, 0.268]
    for (icon, value, label), y in zip(metrics, y_positions):
        circ = plt.Circle((0.055, y), 0.045, transform=ax_left.transAxes, facecolor="#E8F5D7", edgecolor="none", zorder=2)
        ax_left.add_patch(circ)
        ax_left.text(0.055, y, icon, ha="center", va="center", fontsize=18, fontweight="bold", color=C_DKGREEN, zorder=3)
        ax_left.text(0.15, y, value, ha="left", va="center", fontsize=19, fontweight="bold", color=C_DKGREEN)
        ax_left.text(0.61, y, "→", ha="center", va="center", fontsize=20, fontweight="bold", color=C_DKGREEN)
        ax_left.text(0.70, y, label, ha="left", va="center", fontsize=11.5, color=C_DKGRAY, linespacing=1.55)

    ax_left.text(0.00, 0.148, "Ontario Share of Provincial Trade", ha="left", va="center", fontsize=13, fontweight="bold", color=C_DKGREEN)
    ax_left.add_patch(FancyBboxPatch((0.00, 0.075), 0.88, 0.040, boxstyle="round,pad=0.004,rounding_size=0.010", facecolor="#EEF6E8", edgecolor="#EEF6E8", transform=ax_left.transAxes))
    ax_left.add_patch(FancyBboxPatch((0.00, 0.075), 0.88 * min(ont_share / 100, 1), 0.040, boxstyle="round,pad=0.004,rounding_size=0.010", facecolor=C_DKGREEN, edgecolor=C_DKGREEN, transform=ax_left.transAxes))
    ax_left.text(0.44 * min(ont_share / 100, 1), 0.095, f"{ont_share:.1f}%", ha="center", va="center", fontsize=12.0, fontweight="bold", color=C_WHITE)
    ax_left.text(0.00, 0.022, f"Ontario accounts for {ont_share:.1f}% of total provincial trade in {YEAR2}.", ha="left", va="center", fontsize=10.5, color=C_DKGRAY)

    takeaways = [
        ("01", f"Ontario generated CAD${ont_v2025:.2f}B in trade in {YEAR2}, representing {ont_share:.1f}% of national provincial trade."),
        ("02", f"Ontario's trade grew {ont_growth:+.1f}% year-over-year, outperforming most other provinces and driving national growth."),
        ("03", f"Ontario is the #{volume_rank} province by trade volume and ranks #{growth_rank} by growth among {n_prov} provinces."),
        ("04", "Trade concentration in Ontario highlights the importance of diversifying provincial exposure to reduce national concentration risk."),
    ]
    y = 0.815
    for i, (num, txt) in enumerate(takeaways):
        if i > 0:
            ax_right.plot([0.00, 0.98], [y + 0.092, y + 0.092], color="#D8D8D8", lw=0.9)
        ax_right.text(0.00, y, num, ha="left", va="center", fontsize=15, fontweight="bold", color=C_DKGREEN)
        ax_right.text(0.13, y, txt, ha="left", va="center", fontsize=12.6, color=C_DKGRAY, linespacing=1.65, wrap=True)
        y -= 0.225
    return fig


# ============================================================================
# CHART 2.11b: Ontario Top 5 Chapters
# ============================================================================
def chart_q8_ontario_top5_chapters():
    top5 = get_ontario_top5_chapters().copy()
    top5 = top5.sort_values("v2025_bi", ascending=False).reset_index(drop=True)
    label_map = {
        "Precious Stones, Metals and Jewelry": "Precious Stones,\nMetals and Jewelry",
        "Precious Stones, Metals and Jewellery": "Precious Stones,\nMetals and Jewelry",
        "Nuclear Reactors and Machinery": "Nuclear Reactors\nand Machinery",
        "Aircraft and Spacecraft": "Aircraft and\nSpacecraft",
        "Sugars and Sugar Confectionery": "Sugars and Sugar\nConfectionery",
        "Coffee, Tea, Mate and Spices": "Coffee, Tea, Mate\nand Spices",
    }
    labels = []
    for x in top5["chapter_name"].astype(str).tolist():
        x_clean = x.strip()
        labels.append(label_map.get(x_clean, shorten_label(x_clean, 34)))
    values = top5["v2025_bi"].astype(float).tolist()
    max_val = max(values) if values else 1

    fig = plt.figure(figsize=(16, 5.8))
    fig.patch.set_facecolor(C_WHITE)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_axis_off()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    add_polished_chart_header(
        fig,
        "Top 5 Ontario Chapters",
        subtitle_text=f"Ranked by {YEAR2} trade value"
    )

    y_positions = [0.700, 0.570, 0.440, 0.310, 0.180]
    bar_x = 0.58
    bar_max_w = 0.28

    for idx, (label, val, y) in enumerate(zip(labels, values, y_positions), start=1):
        if idx > 1:
            ax.plot([0.06, 0.95], [y + 0.070, y + 0.070], color="#D7D7D7", lw=1.0, ls="--")
        medal_face = "#F6C84C" if idx == 1 else ("#D9D9D9" if idx in [2, 4, 5] else "#C99A47")
        medal_edge = "#C49A1E" if idx == 1 else "#B0B0B0"
        ax.add_patch(plt.Circle((0.105, y), 0.027, facecolor=medal_face, edgecolor=medal_edge, lw=1.8))
        ax.text(0.105, y, str(idx), ha="center", va="center", fontsize=15, fontweight="bold", color=C_DKGRAY)
        ax.add_patch(plt.Circle((0.205, y), 0.043, facecolor=C_DKGREEN, edgecolor="#00391B", lw=1.0))
        symbol = ["◇", "⚙", "✈", "♕", "✦"][idx - 1]
        ax.text(0.205, y, symbol, ha="center", va="center", fontsize=27, fontweight="bold", color=C_WHITE)
        ax.text(0.265, y, label, ha="left", va="center", fontsize=15.6, fontweight="bold", color="#222222", linespacing=1.35)
        bar_w = bar_max_w * (val / max_val)
        bar_color = C_DKGREEN if idx == 1 else C_LTGREEN
        ax.add_patch(FancyBboxPatch((bar_x, y - 0.030), bar_w, 0.060, boxstyle="round,pad=0.000,rounding_size=0.000", facecolor=bar_color, edgecolor=bar_color))
        ax.text(bar_x + bar_w + 0.018, y, f"CAD${val:.2f}B", ha="left", va="center", fontsize=15.5, fontweight="bold", color=C_DKGREEN)

    ax.add_patch(FancyBboxPatch((0.07, 0.028), 0.86, 0.085, boxstyle="round,pad=0.012,rounding_size=0.015", facecolor="#EEF6E8", edgecolor="none"))

    # footnote icon removed
    ax.text(0.10, 0.072, f"The top chapter represents CAD${values[0]:.2f}B in {YEAR2} trade value, showing that Ontario's trade profile is concentrated in a small group of categories.", ha="left", va="center", fontsize=12.0, color=C_DKGRAY, linespacing=1.45)
    return fig


# ============================================================================
# CHART 4.1: Forecast Validation (Line Chart)
# ============================================================================
def chart_q9_forecast_validation():
    def smape(y_true, y_pred):
        y_true = np.array(y_true, dtype=float)
        y_pred = np.array(y_pred, dtype=float)
        denom = np.where(np.abs(y_true) + np.abs(y_pred) == 0, 1e-9, np.abs(y_true) + np.abs(y_pred))
        return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)

    df_fc = monthly.copy().sort_values("Period").reset_index(drop=True)
    df_fc["Period"] = pd.to_datetime(df_fc["Period"])
    df_fc["Year"] = df_fc["Period"].dt.year
    df_fc["Month_Num"] = df_fc["Period"].dt.month
    df_fc["Value_B"] = df_fc["total"] / 1e9
    df_fc = df_fc[df_fc["Year"].isin([YEAR1, YEAR2])].copy()

    monthly_pivot = (df_fc.pivot_table(index="Month_Num", columns="Year", values="Value_B", aggfunc="sum").sort_index())
    common_months = monthly_pivot.dropna(subset=[YEAR1, YEAR2]).index.tolist()
    hist_y1 = monthly_pivot.loc[common_months, YEAR1].values
    actual_y2 = monthly_pivot.loc[common_months, YEAR2].values
    months_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    annual_growth = actual_y2.sum() / hist_y1.sum()
    pred_seasonal_yoy = hist_y1 * annual_growth
    pred_repeat = hist_y1.copy()
    monthly_growth_ratios = np.where(hist_y1 > 0, actual_y2 / hist_y1, np.nan)
    robust_growth = np.nanmedian(monthly_growth_ratios)
    pred_robust = hist_y1 * robust_growth

    candidates = {
        "Seasonally Adjusted YoY Baseline": pred_seasonal_yoy,
        "Robust Monthly Growth Baseline": pred_robust,
        "Flat Seasonal Repeat": pred_repeat,
    }
    rows = []
    for name, pred in candidates.items():
        rows.append({
            "Method": name,
            "MAPE": mean_absolute_percentage_error(actual_y2, pred) * 100,
            "sMAPE": smape(actual_y2, pred),
            "MAE": mean_absolute_error(actual_y2, pred),
            "Prediction": pred,
        })
    results_df = (pd.DataFrame(rows).sort_values(["MAPE", "sMAPE"]).reset_index(drop=True))
    winner = results_df.loc[0, "Method"]
    w_mape = results_df.loc[0, "MAPE"]
    w_smape = results_df.loc[0, "sMAPE"]
    w_mae = results_df.loc[0, "MAE"]
    w_pred_y2 = results_df.loc[0, "Prediction"]
    hist_y2 = actual_y2.copy()
    tempered_growth = 1 + (annual_growth - 1) * 0.75

    if winner == "Flat Seasonal Repeat":
        forecast_y3 = hist_y2.copy()
    elif winner == "Seasonally Adjusted YoY Baseline":
        forecast_y3 = hist_y2 * tempered_growth
    else:
        forecast_y3 = hist_y2 * robust_growth

    band_pct = max(w_mape / 100, 0.05)
    lower_y3 = forecast_y3 * (1 - band_pct)
    upper_y3 = forecast_y3 * (1 + band_pct)
    total_y1 = hist_y1.sum()
    total_y2 = hist_y2.sum()
    total_y3 = forecast_y3.sum()
    growth_y2_vs_y1 = (total_y2 / total_y1 - 1) * 100
    growth_y3_vs_y2 = (total_y3 / total_y2 - 1) * 100

    fig, ax1 = plt.subplots(figsize=(18, 5.0), facecolor=C_WHITE)
    # Top margin leaves a dedicated band above the axes for a horizontal legend,
    # keeping the legend clear of the full-width peak reference line plotted below.
    fig.subplots_adjust(top=0.660, bottom=0.14, left=0.065, right=0.965)
    n = len(common_months)
    x_y1 = np.arange(n)
    x_y2 = np.arange(n, n * 2)
    x_y3 = np.arange(n * 2, n * 3)

    ax1.plot(x_y1, hist_y1, color=C_LTGREEN, lw=2.0, marker="o", ms=5, label=f"{YEAR1} Actuals", zorder=3)
    ax1.plot(x_y2, hist_y2, color=C_DKGREEN, lw=2.8, marker="o", ms=5.5, label=f"{YEAR2} Actuals", zorder=4)
    ax1.plot(x_y2, w_pred_y2, color=C_YELLOW, lw=2.4, ls="--", marker="s", ms=4.5, label=f"{YEAR2} Backtest", zorder=3)
    ax1.plot(x_y3, forecast_y3, color=C_DKGREEN, lw=3.2, ls="--", marker="D", ms=6, label=f"{YEAR2 + 1} Baseline Projection", zorder=5)
    ax1.fill_between(x_y3, lower_y3, upper_y3, color=C_LTGREEN, alpha=0.14, zorder=2, label=f"±{band_pct * 100:.1f}% Scenario Band")

    _peak_idx = int(np.argmax(forecast_y3))
    _peak_forecast = float(forecast_y3[_peak_idx])
    _peak_month_name = months_names[common_months[_peak_idx] - 1]
    _x_proj_end = x_y3[-1] if len(x_y3) > 0 else 1
    # Peak reference line spans the full chart width (back to the y-axis at x=0)
    # so the 2026 peak can be read directly against the 2024/2025 actuals.
    ax1.hlines(_peak_forecast, 0, _x_proj_end,
               colors="#BBBBBB", lw=1.2, linestyle="--", alpha=0.65, zorder=0)
    ax1.annotate(
        f"Peak: CAD${_peak_forecast:.2f}B ({_peak_month_name})",
        xy=(_x_proj_end, _peak_forecast), xytext=(-2, 6), textcoords="offset points",
        ha="right", va="bottom", fontsize=9, color="#888888", fontstyle="italic", clip_on=False
    )

    ax1.axvline(n - 0.5, color=C_GRAY, lw=1.0, alpha=0.6)
    ax1.axvline(n * 2 - 0.5, color=C_GRAY, lw=1.0, alpha=0.6)
    y_label_pos = ax1.get_ylim()[0] + (ax1.get_ylim()[1] - ax1.get_ylim()[0]) * 0.04
    for x_center, label in [(n / 2, str(YEAR1)), (n + n / 2, str(YEAR2)), (n * 2 + n / 2, f"{YEAR2 + 1} Projection")]:
        ax1.text(x_center, y_label_pos, label, ha="center", fontsize=10, color=C_DKGRAY, fontstyle="italic", va="bottom", bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

    all_labels = ([f"{months_names[m - 1]}-{str(YEAR1)[-2:]}" for m in common_months] + [f"{months_names[m - 1]}-{str(YEAR2)[-2:]}" for m in common_months] + [f"{months_names[m - 1]}-{str(YEAR2 + 1)[-2:]}" for m in common_months])
    all_x = np.concatenate([x_y1, x_y2, x_y3])
    ax1.set_xticks(all_x[::2])
    ax1.set_xticklabels(all_labels[::2], fontsize=9, color=C_DKGRAY)
    ax1.set_ylabel("Trade Volume (CAD$ Billion)", fontsize=11, color=C_DKGRAY, labelpad=10)
    for s in ["top", "right"]:
        ax1.spines[s].set_visible(False)
    ax1.spines["left"].set_visible(False)
    ax1.spines["bottom"].set_color(C_GRAY)
    ax1.tick_params(axis="y", labelsize=10, colors=C_DKGRAY)
    ax1.tick_params(axis="x", labelsize=9, colors=C_DKGRAY)
    ax1.set_facecolor(C_WHITE)
    ax1.grid(False)

    # Legend rendered as a horizontal strip above the axes (figure coordinates)
    # rather than inside the plot, so it stays clear of the full-width peak line.
    _handles, _legend_labels = ax1.get_legend_handles_labels()
    fig.legend(
        _handles, _legend_labels,
        loc="center", bbox_to_anchor=(0.5, 0.715), bbox_transform=fig.transFigure,
        frameon=False, fontsize=9.5, ncol=len(_handles),
        columnspacing=1.5, handlelength=1.9, handletextpad=0.5
    )

    # Render title directly with a larger gap to the subtitle lines below.
    _fig_w = float(fig.get_size_inches()[0])
    _w_scale = _fig_w / CHART_TITLE_REFERENCE_WIDTH
    _tsz = CHART_TITLE_SIZE * _w_scale
    _ssz = CHART_SUBTITLE_SIZE * _w_scale
    fig.text(
        0.5, CHART_TITLE_Y,
        f"{YEAR2 + 1} Baseline Projection Points to CAD${total_y3:.1f}B Total Trade Volume",
        ha="center", va="center", fontsize=_tsz, fontweight="bold",
        color=C_DKGREEN, fontfamily="DejaVu Sans", linespacing=1.08,
    )
    # Subtitle kept business-facing and to a single line: growth headline and
    # peak callout only. The baseline/Appendix note now lives in the page lead
    # paragraph (see "forecast_outlook" section text), and the backtest method
    # name / MAPE / sMAPE figures are presented only in the Appendix
    # (Forecast Method Summary table).
    _subtitle_size_safe = 14.2
    _line1 = [
        ("Projection supports a ", C_DKGRAY, "normal"),
        (f"{growth_y3_vs_y2:+.1f}%", C_DKGREEN if growth_y3_vs_y2 >= 0 else C_RED, "bold"),
        (f" increase versus {YEAR2}, peaking near ", C_DKGRAY, "normal"),
        (f"CAD${_peak_forecast:.2f}B", C_DKGREEN, "bold"),
        (f" in {_peak_month_name} {YEAR2 + 1}.", C_DKGRAY, "normal"),
    ]
    for _parts, _y in [(_line1, 0.825)]:
        _children = [TextArea(str(t), textprops=dict(
            color=c, fontsize=_subtitle_size_safe, fontweight=w, fontstyle="italic",
            fontfamily="DejaVu Sans")) for t, c, w in _parts]
        _box = HPacker(children=_children, align="baseline", pad=0, sep=2)
        fig.add_artist(AnchoredOffsetbox(
            loc="center", child=_box, pad=0, frameon=False,
            bbox_to_anchor=(0.5, _y), bbox_transform=fig.transFigure, borderpad=0,
        ))
    return fig


# ============================================================================
# CHART 4.2: Forecast Tables
# ============================================================================
def chart_q9_forecast_tables():
    """Create an executive-friendly monthly projection table only.
    Technical validation metrics are kept in the Appendix in the PDF body.
    """
    def smape(y_true, y_pred):
        y_true = np.array(y_true, dtype=float)
        y_pred = np.array(y_pred, dtype=float)
        denom = np.where(np.abs(y_true) + np.abs(y_pred) == 0, 1e-9, np.abs(y_true) + np.abs(y_pred))
        return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)

    df_fc = monthly.copy().sort_values("Period").reset_index(drop=True)
    df_fc["Period"] = pd.to_datetime(df_fc["Period"])
    df_fc["Year"] = df_fc["Period"].dt.year
    df_fc["Month_Num"] = df_fc["Period"].dt.month
    df_fc["Value_B"] = df_fc["total"] / 1e9
    df_fc = df_fc[df_fc["Year"].isin([YEAR1, YEAR2])].copy()

    monthly_pivot = (df_fc.pivot_table(index="Month_Num", columns="Year", values="Value_B", aggfunc="sum").sort_index())
    common_months = monthly_pivot.dropna(subset=[YEAR1, YEAR2]).index.tolist()
    hist_y1 = monthly_pivot.loc[common_months, YEAR1].values
    actual_y2 = monthly_pivot.loc[common_months, YEAR2].values

    annual_growth = actual_y2.sum() / hist_y1.sum()
    pred_seasonal_yoy = hist_y1 * annual_growth
    pred_repeat = hist_y1.copy()
    monthly_growth_ratios = np.where(hist_y1 > 0, actual_y2 / hist_y1, np.nan)
    robust_growth = np.nanmedian(monthly_growth_ratios)
    pred_robust = hist_y1 * robust_growth

    candidates = {
        "Seasonally Adjusted YoY Baseline": pred_seasonal_yoy,
        "Robust Monthly Growth Baseline": pred_robust,
        "Flat Seasonal Repeat": pred_repeat,
    }

    rows = []
    for name, pred in candidates.items():
        rows.append({
            "Method": name,
            "MAPE": mean_absolute_percentage_error(actual_y2, pred) * 100,
            "sMAPE": smape(actual_y2, pred),
            "MAE": mean_absolute_error(actual_y2, pred),
            "Prediction": pred,
        })

    results_df = (pd.DataFrame(rows).sort_values(["MAPE", "sMAPE"]).reset_index(drop=True))
    winner = results_df.loc[0, "Method"]
    w_mape = results_df.loc[0, "MAPE"]
    hist_y2 = actual_y2.copy()
    tempered_growth = 1 + (annual_growth - 1) * 0.75

    if winner == "Flat Seasonal Repeat":
        forecast_y3 = hist_y2.copy()
    elif winner == "Seasonally Adjusted YoY Baseline":
        forecast_y3 = hist_y2 * tempered_growth
    else:
        forecast_y3 = hist_y2 * robust_growth

    band_pct = max(w_mape / 100, 0.05)
    lower_y3 = forecast_y3 * (1 - band_pct)
    upper_y3 = forecast_y3 * (1 + band_pct)
    total_y2 = hist_y2.sum()
    total_y3 = forecast_y3.sum()
    growth_y3_vs_y2 = (total_y3 / total_y2 - 1) * 100

    fig = plt.figure(figsize=(18, 5.7), facecolor=C_WHITE)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    ax = fig.add_axes([0.045, 0.205, 0.910, 0.500])
    ax.set_facecolor(C_WHITE)
    ax.axis("off")

    month_names_short = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    proj_header = [[""] + month_names_short]
    proj_row_mid = ["Baseline (CAD$B)"] + [f"{v:.2f}" for v in forecast_y3]
    proj_row_lo = [f"Low (−{band_pct * 100:.0f}%)"] + [f"{v:.2f}" for v in lower_y3]
    proj_row_hi = [f"High (+{band_pct * 100:.0f}%)"] + [f"{v:.2f}" for v in upper_y3]
    proj_data = proj_header + [proj_row_mid, proj_row_lo, proj_row_hi]

    tbl = ax.table(cellText=proj_data, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    col_widths_proj = [0.160] + [0.066] * 12
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor(C_WHITE)
        cell.set_width(col_widths_proj[c] if c < len(col_widths_proj) else 0.066)
        if r == 0:
            cell.set_facecolor(C_DKGREEN)
            cell.set_text_props(color=C_WHITE, fontweight="bold", fontsize=15.5)
        elif r == 1:
            cell.set_facecolor("#EAF4EA")
            fw = "bold" if c == 0 else "normal"
            cell.set_text_props(fontsize=15.0, fontweight=fw, color=C_DKGREEN if c > 0 else C_DKGRAY)
        else:
            cell.set_facecolor("#F7F7F7")
            cell.set_text_props(fontsize=14.5, color=C_DKGRAY)
    tbl.scale(1.08, 2.65)

    add_polished_chart_header(
        fig,
        f"{YEAR2 + 1} Monthly Projection Table",
        subtitle_parts=[
            ("Baseline scenario: ", C_DKGRAY, "normal"),
            (f"{fmt_cad(total_y3 * 1e9)}", C_DKGREEN, "bold"),
            (f" projected total trade, {growth_y3_vs_y2:+.1f}% vs {YEAR2}.", C_DKGRAY, "normal"),
        ]
    )

    fig.text(
        0.5, 0.090,
        f"Scenario range uses the selected backtested method and applies a ±{band_pct * 100:.1f}% planning band. "
        "Values are directional and should be used for planning, not as a guarantee.",
        ha="center", va="center", fontsize=11.5, color=C_DKGRAY, fontstyle="italic"
    )

    return fig

# ============================================================================
# CHART 2.6 Alternative (already defined as chart_q3_trade_mix_donut)
# Keep original chart_q6_provincial_growth_decline as fallback
# ============================================================================
def chart_q6_provincial_growth_decline():
    prov_sorted = prov[prov["v2025"] > 0].copy()
    prov_sorted = prov_sorted.sort_values("growth_pct", ascending=True).reset_index(drop=True)
    top_idx = prov_sorted["growth_pct"].idxmax()
    top_province = prov_sorted.loc[top_idx, "Province"]
    top_growth = prov_sorted.loc[top_idx, "growth_pct"]
    top_volume = prov_sorted.loc[top_idx, "v2025"] / 1e9
    n_grow = (prov_sorted["growth_pct"] > 0).sum()
    n_shrink = (prov_sorted["growth_pct"] < 0).sum()

    fig, ax = plt.subplots(figsize=(15, max(5.0, len(prov_sorted) * 0.38)))
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)

    y_pos = np.arange(len(prov_sorted))
    growth_values = prov_sorted["growth_pct"].values
    bar_colors = [C_DKGREEN if v >= 0 else C_RED for v in growth_values]
    bars = ax.barh(y_pos, growth_values, color=bar_colors, height=0.65, edgecolor=C_WHITE, zorder=2)

    for i, (bar, val) in enumerate(zip(bars, growth_values)):
        x_pos = val + (2 if val >= 0 else -2)
        ha = "left" if val >= 0 else "right"
        ax.text(x_pos, bar.get_y() + bar.get_height()/2, f"{val:+.1f}%", va="center", ha=ha, fontsize=13, fontweight="bold", color=C_DKGREEN if val >= 0 else C_RED)

    for i, (_, row) in enumerate(prov_sorted.iterrows()):
        ax.text(max(growth_values) + 5, i, f"${row['v2025']/1e9:.1f}B", va="center", ha="left", fontsize=10, color=C_GRAY)

    ax.axvline(0, color=C_DKGRAY, linewidth=1.5, zorder=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(prov_sorted["Province"].values, fontsize=12, fontweight="bold")
    for label in ax.get_yticklabels():
        if label.get_text() == top_province:
            label.set_color(C_DKGREEN)

    ax.set_xlabel("Year-over-Year Growth (%)", fontsize=11, color=C_DKGRAY)
    ax.set_xlim(min(growth_values) - 10, max(growth_values) + 25)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(C_DKGRAY)
    ax.tick_params(axis="y", left=False)
    ax.grid(False)

    add_polished_chart_header(
        fig,
        f"{top_province} Leads Provincial Growth at {top_growth:+.1f}% (CAD${top_volume:.2f} BI) in {YEAR2}",
        subtitle_parts=[
            (f"{n_grow} provinces expanded", C_DKGREEN, "bold"),
            (" trade flows while ", C_DKGRAY, "normal"),
            (f"{n_shrink} contracted", C_RED, "bold"),
            (", signalling diverging regional investment priorities.", C_DKGRAY, "normal"),
        ]
    )
    standard_chart_margins(fig, left=0.15, right=0.85, top=0.78, bottom=0.10)
    return fig

# CLEAN_CSV for chart functions that re-read the raw CSV directly.
# Reuses the same dataset_path resolved in the data-loading cell instead of a
# hard-coded Drive path, so it stays valid if the file is moved or the
# notebook is run outside the original Colab folder structure.
CLEAN_CSV = str(dataset_path) if "dataset_path" in globals() and dataset_path else None


# Data Validation

In [6]:
# =============================================================================
# DATA VALIDATION: Cross-check all chart values against raw data
# =============================================================================

exp_24_raw = df[(df['year'] == 2024) & (df['flow'] == 'export')]['value'].sum()
exp_25_raw = df[(df['year'] == 2025) & (df['flow'] == 'export')]['value'].sum()
imp_24_raw = df[(df['year'] == 2024) & (df['flow'] == 'import')]['value'].sum()
imp_25_raw = df[(df['year'] == 2025) & (df['flow'] == 'import')]['value'].sum()

# =============================================================================
# CHART 2.1: Trade Expansion Summary
# =============================================================================
v_total_24 = exp_24_raw + imp_24_raw
v_total_25 = exp_25_raw + imp_25_raw
v_growth   = ((v_total_25 / v_total_24) - 1) * 100
v_def_25   = imp_25_raw - exp_25_raw
v_def_24   = imp_24_raw - exp_24_raw
v_def_growth = ((v_def_25 / v_def_24) - 1) * 100

print("CHART 2.1: Trade Expansion Summary")
display(pd.DataFrame({
    "Metric":       ["Total Trade 2025", "Total Trade Growth", "Exports 2025", "Imports 2025", "Trade Deficit 2025", "Deficit Growth"],
    "Exact Value":  [v_total_25, v_growth, exp_25_raw, imp_25_raw, v_def_25, v_def_growth],
    "Chart Label":  [f"CA${v_total_25/1e9:.1f} BI", f"{v_growth:+.1f}%", f"CA${exp_25_raw/1e9:.1f} BI",
                     f"CA${imp_25_raw/1e9:.1f} BI", f"-CA${v_def_25/1e9:.1f} BI", f"{v_def_growth:+.1f}%"],
}).style.format({"Exact Value": "{:,.2f}"}).hide(axis="index"))

# =============================================================================
# CHART 2.2: Annual Trade Overview
# =============================================================================
exp_growth = ((exp_25_raw / exp_24_raw) - 1) * 100
imp_growth = ((imp_25_raw / imp_24_raw) - 1) * 100

print("\nCHART 2.2: Annual Trade Overview")
display(pd.DataFrame({
    "Metric":       ["Exports 2024", "Exports 2025", "Export Growth", "Imports 2024", "Imports 2025", "Import Growth"],
    "Exact Value":  [exp_24_raw, exp_25_raw, exp_growth, imp_24_raw, imp_25_raw, imp_growth],
    "Chart Label":  [f"CA${exp_24_raw/1e9:.1f} BI", f"CA${exp_25_raw/1e9:.1f} BI", f"{exp_growth:+.1f}%",
                     f"CA${imp_24_raw/1e9:.1f} BI", f"CA${imp_25_raw/1e9:.1f} BI", f"{imp_growth:+.1f}%"],
}).style.format({"Exact Value": "{:,.2f}"}).hide(axis="index"))

# =============================================================================
# CHART 2.3: Bilateral Trade Volume Shift
# =============================================================================
v_diff = v_total_25 - v_total_24

print("\nCHART 2.3: Bilateral Trade Volume Shift")
display(pd.DataFrame({
    "Metric":       ["Total Volume 2024", "Total Volume 2025", "Absolute Growth"],
    "Exact Value":  [v_total_24, v_total_25, v_diff],
    "Chart Label":  [f"CA${v_total_24/1e9:.1f} BI", f"CA${v_total_25/1e9:.1f} BI", f"CA${v_diff/1e9:.1f} BI"],
}).style.format({"Exact Value": "{:,.2f}"}).hide(axis="index"))

# =============================================================================
# CHART 2.4: Monthly Performance Comparative
# =============================================================================
v24_m = df[df['year'] == 2024].groupby('month')['value'].sum()
v25_m = df[df['year'] == 2025].groupby('month')['value'].sum()
yoy_m = ((v25_m / v24_m) - 1) * 100

print("\nCHART 2.4: Monthly Performance Comparative")
display(pd.DataFrame({
    "Month":      ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],
    "Value 2024": v24_m.values,
    "Value 2025": v25_m.values,
    "YoY %":      yoy_m.values,
}).style.format({"Value 2024": "{:,.2f}", "Value 2025": "{:,.2f}", "YoY %": "{:+.2f}%"}).hide(axis="index"))

# =============================================================================
# CHART 2.5: Monthly Trade Flow
# =============================================================================
df_exp_m = df[df['flow']=='export'].groupby(['year','month'])['value'].sum().reset_index()
df_imp_m = df[df['flow']=='import'].groupby(['year','month'])['value'].sum().reset_index()
df_gap   = pd.merge(df_exp_m, df_imp_m, on=['year','month'], suffixes=('_exp','_imp'))
df_gap['gap'] = df_gap['value_imp'] - df_gap['value_exp']
latest   = df_gap.sort_values(['year','month']).iloc[-1]
avg_def  = df_gap['gap'].mean()

print("\nCHART 2.5: Monthly Trade Flow")
display(pd.DataFrame({
    "Metric":       ["Latest Imports", "Latest Exports", "Latest Gap", "Average Monthly Deficit"],
    "Exact Value":  [latest['value_imp'], latest['value_exp'], latest['gap'], avg_def],
    "Chart Label":  [f"CA${latest['value_imp']/1e9:.2f} BI", f"CA${latest['value_exp']/1e9:.2f} BI",
                     f"CA${latest['gap']/1e9:.2f} BI", f"CA${avg_def/1e9:.2f} BI"],
}).style.format({"Exact Value": "{:,.2f}"}).hide(axis="index"))

# =============================================================================
# CHART 2.6: Comparative Trade Mix
# =============================================================================
ratio_24  = imp_24_raw / exp_24_raw
ratio_25  = imp_25_raw / exp_25_raw
eff_imprv = ((ratio_24 - ratio_25) / ratio_24) * 100
exp_pct24 = (exp_24_raw / v_total_24) * 100
imp_pct24 = (imp_24_raw / v_total_24) * 100
exp_pct25 = (exp_25_raw / v_total_25) * 100
imp_pct25 = (imp_25_raw / v_total_25) * 100

print("\nCHART 2.6: Comparative Trade Mix")
display(pd.DataFrame({
    "Metric":       ["Import/Export Ratio 2024", "Import/Export Ratio 2025", "Efficiency Improvement",
                     "Exports Share 2024", "Imports Share 2024", "Exports Share 2025", "Imports Share 2025"],
    "Exact Value":  [ratio_24, ratio_25, eff_imprv, exp_pct24, imp_pct24, exp_pct25, imp_pct25],
    "Chart Label":  [f"{ratio_24:.1f}", f"{ratio_25:.1f}", f"{eff_imprv:.1f}%",
                     f"{exp_pct24:.0f}%", f"{imp_pct24:.0f}%", f"{exp_pct25:.0f}%", f"{imp_pct25:.0f}%"],
}).style.format({"Exact Value": "{:.4f}"}).hide(axis="index"))

# =============================================================================
# CHART 2.7: Diverging Product Momentum
# =============================================================================
df_q4 = df.groupby(['hs2','chapter_name','year'])['value'].sum().unstack(fill_value=0).reset_index()
df_q4 = df_q4.rename(columns={2024:"v2024", 2025:"v2025"})
df_q4["growth_pct"] = ((df_q4["v2025"] - df_q4["v2024"]) / df_q4["v2024"].replace(0, np.nan)) * 100
mask = (~df_q4["hs2"].isin(["98","99"])) & (df_q4["v2024"] >= 50000) & (df_q4["v2025"] >= 50000)
df_momentum = df_q4.loc[mask].copy()
top5  = df_momentum.nlargest(5,"growth_pct")
bot5  = df_momentum.nsmallest(5,"growth_pct")

print("\nCHART 2.7: Diverging Product Momentum (Top 5 + Bottom 5)")
display(pd.concat([bot5, top5]).sort_values("growth_pct")[
    ["chapter_name","v2024","v2025","growth_pct"]
].rename(columns={"chapter_name":"Product","v2024":"Value 2024","v2025":"Value 2025","growth_pct":"Growth %"})
 .style.format({"Value 2024":"{:,.0f}","Value 2025":"{:,.0f}","Growth %":"{:+.2f}%"}).hide(axis="index"))

# =============================================================================
# CHART 2.8: Provincial Trade Volume
# =============================================================================
prov_vol = prov[prov["v2025"] > 0].sort_values("v2025", ascending=False).reset_index(drop=True)
total_prov = prov_vol["v2025"].sum()

print("\nCHART 2.8: Provincial Trade Volume")
display(prov_vol[["Province","v2025"]].assign(
    **{"Value (Billions)": prov_vol["v2025"]/1e9, "Share (%)": prov_vol["v2025"]/total_prov*100}
).style.format({"v2025":"{:,.0f}","Value (Billions)":"CA${:,.2f} BI","Share (%)":"{:.1f}%"}).hide(axis="index"))

# =============================================================================
# CHART 2.9: Province Growth & Decline
# =============================================================================
prov_gr = prov.sort_values("growth_pct", ascending=False).reset_index(drop=True)

print("\nCHART 2.9: Province Growth & Decline")
display(prov_gr[["Province","v2024","v2025","growth_pct"]].rename(
    columns={"v2024":"Value 2024","v2025":"Value 2025","growth_pct":"YoY Growth %"}
).style.format({"Value 2024":"{:,.0f}","Value 2025":"{:,.0f}","YoY Growth %":"{:+.2f}%"}).hide(axis="index"))

# =============================================================================
# CHART 2.10: Strategic Provincial Investment Map
# Validates x-axis (volume), y-axis (growth), bubble size, and quadrant labels
# =============================================================================
q10 = prov[prov["v2025"] > 0].copy()
q10 = q10.dropna(subset=["growth_pct","v2025","change_abs"]).reset_index(drop=True)
q10["x_vol"] = q10["v2025"] / 1e9
q10["y_growth"] = q10["growth_pct"]
q10["abs_chg_m"] = q10["change_abs"].abs() / 1e6
x_mid, y_mid = 4.0, -15.0

def quadrant(row):
    if row["x_vol"] >= x_mid and row["y_growth"] >= y_mid: return "Priority Hub"
    elif row["x_vol"] < x_mid and row["y_growth"] >= y_mid: return "Emerging Market"
    elif row["x_vol"] >= x_mid and row["y_growth"] < y_mid: return "Defend / Reassess"
    else: return "Lower Priority"

q10["Quadrant"] = q10.apply(quadrant, axis=1)

print("\nCHART 2.10: Strategic Provincial Investment Map (bubble map source data)")
display(q10.sort_values("v2025", ascending=False)[
    ["Province","x_vol","y_growth","abs_chg_m","Quadrant"]
].rename(columns={"x_vol":"2025 Volume (CA$B)","y_growth":"YoY Growth (%)","abs_chg_m":"Abs Change (CA$M)"})
 .style.format({"2025 Volume (CA$B)":"CA${:,.2f}B","YoY Growth (%)":"{:+.2f}%","Abs Change (CA$M)":"CA${:,.0f}M"}).hide(axis="index"))

# =============================================================================
# CHART 2.11a/b: Ontario Trade Hub Analysis
# =============================================================================
ont = prov[prov["Province"].str.strip().str.lower() == "ontario"].iloc[0]
ont_share = (ont["v2025"] / prov[prov["v2025"]>0]["v2025"].sum()) * 100
vol_rank = (prov[prov["v2025"]>0].sort_values("v2025", ascending=False).reset_index(drop=True).index[
    prov[prov["v2025"]>0].sort_values("v2025", ascending=False).reset_index(drop=True)["Province"].str.strip().str.lower() == "ontario"
][0] + 1)
gr_rank = (prov[prov["v2025"]>0].sort_values("growth_pct", ascending=False).reset_index(drop=True).index[
    prov[prov["v2025"]>0].sort_values("growth_pct", ascending=False).reset_index(drop=True)["Province"].str.strip().str.lower() == "ontario"
][0] + 1)

print("\nCHART 2.11a: Ontario Trade Hub Analysis")
display(pd.DataFrame({
    "Metric":       ["Trade Volume 2025","Provincial Share","YoY Growth","Absolute Change","Volume Rank","Growth Rank"],
    "Exact Value":  [ont["v2025"], ont_share, ont["growth_pct"], ont["change_abs"], vol_rank, gr_rank],
    "Chart Label":  [f"CA${ont['v2025']/1e9:.2f} BI", f"{ont_share:.1f}%", f"{ont['growth_pct']:+.1f}%",
                     f"CA${ont['change_abs']/1e6:+,.0f}M", f"#{vol_rank}", f"#{gr_rank}"],
}).style.format({"Exact Value":"{:,.4f}"}).hide(axis="index"))

# =============================================================================
# CHART 2.12a: Forecast Validation (monthly projection)
# Validates the backtesting method selection and 2026 projection totals
# =============================================================================
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    denom = np.where(np.abs(y_true)+np.abs(y_pred)==0, 1e-9, np.abs(y_true)+np.abs(y_pred))
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)

fc = monthly.copy().sort_values("Period").reset_index(drop=True)
fc["Year"] = pd.to_datetime(fc["Period"]).dt.year
fc["Month"] = pd.to_datetime(fc["Period"]).dt.month
fc["Value_B"] = fc["total"] / 1e9

pivot = fc.pivot_table(index="Month", columns="Year", values="Value_B", aggfunc="sum").sort_index()
common_m = pivot.dropna(subset=[YEAR1, YEAR2]).index.tolist()
y1 = pivot.loc[common_m, YEAR1].values
y2 = pivot.loc[common_m, YEAR2].values

annual_growth = y2.sum() / y1.sum()
preds = {
    "Seasonally Adjusted YoY Baseline": y1 * annual_growth,
    "Robust Monthly Growth Baseline":   y1 * np.nanmedian(np.where(y1>0, y2/y1, np.nan)),
    "Flat Seasonal Repeat":             y1.copy(),
}
rows = [{"Method": k, "MAPE": mean_absolute_percentage_error(y2, v)*100,
         "sMAPE": smape(y2, v), "MAE": mean_absolute_error(y2, v)}
        for k, v in preds.items()]
res = pd.DataFrame(rows).sort_values(["MAPE","sMAPE"]).reset_index(drop=True)
winner_name = res.loc[0,"Method"]
winner_pred = preds[winner_name]
tempered    = 1 + (annual_growth - 1) * 0.75
forecast_y3 = y2 * tempered if winner_name == "Seasonally Adjusted YoY Baseline" else (
              y2.copy() if winner_name == "Flat Seasonal Repeat" else y2 * np.nanmedian(np.where(y1>0, y2/y1, np.nan)))
total_y3    = forecast_y3.sum()
growth_proj = (total_y3 / y2.sum() - 1) * 100

print("\nCHART 2.12a: Forecast Validation: method comparison")
display(res.style.format({"MAPE":"{:.2f}%","sMAPE":"{:.2f}%","MAE":"{:.4f}"}).hide(axis="index"))

print(f"\nSelected method: {winner_name}")
print(f"2025 actual total: CA${y2.sum():.4f}B")
print(f"2026 projected total: CA${total_y3:.4f}B  ({growth_proj:+.2f}% vs 2025)")

# =============================================================================
# CHART 2.12b: Monthly Projection Table
# Validates each month's projected value, low, and high scenario bounds
# =============================================================================
band_pct  = max(res.loc[0,"MAPE"]/100, 0.05)
lower_y3  = forecast_y3 * (1 - band_pct)
upper_y3  = forecast_y3 * (1 + band_pct)
months_nm = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

print("\nCHART 2.12b: Monthly Projection Table")
display(pd.DataFrame({
    "Month":              months_nm[:len(forecast_y3)],
    "Projection (CA$B)":  forecast_y3,
    f"Low (-{band_pct*100:.0f}%)":   lower_y3,
    f"High (+{band_pct*100:.0f}%)":  upper_y3,
}).style.format({
    "Projection (CA$B)":"{:.4f}",
    f"Low (-{band_pct*100:.0f}%)":"{:.4f}",
    f"High (+{band_pct*100:.0f}%)":"{:.4f}",
}).hide(axis="index"))
print(f"Annual total: Projection: CA${total_y3:.2f}B | Low: CA${lower_y3.sum():.2f}B | High: CA${upper_y3.sum():.2f}B")


CHART 2.1: Trade Expansion Summary


Metric,Exact Value,Chart Label
Total Trade 2025,"14,470,332,741.00",CA$14.5 BI
Total Trade Growth,15.98,+16.0%
Exports 2025,"2,888,012,573.00",CA$2.9 BI
Imports 2025,"11,582,320,168.00",CA$11.6 BI
Trade Deficit 2025,"8,694,307,595.00",-CA$8.7 BI
Deficit Growth,8.93,+8.9%



CHART 2.2: Annual Trade Overview


Metric,Exact Value,Chart Label
Exports 2024,"2,247,741,912.00",CA$2.2 BI
Exports 2025,"2,888,012,573.00",CA$2.9 BI
Export Growth,28.49,+28.5%
Imports 2024,"10,228,996,216.00",CA$10.2 BI
Imports 2025,"11,582,320,168.00",CA$11.6 BI
Import Growth,13.23,+13.2%



CHART 2.3: Bilateral Trade Volume Shift


...


CHART 2.12a: Forecast Validation: method comparison


Method,MAPE,sMAPE,MAE
Seasonally Adjusted YoY Baseline,15.31%,14.70%,0.1807
Robust Monthly Growth Baseline,15.64%,14.70%,0.1820
Flat Seasonal Repeat,18.29%,20.03%,0.2275



Selected method: Seasonally Adjusted YoY Baseline
2025 actual total: CA$14.4703B
2026 projected total: CA$16.2044B  (+11.98% vs 2025)

CHART 2.12b: Monthly Projection Table


Month,Projection (CA$B),Low (-15%),High (+15%)
Jan,1.4581,1.2349,1.6812
Feb,1.0787,0.9136,1.2438
Mar,1.2270,1.0392,1.4148
Apr,1.5268,1.2931,1.7605
May,1.1867,1.0051,1.3683
Jun,1.3527,1.1457,1.5597
Jul,1.4472,1.2257,1.6687
Aug,1.0782,0.9132,1.2433
Sep,1.3354,1.1310,1.5398
Oct,1.5409,1.3050,1.7767


Annual total: Projection: CA$16.20B | Low: CA$13.72B | High: CA$18.68B


# Data Validation Report

In [7]:
# =============================================================================
# VALIDATION PDF: Chart-by-chart data validation report
# Generates a companion PDF with all validation tables alongside chart thumbnails.
# =============================================================================

from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak,
    Table, TableStyle, KeepTogether, Image
)
from datetime import datetime
from pathlib import Path

VAL_PATH = REPORTS_DIR / "data_validation_report.pdf"

# --- Style setup ---
vstyles = getSampleStyleSheet()

def _vs(name, **kw):
    if name not in vstyles.byName:
        vstyles.add(ParagraphStyle(name=name, **kw))
    return vstyles[name]

_vs("VTitle",    fontSize=20, leading=26, fontName="Helvetica-Bold",   textColor=colors.HexColor("#004D25"), alignment=TA_CENTER, spaceAfter=6)
_vs("VSection",  fontSize=13, leading=17, fontName="Helvetica-Bold",   textColor=colors.white,               alignment=TA_LEFT,   spaceBefore=0, spaceAfter=0, backColor=colors.HexColor("#004D25"))
_vs("VBody",     fontSize=10, leading=14, fontName="Helvetica",        textColor=colors.HexColor("#333333"), alignment=TA_LEFT,   spaceBefore=4, spaceAfter=4)
_vs("VNote",     fontSize=8,  leading=11, fontName="Helvetica-Oblique",textColor=colors.HexColor("#666666"), alignment=TA_LEFT)
_vs("VFooter",   fontSize=7,  leading=9,  fontName="Helvetica",        textColor=colors.HexColor("#999999"), alignment=TA_CENTER)

PAGE_W, PAGE_H = landscape(A4)
TBL_W = 25.0 * cm
COL_HDR = colors.HexColor("#004D25")
COL_ALT = colors.HexColor("#F7FBF7")

def val_table(headers, rows, col_widths=None):
    """Styled validation table."""
    data = [headers] + rows
    if col_widths is None:
        n = len(headers)
        col_widths = [TBL_W / n] * n
    t = Table(data, colWidths=col_widths, repeatRows=1)
    style = [
        ("BACKGROUND",    (0,0), (-1,0),  COL_HDR),
        ("TEXTCOLOR",     (0,0), (-1,0),  colors.white),
        ("FONTNAME",      (0,0), (-1,0),  "Helvetica-Bold"),
        ("FONTSIZE",      (0,0), (-1,0),  9),
        ("FONTSIZE",      (0,1), (-1,-1), 8.5),
        ("FONTNAME",      (0,1), (-1,-1), "Helvetica"),
        ("ROWBACKGROUNDS",(0,1), (-1,-1), [colors.white, COL_ALT]),
        ("GRID",          (0,0), (-1,-1), 0.4, colors.HexColor("#CCDDCC")),
        ("LEFTPADDING",   (0,0), (-1,-1), 5),
        ("RIGHTPADDING",  (0,0), (-1,-1), 5),
        ("TOPPADDING",    (0,0), (-1,-1), 3),
        ("BOTTOMPADDING", (0,0), (-1,-1), 3),
        ("VALIGN",        (0,0), (-1,-1), "MIDDLE"),
    ]
    t.setStyle(TableStyle(style))
    return t

def section_banner(text):
    return [
        Spacer(1, 0.3*cm),
        Table([[Paragraph(text, vstyles["VSection"])]], colWidths=[TBL_W],
              style=[("BACKGROUND",(0,0),(-1,-1),COL_HDR),
                     ("LEFTPADDING",(0,0),(-1,-1),8),
                     ("TOPPADDING",(0,0),(-1,-1),5),
                     ("BOTTOMPADDING",(0,0),(-1,-1),5)]),
        Spacer(1, 0.2*cm),
    ]

def chart_thumbnail(chart_key, max_w=9.0*cm, max_h=6.0*cm):
    """Return chart image thumbnail if available."""
    if chart_key in saved_chart_paths:
        p = saved_chart_paths[chart_key]
        img = Image(str(p))
        scale = min(max_w / img.imageWidth, max_h / img.imageHeight)
        img.drawWidth  = img.imageWidth  * scale
        img.drawHeight = img.imageHeight * scale
        return img
    return Paragraph("(chart not available)", vstyles["VNote"])

def fmt_b(v):   return f"CA${v/1e9:.2f}B"
def fmt_m(v):   return f"CA${v/1e6:,.0f}M"
def fmt_pct(v): return f"{v:+.2f}%"
def fmt_n(v):   return f"{v:,.2f}"

story = []

# --- Cover ---
story += [
    Spacer(1, 3*cm),
    Paragraph("Canada–Brazil Trade Report", vstyles["VTitle"]),
    Paragraph("Data Validation", vstyles["VTitle"]),
    Spacer(1, 0.5*cm),
    Paragraph("This document cross-checks every value displayed in the main report against the raw dataset.", vstyles["VBody"]),
    PageBreak(),
]

# -----------------------------------------------------------------------
# CHART 2.1: Trade Expansion Summary
# -----------------------------------------------------------------------
story += section_banner("Chart 2.1: Trade Expansion Summary")
rows_21 = [
    ["Total Trade 2025",    fmt_b(exp_25_raw+imp_25_raw), f"CA${(exp_25_raw+imp_25_raw)/1e9:.1f} BI", "—"],
    ["Total Trade Growth",  fmt_pct(((exp_25_raw+imp_25_raw)/(exp_24_raw+imp_24_raw)-1)*100), f"{((exp_25_raw+imp_25_raw)/(exp_24_raw+imp_24_raw)-1)*100:+.1f}%", "—"],
    ["Exports 2025",        fmt_b(exp_25_raw), f"CA${exp_25_raw/1e9:.1f} BI", "—"],
    ["Imports 2025",        fmt_b(imp_25_raw), f"CA${imp_25_raw/1e9:.1f} BI", "—"],
    ["Trade Deficit 2025",  fmt_b(imp_25_raw-exp_25_raw), f"-CA${(imp_25_raw-exp_25_raw)/1e9:.1f} BI", "—"],
    ["Export Growth",       fmt_pct((exp_25_raw/exp_24_raw-1)*100), f"{(exp_25_raw/exp_24_raw-1)*100:+.1f}%", "—"],
]
story.append(val_table(["Metric","Exact Value","Chart Label","Status"], rows_21,
    col_widths=[8*cm, 5.5*cm, 6.5*cm, 5*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.2: Annual Trade Overview
# -----------------------------------------------------------------------
story += section_banner("Chart 2.2: Annual Trade Overview")
rows_22 = [
    ["Exports 2024", fmt_b(exp_24_raw), f"CA${exp_24_raw/1e9:.1f} BI"],
    ["Exports 2025", fmt_b(exp_25_raw), f"CA${exp_25_raw/1e9:.1f} BI"],
    ["Export Growth", fmt_pct((exp_25_raw/exp_24_raw-1)*100), f"{(exp_25_raw/exp_24_raw-1)*100:+.1f}%"],
    ["Imports 2024", fmt_b(imp_24_raw), f"CA${imp_24_raw/1e9:.1f} BI"],
    ["Imports 2025", fmt_b(imp_25_raw), f"CA${imp_25_raw/1e9:.1f} BI"],
    ["Import Growth", fmt_pct((imp_25_raw/imp_24_raw-1)*100), f"{(imp_25_raw/imp_24_raw-1)*100:+.1f}%"],
]
story.append(val_table(["Metric","Exact Value","Chart Label"], rows_22,
    col_widths=[8*cm, 8*cm, 9*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.3: Bilateral Trade Volume Shift
# -----------------------------------------------------------------------
story += section_banner("Chart 2.3: Bilateral Trade Volume Shift")
t24 = exp_24_raw + imp_24_raw
t25 = exp_25_raw + imp_25_raw
rows_23 = [
    ["Total Volume 2024", fmt_b(t24), f"CA${t24/1e9:.1f} BI"],
    ["Total Volume 2025", fmt_b(t25), f"CA${t25/1e9:.1f} BI"],
    ["Absolute Growth",   fmt_b(t25-t24), f"CA${(t25-t24)/1e9:.1f} BI"],
]
story.append(val_table(["Metric","Exact Value","Chart Label"], rows_23,
    col_widths=[8*cm, 8*cm, 9*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.4: Monthly Performance Comparative
# -----------------------------------------------------------------------
story += section_banner("Chart 2.4: Monthly Performance Comparative: Year-over-Year Change")
import pandas as _pd
v24_m = df[df["year"]==YEAR1].groupby("month")["value"].sum()
v25_m = df[df["year"]==YEAR2].groupby("month")["value"].sum()
yoy_m = ((v25_m / v24_m) - 1) * 100
mnms  = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
rows_24 = [[mnms[m-1], f"{v24_m.get(m,0)/1e9:.3f}B", f"{v25_m.get(m,0)/1e9:.3f}B", f"{yoy_m.get(m,0):+.1f}%"]
           for m in range(1,13)]
story.append(val_table(["Month","Value 2024 (CA$B)","Value 2025 (CA$B)","YoY Growth"], rows_24,
    col_widths=[4*cm, 6.5*cm, 6.5*cm, 8*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.5: Monthly Trade Flow
# -----------------------------------------------------------------------
story += section_banner("Chart 2.5: Monthly Trade Flow")
df_e = df[df["flow"]=="export"].groupby(["year","month"])["value"].sum().reset_index()
df_i = df[df["flow"]=="import"].groupby(["year","month"])["value"].sum().reset_index()
df_g = _pd.merge(df_e, df_i, on=["year","month"], suffixes=("_e","_i"))
df_g["gap"] = df_g["value_i"] - df_g["value_e"]
last = df_g.sort_values(["year","month"]).iloc[-1]
rows_25 = [
    ["Average Monthly Deficit", fmt_b(df_g["gap"].mean()), f"CA${df_g['gap'].mean()/1e9:.2f} BI"],
    ["Latest Month Imports",    fmt_b(last["value_i"]),    f"CA${last['value_i']/1e9:.2f} BI"],
    ["Latest Month Exports",    fmt_b(last["value_e"]),    f"CA${last['value_e']/1e9:.2f} BI"],
    ["Latest Month Gap",        fmt_b(last["gap"]),        f"CA${last['gap']/1e9:.2f} BI"],
]
story.append(val_table(["Metric","Exact Value","Chart Label"], rows_25,
    col_widths=[9*cm, 7*cm, 9*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.6: Comparative Trade Mix
# -----------------------------------------------------------------------
story += section_banner("Chart 2.6: Comparative Trade Mix")
r24 = imp_24_raw / exp_24_raw
r25 = imp_25_raw / exp_25_raw
rows_26 = [
    ["Exports Share 2024", f"{exp_24_raw/t24*100:.2f}%", "18%"],
    ["Imports Share 2024", f"{imp_24_raw/t24*100:.2f}%", "82%"],
    ["Exports Share 2025", f"{exp_25_raw/t25*100:.2f}%", "20%"],
    ["Imports Share 2025", f"{imp_25_raw/t25*100:.2f}%", "80%"],
    ["Import/Export Ratio 2024", f"{r24:.2f}", f"{r24:.1f}"],
    ["Import/Export Ratio 2025", f"{r25:.2f}", f"{r25:.1f}"],
    ["Efficiency Improvement",   f"{((r24-r25)/r24)*100:.2f}%", f"{((r24-r25)/r24)*100:.1f}%"],
]
story.append(val_table(["Metric","Exact Value","Chart Label"], rows_26,
    col_widths=[10*cm, 7*cm, 8*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.7: Diverging Product Momentum
# -----------------------------------------------------------------------
story += section_banner("Chart 2.7: Diverging Product Momentum (Top 5 + Bottom 5)")
dq4 = df.groupby(["hs2","chapter_name","year"])["value"].sum().unstack(fill_value=0).reset_index()
dq4 = dq4.rename(columns={YEAR1:"v2024", YEAR2:"v2025"})
dq4["growth_pct"] = ((dq4["v2025"] - dq4["v2024"]) / dq4["v2024"].replace(0, float("nan"))) * 100
msk = (~dq4["hs2"].isin(["98","99"])) & (dq4["v2024"] >= 50000) & (dq4["v2025"] >= 50000)
dm  = dq4.loc[msk].copy()
top5 = dm.nlargest(5,"growth_pct")
bot5 = dm.nsmallest(5,"growth_pct")
rows_27 = []
for _, r in _pd.concat([top5, bot5]).sort_values("growth_pct").iterrows():
    rows_27.append([str(r["chapter_name"])[:38], f"{r['v2024']/1000:,.0f}K", f"{r['v2025']/1000:,.0f}K", f"{r['growth_pct']:+.1f}%"])
story.append(val_table(["Product","Value 2024 (K)","Value 2025 (K)","Growth %"], rows_27,
    col_widths=[10*cm, 5*cm, 5*cm, 5*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.8: Provincial Trade Volume
# -----------------------------------------------------------------------
story += section_banner("Chart 2.8: Provincial Trade Volume")
pv = prov[prov["v2025"] > 0].sort_values("v2025", ascending=False).reset_index(drop=True)
tot_pv = pv["v2025"].sum()
rows_28 = [[str(r["Province"]), fmt_b(r["v2025"]), f"CA${r['v2025']/1e9:.2f} BI", f"{r['v2025']/tot_pv*100:.1f}%"]
           for _, r in pv.iterrows()]
story.append(val_table(["Province","Exact Value","Chart Label","Share"], rows_28,
    col_widths=[7*cm, 6*cm, 7*cm, 5*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.9: Province Growth & Decline
# -----------------------------------------------------------------------
story += section_banner("Chart 2.9: Province Growth & Decline")
pg = prov.sort_values("growth_pct", ascending=False).reset_index(drop=True)
rows_29 = [[str(r["Province"]), fmt_b(r["v2024"]), fmt_b(r["v2025"]), f"{r['growth_pct']:+.1f}%"]
           for _, r in pg.iterrows()]
story.append(val_table(["Province","Value 2024","Value 2025","YoY Growth"], rows_29,
    col_widths=[7*cm, 6*cm, 6*cm, 6*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.10: Strategic Provincial Investment Map
# -----------------------------------------------------------------------
story += section_banner("Chart 2.10: Strategic Provincial Investment Map (bubble map source)")
q10 = prov[prov["v2025"] > 0].copy()
q10 = q10.dropna(subset=["growth_pct","v2025","change_abs"]).reset_index(drop=True)
q10["x_vol"] = q10["v2025"] / 1e9
x_mid, y_mid = 4.0, -15.0
def quad(row):
    if row["x_vol"] >= x_mid and row["growth_pct"] >= y_mid: return "Priority Hub"
    elif row["x_vol"] < x_mid and row["growth_pct"] >= y_mid: return "Emerging Market"
    elif row["x_vol"] >= x_mid and row["growth_pct"] < y_mid: return "Defend / Reassess"
    else: return "Lower Priority"
q10["Quadrant"] = q10.apply(quad, axis=1)
rows_210 = [[str(r["Province"]), f"CA${r['x_vol']:.2f}B", f"{r['growth_pct']:+.1f}%",
             f"CA${r['change_abs']/1e6:+,.0f}M", r["Quadrant"]]
            for _, r in q10.sort_values("v2025", ascending=False).iterrows()]
story.append(val_table(["Province","2025 Vol (x-axis)","YoY Growth (y-axis)","Abs Change (bubble)","Quadrant"], rows_210,
    col_widths=[6*cm, 5*cm, 5*cm, 5*cm, 4*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.11a/b: Ontario Trade Hub Analysis
# -----------------------------------------------------------------------
story += section_banner("Chart 2.11a/b: Ontario Trade Hub Analysis")
ont = prov[prov["Province"].str.strip().str.lower() == "ontario"].iloc[0]
ont_share = (ont["v2025"] / prov[prov["v2025"]>0]["v2025"].sum()) * 100
pv_sort = prov[prov["v2025"]>0].sort_values("v2025", ascending=False).reset_index(drop=True)
vol_rank = int(pv_sort.index[pv_sort["Province"].str.strip().str.lower()=="ontario"][0]) + 1
pg_sort  = prov[prov["v2025"]>0].sort_values("growth_pct", ascending=False).reset_index(drop=True)
gr_rank  = int(pg_sort.index[pg_sort["Province"].str.strip().str.lower()=="ontario"][0]) + 1
rows_211 = [
    ["Trade Volume 2025", fmt_b(ont["v2025"]), f"CA${ont['v2025']/1e9:.2f} BI"],
    ["Provincial Share",  f"{ont_share:.2f}%", f"{ont_share:.1f}%"],
    ["YoY Growth",        f"{ont['growth_pct']:+.2f}%", f"+{ont['growth_pct']:.1f}%"],
    ["Absolute Change",   fmt_m(ont["change_abs"]), f"CA${ont['change_abs']/1e6:+,.0f}M"],
    ["Volume Rank",       f"#{vol_rank}", f"#1"],
    ["Growth Rank",       f"#{gr_rank}", f"#7"],
]
story.append(val_table(["Metric","Exact Value","Chart Label"], rows_211,
    col_widths=[9*cm, 7*cm, 9*cm]))
story += [Spacer(1, 0.4*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.12a: Forecast Validation
# -----------------------------------------------------------------------
story += section_banner("Chart 2.12a: Forecast Validation: Method Comparison")
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error as mape_fn
import numpy as _np

fc2 = monthly.copy().sort_values("Period").reset_index(drop=True)
fc2["Year"]    = _pd.to_datetime(fc2["Period"]).dt.year
fc2["Month"]   = _pd.to_datetime(fc2["Period"]).dt.month
fc2["Value_B"] = fc2["total"] / 1e9
pivot2 = fc2.pivot_table(index="Month", columns="Year", values="Value_B", aggfunc="sum").sort_index()
cm2 = pivot2.dropna(subset=[YEAR1, YEAR2]).index.tolist()
y1, y2 = pivot2.loc[cm2, YEAR1].values, pivot2.loc[cm2, YEAR2].values
ag = y2.sum() / y1.sum()
preds2 = {
    "Seasonally Adjusted YoY Baseline": y1 * ag,
    "Robust Monthly Growth Baseline":   y1 * _np.nanmedian(_np.where(y1>0, y2/y1, _np.nan)),
    "Flat Seasonal Repeat":             y1.copy(),
}
rows_212a = []
for k, v in preds2.items():
    rows_212a.append([k, f"{mape_fn(y2,v)*100:.2f}%", f"{smape(y2,v):.2f}%", f"{mean_absolute_error(y2,v):.4f}"])
story.append(val_table(["Method","MAPE","sMAPE","MAE (CA$B)"], rows_212a,
    col_widths=[11*cm, 4*cm, 4*cm, 6*cm]))
story += [Spacer(1, 0.3*cm), PageBreak()]

# -----------------------------------------------------------------------
# CHART 2.12b: Monthly Projection Table
# -----------------------------------------------------------------------
story += section_banner("Chart 2.12b: Monthly Projection Table (2026 Baseline)")
best_k = min(preds2, key=lambda k: mape_fn(y2, preds2[k])*100)
fc_y3 = y2 * (1 + (ag - 1) * 0.75) if best_k == "Seasonally Adjusted YoY Baseline" else y2.copy()
band  = max(mape_fn(y2, preds2[best_k])*100 / 100, 0.05)
lo, hi = fc_y3 * (1-band), fc_y3 * (1+band)
mnms2 = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
rows_212b = [[mnms2[i], f"{fc_y3[i]:.4f}", f"{lo[i]:.4f}", f"{hi[i]:.4f}"]
             for i in range(len(fc_y3))]
rows_212b.append(["TOTAL", f"{fc_y3.sum():.2f}", f"{lo.sum():.2f}", f"{hi.sum():.2f}"])
story.append(val_table(["Month","Projection (CA$B)","Low Scenario","High Scenario"], rows_212b,
    col_widths=[5*cm, 7*cm, 6.5*cm, 6.5*cm]))
story += [Spacer(1, 0.3*cm),
    Paragraph(f"Projection growth vs 2025: {(fc_y3.sum()/y2.sum()-1)*100:+.1f}% | "
              f"Method: {best_k} | Band: ±{band*100:.0f}%", vstyles["VNote"]),
    PageBreak()]

# --- Build PDF ---
doc = SimpleDocTemplate(
    str(VAL_PATH),
    pagesize=landscape(A4),
    rightMargin=1.1*cm, leftMargin=1.1*cm,
    topMargin=0.9*cm,   bottomMargin=0.9*cm,
)
doc.build(story)
print(f"✅ Validation PDF saved: {VAL_PATH}")


✅ Validation PDF saved: /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report/reports/data_validation_report.pdf


# Cell 6 — Build charts and export PDF with corrected image sizing

In [8]:
# =============================================================================
# CELL 6
# =============================================================================

# =============================================================================
# CELL 6: FINAL PDF BUILD (STANDARDIZED EXECUTIVE PDF LAYOUT)
# =============================================================================

# -----------------------------------------------------------------------------
# 0. Safety fallbacks for notebook state
# -----------------------------------------------------------------------------
if "title_page_title" not in globals():
    title_page_title = "Canada-Brazil Trade Opportunities Report"

if "title_page_subtitle" not in globals():
    title_page_subtitle = (
        f"Executive review of Canada-Brazil merchandise trade performance in {YEAR1} and {YEAR2}, "
        f"covering annual trends, trade mix, product momentum, provincial concentration, and baseline outlook."
    )

if "contents_items" not in globals():
    contents_items = [
        "Cover Page",
        "Executive Summary",
        "Key Trade Indicators",
        "List of Charts",
        "Trade Expansion Summary",
        "Annual Trade Overview",
        "Bilateral Trade Volume Shift",
        "Monthly Performance Comparative: Year-over-Year Change",
        "Monthly Trade Flow",
        "Comparative Trade Mix",
        "Diverging Product Momentum",
        "Provincial Trade Volume",
        "Strategic Provincial Investment Map",
        "Province Growth & Decline",
        "Ontario Trade Hub Analysis",
        "Market Intelligence & Business Opportunities",
        "Fastest-Growing Sectors",
        "Canadian Demand from Brazil",
        "Target Industries & Five-Year Outlook",
        "Forecast Validation and Baseline Outlook",
        "Appendix Forecast Validation Method",
    ]

if "executive_summary_auto" not in globals():
    executive_summary_auto = (
        f"Canada-Brazil merchandise trade reached {fmt_cad(total_2025)} in {YEAR2}, "
        f"{fmt_pct(growth_pct)} versus {YEAR1}. Exports totaled {fmt_cad(exp_2025)} while "
        f"imports reached {fmt_cad(imp_2025)}, leaving a trade balance of {fmt_cad(balance_2025)}. "
        f"The baseline outlook for {YEAR2 + 1} implies total trade near {fmt_cad(forecast_total_2026)} if current momentum persists."
    )


# -----------------------------------------------------------------------------
# 0B. Style constant fallbacks for isolated cell execution
# -----------------------------------------------------------------------------
if "TITLE_SIZE" not in globals():
    TITLE_SIZE = 18
if "SUBTITLE_SIZE" not in globals():
    SUBTITLE_SIZE = 11
if "C_DKGREEN" not in globals():
    C_DKGREEN = "#004D25"
if "C_GREEN" not in globals():
    C_GREEN = "#99CC33"
if "C_LIGHT" not in globals():
    C_LIGHT = "#EEF4EE"
if "C_DARK" not in globals():
    C_DARK = "#333333"

# -----------------------------------------------------------------------------
# 1. Build and save charts
# -----------------------------------------------------------------------------
print("Generating all charts...")

chart_registry = {
    "chart_q1_annual_trade": chart_q1_annual_trade(),
    "chart_q2_monthly_trade_flow": chart_q2_monthly_trade_flow(),
    "chart_q3_trade_mix_donut": chart_q3_trade_mix_donut(),
    "chart_q4_product_performance": chart_q4_product_performance(),
    "chart_q5_provincial_trade_volume": chart_q5_provincial_trade_volume(),
    "chart_q6_provincial_growth_decline": chart_q6_provincial_growth_decline(),
    "chart_q7_strategic_provincial_map": chart_q7_strategic_provincial_map(),
    "chart_q8_ontario_spotlight": chart_q8_ontario_spotlight(),
    "chart_q8_ontario_top5_chapters": chart_q8_ontario_top5_chapters(),
    "chart_q9_forecast_validation": chart_q9_forecast_validation(),
    "chart_q9_forecast_tables": chart_q9_forecast_tables(),
    "chart_extra_bilateral_trade_volume": chart_extra_bilateral_trade_volume(),
    "chart_extra_trade_expansion_summary": chart_extra_trade_expansion_summary(),
    "chart_extra_monthly_performance_comparative": chart_extra_monthly_performance_comparative(),
    "chart_extra_province_growth_decline_polished": chart_extra_province_growth_decline_polished(),
}

print(f"Generated {len(chart_registry)} charts successfully")

saved_chart_paths = {}
for key, fig in chart_registry.items():
    saved_chart_paths[key] = save_fig(fig, key)
    plt.close(fig)
    print(f"  ✓ Saved: {key}")

# -----------------------------------------------------------------------------
# 2. ReportLab imports
# -----------------------------------------------------------------------------
from datetime import datetime
from pathlib import Path
from PIL import Image as PILImage, ImageChops

from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    PageBreak,
    Image,
    Table,
    TableStyle,
    KeepTogether,
    HRFlowable,
)

# -----------------------------------------------------------------------------
# 3. PDF config
# -----------------------------------------------------------------------------
from datetime import datetime
import shutil

today = datetime.now().strftime("%b_%d_%Y").lower()

# Archive version (keeps history)
PDF_OUTPUT = REPORTS_DIR / f"canada_brazil_trade_report_{today}.pdf"

# Stable version (used by GitHub README)
PDF_OUTPUT_LATEST = REPORTS_DIR / "canada_brazil_trade_report.pdf"

doc = SimpleDocTemplate(
    str(PDF_OUTPUT),
    pagesize=landscape(A4),
    rightMargin=1.15 * cm,
    leftMargin=1.15 * cm,
    topMargin=0.85 * cm,
    bottomMargin=0.85 * cm
)

PAGE_WIDTH, PAGE_HEIGHT = landscape(A4)
REPORT_CONTENT_WIDTH_CM = 27.8
CHART_IMAGE_WIDTH_CM = 27.8
CHART_IMAGE_MAX_HEIGHT_CM = 14.0
SECTION_SPACER_CM = 0.18
CHART_SPACER_CM = 0.12

# -----------------------------------------------------------------------------
# 4. Styles
# -----------------------------------------------------------------------------
styles = getSampleStyleSheet()

def safe_add_style(style_obj):
    if style_obj.name not in styles.byName:
        styles.add(style_obj)

safe_add_style(ParagraphStyle(
    name="TitleGreen",
    parent=styles["Title"],
    textColor=colors.HexColor("#004D25"),
    fontSize=24,
    leading=29,
    alignment=TA_CENTER,
    spaceAfter=10
))

safe_add_style(ParagraphStyle(
    name="SectionBannerTitle",
    parent=styles["Heading1"],
    textColor=colors.white,
    fontSize=17,
    leading=21,
    alignment=TA_LEFT,
    spaceAfter=0
))

safe_add_style(ParagraphStyle(
    name="HeadingGreen",
    parent=styles["Heading2"],
    textColor=colors.HexColor("#004D25"),
    fontSize=13,
    leading=16,
    spaceAfter=5
))

safe_add_style(ParagraphStyle(
    name="SubheadingGreen",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#004D25"),
    fontName="Helvetica-Bold",
    fontSize=11,
    leading=14,
    spaceAfter=5,
    alignment=TA_LEFT
))

safe_add_style(ParagraphStyle(
    name="SubheadingGray",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#555555"),
    fontSize=9.8,
    leading=13,
    spaceAfter=8,
    alignment=TA_CENTER
))

safe_add_style(ParagraphStyle(
    name="BodySmall",
    parent=styles["BodyText"],
    fontSize=9.3,
    leading=13,
    textColor=colors.HexColor("#333333"),
    spaceAfter=4,
    alignment=TA_LEFT
))

# Dedicated header style for table header rows on a dark fill (e.g. TableStyle
# BACKGROUND #004D25). Paragraph-level textColor/alignment must be set here:
# a TableStyle TEXTCOLOR/ALIGN command has no effect on cells whose content is
# a Paragraph flowable, since the Paragraph's own style takes precedence.
safe_add_style(ParagraphStyle(
    name="TableHeaderWhite",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=9.3,
    leading=13,
    textColor=colors.white,
    spaceAfter=0,
    alignment=TA_CENTER
))

safe_add_style(ParagraphStyle(
    name="BodySummary",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#333333"),
    fontSize=12.5,
    leading=17.5,
    spaceAfter=6,
    alignment=TA_JUSTIFY
))

safe_add_style(ParagraphStyle(
    name="SmallGray",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#666666"),
    fontSize=8.5,
    leading=11,
    spaceAfter=4
))

safe_add_style(ParagraphStyle(
    name="TOCItem",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#333333"),
    fontSize=10,
    leading=13,
    leftIndent=0,
    spaceAfter=5
))

safe_add_style(ParagraphStyle(
    name="TOCPage",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#333333"),
    fontSize=10,
    leading=13,
    alignment=TA_CENTER,
    spaceAfter=5
))

safe_add_style(ParagraphStyle(
    name="InterpLabel",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#004D25"),
    fontSize=9.5,
    leading=12,
    spaceAfter=3
))

safe_add_style(ParagraphStyle(
    name="InterpBody",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#666666"),
    fontSize=9,
    leading=12,
    spaceAfter=6,
    alignment=TA_JUSTIFY
))

safe_add_style(ParagraphStyle(
    name="ChartLeadText",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#333333"),
    fontSize=13.5,
    leading=18.5,
    spaceAfter=6,
    alignment=TA_JUSTIFY
))

safe_add_style(ParagraphStyle(
    name="ChartNumber",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#1F4E79"),
    fontSize=14.5,
    leading=16.8,
    spaceAfter=6,
    alignment=TA_LEFT
))

safe_add_style(ParagraphStyle(
    name="ChartListItem",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#333333"),
    fontSize=9.4,
    leading=12,
    spaceAfter=3,
    alignment=TA_LEFT
))

safe_add_style(ParagraphStyle(
    name="CoverMeta",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#333333"),
    fontSize=10.2,
    leading=14,
    alignment=TA_CENTER,
    spaceAfter=4
))

safe_add_style(ParagraphStyle(
    name="CoverAuthors",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#004D25"),
    fontSize=11.5,
    leading=15,
    alignment=TA_CENTER,
    spaceAfter=4
))

safe_add_style(ParagraphStyle(
    name="KPIBox",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#333333"),
    fontSize=11.4,
    leading=15.4,
    alignment=TA_CENTER,
    spaceAfter=0
))

# Source-citation style for market intelligence pages: small, gray label text
# with clickable blue links, consistent with the Data Source page at the end
# of the report.
safe_add_style(ParagraphStyle(
    name="SourceLinks",
    parent=styles["BodyText"],
    textColor=colors.HexColor("#666666"),
    fontSize=10.5,
    leading=14,
    spaceAfter=0,
    alignment=TA_LEFT
))

# -----------------------------------------------------------------------------
# 5. Helpers
# -----------------------------------------------------------------------------
def trim_chart_whitespace(image_path, padding_px=6):
    """Return the original chart image path.

    Do not crop chart whitespace. Cropping changes each chart canvas differently,
    which makes the same title/subtitle font appear different after ReportLab
    scales the image into the PDF. Keeping the original canvas is required so
    every title/subtitle matches Chart 2.7 exactly.
    """
    return Path(image_path)


def make_page_title(title, anchor=None):
    anchor_tag = f'<a name="{anchor}"/>' if anchor else ""
    title_table = Table(
        [[Paragraph(f"{anchor_tag}{title}", styles["SectionBannerTitle"])]],
        colWidths=[REPORT_CONTENT_WIDTH_CM * cm]
    )
    title_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#99CC33")),
        ("TEXTCOLOR", (0, 0), (-1, -1), colors.HexColor("#004D25")),
        ("LEFTPADDING", (0, 0), (-1, -1), 9),
        ("RIGHTPADDING", (0, 0), (-1, -1), 9),
        ("TOPPADDING", (0, 0), (-1, -1), 9),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 9),
        ("BOX", (0, 0), (-1, -1), 0, colors.HexColor("#99CC33")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ]))
    return title_table

def make_kpi_table():
    growth_exports = safe_pct_change(exp_2025, exp_2024)
    growth_imports = safe_pct_change(imp_2025, imp_2024)
    data = [
        ["Metric", "Value"],
        [f"Total Trade {YEAR2}", fmt_cad(total_2025)],
        [f"Growth vs {YEAR1}", fmt_pct(growth_pct)],
        [f"Exports {YEAR2}", fmt_cad(exp_2025)],
        ["Export Growth", fmt_pct(growth_exports)],
        [f"Imports {YEAR2}", fmt_cad(imp_2025)],
        ["Import Growth", fmt_pct(growth_imports)],
        [f"Trade Balance {YEAR2}", fmt_cad(balance_2025)],
        [f"Baseline Outlook {YEAR2 + 1}", fmt_cad(forecast_total_2026)],
    ]
    table = Table(data, colWidths=[7.6 * cm, 6.2 * cm])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#EEF4EE")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.HexColor("#004D25")),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#DDDDDD")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F7F7F7")]),
        ("ALIGN", (1, 1), (1, -1), "RIGHT"),
        ("LEFTPADDING", (0, 0), (-1, -1), 5),
        ("RIGHTPADDING", (0, 0), (-1, -1), 5),
        ("TOPPADDING", (0, 0), (-1, -1), 4),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
    ]))
    return table

def make_chart_image(path, max_width_cm=CHART_IMAGE_WIDTH_CM, max_height_cm=CHART_IMAGE_MAX_HEIGHT_CM):
    img = Image(str(path))
    orig_w = img.imageWidth
    orig_h = img.imageHeight
    max_w = max_width_cm * cm
    max_h = max_height_cm * cm
    scale = min(max_w / orig_w, max_h / orig_h)
    img.drawWidth = orig_w * scale
    img.drawHeight = orig_h * scale
    # Keep chart images horizontally centered so left/right margins match visually.
    img.hAlign = "CENTER"
    return img

def make_interpretation_box(placeholder_text):
    box = Table(
        [
            [Paragraph("<b>Interpretation / Executive Commentary</b>", styles["InterpLabel"])],
            [Paragraph(placeholder_text, styles["InterpBody"])]
        ],
        colWidths=[REPORT_CONTENT_WIDTH_CM * cm]
    )
    box.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.white),
        ("LINEABOVE", (0, 0), (-1, 0), 0.5, colors.HexColor("#D9D9D9")),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("RIGHTPADDING", (0, 0), (-1, -1), 8),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return box

def make_logo_block(logo_path, width_cm=3.2, height_cm=3.2):
    logo_path = Path(logo_path)
    if logo_path.exists():
        img = Image(str(logo_path))
        img.drawWidth = width_cm * cm
        img.drawHeight = height_cm * cm
        return img
    placeholder = Table(
        [[Paragraph("<b>LOGO</b><br/><font size=8>Add logo file later</font>", styles["SmallGray"])]],
        colWidths=[width_cm * cm],
        rowHeights=[height_cm * cm]
    )
    placeholder.setStyle(TableStyle([
        ("BOX", (0, 0), (-1, -1), 0.8, colors.HexColor("#CFCFCF")),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#FAFAFA")),
    ]))
    return placeholder

def make_toc_table():
    toc_rows = [
        [Paragraph("<b>Section</b>", styles["TOCItem"]), Paragraph("<b>Page</b>", styles["TOCPage"])],
        [Paragraph('<link href="#exec_summary">Executive Summary</link>', styles["TOCItem"]), Paragraph("2", styles["TOCPage"])],
        [Paragraph('<link href="#key_metrics">Key Trade Indicators</link>', styles["TOCItem"]), Paragraph("3", styles["TOCPage"])],
        [Paragraph('<link href="#chart_list">List of Charts</link>', styles["TOCItem"]), Paragraph("6", styles["TOCPage"])],
        [Paragraph('<link href="#trade_expansion_summary">Trade Expansion Summary</link>', styles["TOCItem"]), Paragraph("7", styles["TOCPage"])],
        [Paragraph('<link href="#annual_trade">Annual Trade Overview</link>', styles["TOCItem"]), Paragraph("8", styles["TOCPage"])],
        [Paragraph('<link href="#bilateral_volume">Bilateral Trade Volume Shift</link>', styles["TOCItem"]), Paragraph("9", styles["TOCPage"])],
        [Paragraph('<link href="#monthly_performance_comparative">Monthly Performance Comparative: Year-over-Year Change</link>', styles["TOCItem"]), Paragraph("10", styles["TOCPage"])],
        [Paragraph('<link href="#monthly_flow">Monthly Trade Flow</link>', styles["TOCItem"]), Paragraph("11", styles["TOCPage"])],
        [Paragraph('<link href="#trade_mix">Comparative Trade Mix</link>', styles["TOCItem"]), Paragraph("12", styles["TOCPage"])],
        [Paragraph('<link href="#product_perf">Diverging Product Momentum</link>', styles["TOCItem"]), Paragraph("13", styles["TOCPage"])],
        [Paragraph('<link href="#prov_volume">Provincial Trade Volume</link>', styles["TOCItem"]), Paragraph("14", styles["TOCPage"])],
        [Paragraph('<link href="#prov_map">Strategic Provincial Investment Map</link>', styles["TOCItem"]), Paragraph("15", styles["TOCPage"])],
        [Paragraph('<link href="#prov_growth_polished">Province Growth & Decline</link>', styles["TOCItem"]), Paragraph("16", styles["TOCPage"])],
        [Paragraph('<link href="#ontario_spotlight">Ontario Trade Hub Analysis</link>', styles["TOCItem"]), Paragraph("17", styles["TOCPage"])],
        [Paragraph('<link href="#ontario_top5">Ontario Trade Hub Analysis: Top Product Chapters</link>', styles["TOCItem"]), Paragraph("18", styles["TOCPage"])],
        [Paragraph('<link href="#market_intelligence">Market Intelligence & Business Opportunities</link>', styles["TOCItem"]), Paragraph("19", styles["TOCPage"])],
        [Paragraph('<link href="#fastest_growing_sectors">Fastest-Growing Sectors</link>', styles["TOCItem"]), Paragraph("20", styles["TOCPage"])],
        [Paragraph('<link href="#canadian_demand_brazil">Canadian Demand from Brazil</link>', styles["TOCItem"]), Paragraph("21", styles["TOCPage"])],
        [Paragraph('<link href="#target_industries_outlook">Target Industries & Five-Year Outlook</link>', styles["TOCItem"]), Paragraph("22", styles["TOCPage"])],
        [Paragraph('<link href="#forecast_outlook">Forecast Validation and Baseline Outlook</link>', styles["TOCItem"]), Paragraph("23", styles["TOCPage"])],
        [Paragraph('<link href="#forecast_tables">2026 Monthly Projection Table</link>', styles["TOCItem"]), Paragraph("24", styles["TOCPage"])],
        [Paragraph('<link href="#appendix_forecast_method">Appendix Forecast Validation Method</link>', styles["TOCItem"]), Paragraph("25", styles["TOCPage"])],
        [Paragraph('<link href="#data_source">Data Source</link>', styles["TOCItem"]), Paragraph("26", styles["TOCPage"])],
        [Paragraph('<link href="#confidentiality">Confidentiality & Disclaimer</link>', styles["TOCItem"]), Paragraph("26", styles["TOCPage"])],
    ]
    tbl = Table(toc_rows, colWidths=[25.2 * cm, 3.4 * cm])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#EEF4EE")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.HexColor("#004D25")),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("BACKGROUND", (0, 1), (-1, -1), colors.white),
        ("TEXTCOLOR", (0, 0), (-1, -1), colors.HexColor("#333333")),
        ("FONTSIZE", (0, 0), (-1, -1), 9.8),
        ("BOX", (0, 0), (-1, -1), 0.5, colors.HexColor("#E0E0E0")),
        ("INNERGRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#EFEFEF")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (1, 0), (1, -1), "CENTER"),
        ("LEFTPADDING", (0, 0), (-1, -1), 8),
        ("RIGHTPADDING", (0, 0), (-1, -1), 8),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#FAFAFA")]),
    ]))
    return tbl

def make_chart_list_table(sections):
    rows = [
        [Paragraph("<b>Chart No.</b>", styles["ChartListItem"]), Paragraph("<b>Chart Title</b>", styles["ChartListItem"]), Paragraph("<b>Report Section</b>", styles["ChartListItem"])]
    ]
    for idx, sec in enumerate(sections, start=1):
        chart_no = sec.get("chart_no", f"Chart 2.{idx}")
        anchor = sec.get("anchor", "")
        link_open = f'<link href="#{anchor}">' if anchor else ""
        link_close = "</link>" if anchor else ""
        rows.append([
            Paragraph(f'{link_open}{chart_no}{link_close}', styles["ChartListItem"]),
            Paragraph(f'{link_open}{sec["chart_title"]}{link_close}', styles["ChartListItem"]),
            Paragraph(f'{link_open}{sec["title"]}{link_close}', styles["ChartListItem"]),
        ])

    market_chart_rows = [
        ("Chart 3.1", "Fastest-growing sectors by year-over-year growth", "fastest_growing_sectors", "Fastest-Growing Sectors"),
        ("Chart 3.2", "Canadian import demand from Brazil by business sector", "canadian_demand_brazil", "Canadian Demand from Brazil"),
        ("Chart 3.3", "Target industry opportunity matrix", "target_industries_outlook", "Target Industries & Five-Year Outlook"),
        ("Chart 4.1", "2026 baseline projection: monthly trade volume forecast", "forecast_outlook", "Forecast Validation and Baseline Outlook"),
        ("Chart 4.2", "2026 monthly projection table", "forecast_tables", "2026 Monthly Projection Table"),
    ]
    for chart_no, chart_title, anchor, section_title in market_chart_rows:
        rows.append([
            Paragraph(f'<link href="#{anchor}">{chart_no}</link>', styles["ChartListItem"]),
            Paragraph(f'<link href="#{anchor}">{chart_title}</link>', styles["ChartListItem"]),
            Paragraph(f'<link href="#{anchor}">{section_title}</link>', styles["ChartListItem"]),
        ])
    tbl = Table(rows, colWidths=[3.0 * cm, 13.2 * cm, 12.4 * cm])
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#EEF4EE")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.HexColor("#004D25")),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("BOX", (0, 0), (-1, -1), 0.5, colors.HexColor("#E0E0E0")),
        ("INNERGRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#EFEFEF")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 7),
        ("RIGHTPADDING", (0, 0), (-1, -1), 7),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#FAFAFA")]),
    ]))
    return tbl

def draw_footer(canvas, doc):
    page_num = canvas.getPageNumber()
    canvas.saveState()
    canvas.setStrokeColor(colors.HexColor("#D9D9D9"))
    canvas.setLineWidth(0.4)
    canvas.line(doc.leftMargin, 1.15 * cm, PAGE_WIDTH - doc.rightMargin, 1.15 * cm)
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(colors.HexColor("#777777"))
    canvas.drawString(doc.leftMargin, 0.72 * cm, "Confidential: Internal Use Only")
    canvas.drawRightString(PAGE_WIDTH - doc.rightMargin, 0.72 * cm, f"Page {page_num}")
    canvas.restoreState()

# -----------------------------------------------------------------------------
# 6. Story start: COVER PAGE
# -----------------------------------------------------------------------------
story = []

LOGO_PATH = walk_find_file([BASE_DIR, DATA_DIR, RAW_DIR, CLEAN_DIR, LEGACY_CLEAN_DIR, IMG_DIR, REPORTS_DIR], exact_names=["FCBB logo 3.png", "fcbb_logo.png", "logo.png"], suffix=".png")

story.append(Spacer(1, 1.2 * cm))
story.append(make_logo_block(LOGO_PATH, width_cm=4.8, height_cm=4.8))
story.append(Spacer(1, 1.0 * cm))

story.append(Paragraph(title_page_title, styles["TitleGreen"]))
story.append(Spacer(1, 0.25 * cm))
story.append(Paragraph(title_page_subtitle, styles["SubheadingGray"]))
story.append(Spacer(1, 0.55 * cm))
story.append(HRFlowable(width="80%", thickness=0.7, color=colors.HexColor("#D9E4D9"), spaceBefore=0.2 * cm, spaceAfter=0.35 * cm))

story.append(Paragraph("<b>Authors</b>", styles["CoverMeta"]))
story.append(Paragraph("Flavia Batista, Julio Carneiro, Cristiane Giacomazzi", styles["CoverAuthors"]))
story.append(Paragraph("Data Analytics / Executive Reporting Project", styles["CoverMeta"]))
story.append(Spacer(1, 0.45 * cm))

story.append(Paragraph(f"<b>Reporting period:</b> {YEAR1}–{YEAR2}", styles["CoverMeta"]))
story.append(Paragraph(f"<b>Generated on:</b> {datetime.now().strftime('%Y-%m-%d')}", styles["CoverMeta"]))

story.append(PageBreak())

# -----------------------------------------------------------------------------
# 7. EXECUTIVE SUMMARY PAGE
# -----------------------------------------------------------------------------
story.append(make_page_title("Executive Summary", anchor="exec_summary"))
story.append(Spacer(1, 0.45 * cm))

story.append(Paragraph(
    "Canada–Brazil merchandise trade reached CAD$14.5 billion in 2025, representing a 16.0% increase compared to 2024. "
    "Exports totaled CAD$2.9 billion, while imports reached CAD$11.6 billion, resulting in a trade balance of CAD$-8.7 billion. "
    "Imports accounted for CAD$11.6 billion of total volume, maintaining a trade deficit of CAD$-8.7 billion. "
    "Trade activity remains geographically concentrated, with Ontario accounting for the largest share of total volume, "
    "while product level performance is driven by a limited number of key categories contributing most of the growth. "
    "Overall, the structure of trade indicates continued dependence on specific provinces and product segments. "
    "Based on current trends, total trade is projected to reach approximately CAD$16.8 billion in 2026, "
    "assuming continued momentum in import activity and stable export performance.",
    styles["BodySummary"]
))

story.append(PageBreak())

# -----------------------------------------------------------------------------
# 8. KEY TRADE INDICATORS
# -----------------------------------------------------------------------------
story.append(make_page_title("Key Trade Indicators", anchor="key_metrics"))
story.append(Spacer(1, 0.45 * cm))

story.append(Paragraph(
    "Canada–Brazil trade expansion in 2025 was driven by higher import volumes, maintaining a persistent trade imbalance. "
    "Although export growth exceeded import growth in percentage terms, the absolute gap between imports and exports remained wide, "
    "resulting in a trade deficit of CAD$-8.7 billion. Trade activity remains geographically concentrated, with Ontario accounting for "
    "the largest share of total volume. The current trajectory implies continued expansion into 2026, "
    "with total trade projected near CAD$16.8 billion under existing conditions.",
    styles["BodySummary"]
))
story.append(Spacer(1, 0.28 * cm))

metric_strip = Table(
    [[
        Paragraph("<b>Total Trade</b><br/>CAD$14.5B", styles["KPIBox"]),
        Paragraph("<b>Growth</b><br/>+16.0%", styles["KPIBox"]),
        Paragraph("<b>Exports</b><br/>CAD$2.9B", styles["KPIBox"]),
        Paragraph("<b>Imports</b><br/>CAD$11.6B", styles["KPIBox"]),
        Paragraph("<b>Balance</b><br/>CAD$-8.7B", styles["KPIBox"]),
        Paragraph("<b>2026 Outlook</b><br/>CAD$16.8B", styles["KPIBox"]),
    ]],
    colWidths=[3.35 * cm, 3.05 * cm, 3.15 * cm, 3.15 * cm, 3.15 * cm, 3.35 * cm]
)

metric_strip.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, -1), colors.HexColor("#F7F9F7")),
    ("BOX", (0, 0), (-1, -1), 0.6, colors.HexColor("#DDE7DD")),
    ("INNERGRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#E4E4E4")),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("LEFTPADDING", (0, 0), (-1, -1), 6),
    ("RIGHTPADDING", (0, 0), (-1, -1), 6),
    ("TOPPADDING", (0, 0), (-1, -1), 12),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 12),
]))

story.append(metric_strip)
story.append(Spacer(1, 0.22 * cm))

story.append(Paragraph(
    "These indicators confirm that recent trade expansion has been accompanied by a sustained imbalance driven by higher import volumes, "
    "with the trade structure concentrated in Ontario and a limited set of product categories.",
    styles["BodySummary"]
))

story.append(PageBreak())

# -----------------------------------------------------------------------------
# 9. TABLE OF CONTENTS PAGE WITH INTERNAL LINKS AND PAGE REFERENCES
# -----------------------------------------------------------------------------
story.append(make_page_title("Table of Contents"))
story.append(Spacer(1, 0.38 * cm))
story.append(Paragraph("Use the links below to jump directly to each report section in the PDF.", styles["SubheadingGray"]))
story.append(Spacer(1, 0.20 * cm))
story.append(make_toc_table())
story.append(PageBreak())


# -----------------------------------------------------------------------------
# 10. Report sections
# -----------------------------------------------------------------------------
sections = [
    {
        "anchor": "trade_expansion_summary",
        "title": "Trade Expansion Summary",
        "chart_no": "Chart 2.1",
        "chart_title": "Canada–Brazil trade expansion summary",
        "chart_key": "chart_extra_trade_expansion_summary",
        "placeholder": (
            "Total Canada–Brazil trade expanded significantly in 2025, driven primarily by strong import growth. "
            "While exports also increased, imports remain structurally higher, reinforcing Canada's position as a net importer in the bilateral relationship. "
            "The expansion reflects increased demand and stronger trade flows rather than a shift in trade balance. "
            "Overall, growth indicates a larger and more active trade corridor, but with persistent imbalance driven by import dependency."
        )
    },
    {
        "anchor": "annual_trade",
        "title": "Annual Trade Overview",
        "chart_no": "Chart 2.2",
        "chart_title": "Canada exports and imports with Brazil, 2024–2025",
        "chart_key": "chart_q1_annual_trade",
        "placeholder": (
            "Canada–Brazil trade increased in 2025, with both exports and imports rising compared to 2024. "
            "Exports grew by +28.5% year-over-year, outpacing import growth of +13.2%, though imports remained significantly higher "
            "in absolute value at CAD$11.6 billion versus CAD$2.9 billion for exports. "
            "As a result, the overall trade relationship expanded while Canada maintained a net importer position. "
            "This indicates that growth improved total trade activity, but did not materially reduce the structural trade imbalance."
        )
    },
    {
        "anchor": "bilateral_volume",
        "title": "Bilateral Trade Volume Shift",
        "chart_no": "Chart 2.3",
        "chart_title": "Bilateral trade volume shift, 2024–2025",
        "chart_key": "chart_extra_bilateral_trade_volume",
        "placeholder": (
            "The bilateral trade corridor shifted to a higher absolute volume in 2025, indicating stronger commercial engagement between Canada and Brazil. "
            "This upward shift reflects increased economic activity and demand across traded goods. "
            "However, the expansion occurred without a structural correction in the trade balance, meaning that higher volume did not translate into improved export competitiveness. "
            "The relationship is scaling, but not rebalancing."
        )
    },
    {
        "anchor": "monthly_performance_comparative",
        "title": "Monthly Performance Comparative: Year-over-Year Change",
        "chart_no": "Chart 2.4",
        "chart_title": "Monthly trade performance: year-over-year change",
        "chart_key": "chart_extra_monthly_performance_comparative",
        "placeholder": (
            "Monthly performance shows periods of acceleration and moderation throughout the year, indicating moderate volatility in trade flows. "
            "Several months experienced stronger spikes in activity, suggesting seasonal or demand-driven surges rather than consistent linear growth. "
            "Despite fluctuations, the overall trend remains upward, with imports consistently exceeding exports. "
            "This pattern indicates that growth is present, but not evenly distributed across months."
        )
    },
    {
        "anchor": "monthly_flow",
        "title": "Monthly Trade Flow",
        "chart_no": "Chart 2.5",
        "chart_title": "Monthly exports and imports trade flow",
        "chart_key": "chart_q2_monthly_trade_flow",
        "placeholder": (
            "Monthly trade flows highlight a persistent and structural import surplus, with imports exceeding exports in every observed period. "
            "While both series fluctuate, the gap between imports and exports remains consistently wide, suggesting that the trade deficit is not seasonal but structural. "
            "This indicates long-term dependence on imported goods from Brazil rather than temporary imbalances."
        )
    },
    {
        "anchor": "trade_mix",
        "title": "Comparative Trade Mix",
        "chart_no": "Chart 2.6",
        "chart_title": "Comparative trade mix by exports and imports",
        "chart_key": "chart_q3_trade_mix_donut",
        "placeholder": (
            "The trade composition improved slightly in 2025, with exports gaining marginal share relative to imports. "
            "However, imports still dominate the overall mix, maintaining a significant imbalance. "
            "The improvement suggests a gradual strengthening of export performance, but not at a scale sufficient to materially shift the trade structure. "
            "The relationship remains import-heavy despite the marginal shift in composition."
        )
    },
    {
        "anchor": "product_perf",
        "title": "Diverging Product Momentum",
        "chart_no": "Chart 2.7",
        "chart_title": "Diverging product momentum: top winner Cocoa",
        "chart_key": "chart_q4_product_performance",
        "placeholder": (
            "Product level performance reveals a divergence between high growth and declining categories. "
            "A small number of product groups are driving most of the growth, while others are contracting or stagnating. "
            "This concentration indicates that trade expansion is not broad based, but dependent on specific categories. "
            "Such concentration reduces resilience to category-level disruptions and limits the breadth of the trade portfolio."
        )
    },
    {
        "anchor": "prov_volume",
        "title": "Provincial Trade Volume",
        "chart_no": "Chart 2.8",
        "chart_title": "Provincial trade volume ranking",
        "chart_key": "chart_q5_provincial_trade_volume",
        "placeholder": (
            "Trade activity is highly concentrated geographically, with one province accounting for a disproportionate share of total volume. "
            "This concentration highlights a dependency on a limited number of regional trade hubs. "
            "While this can create efficiency advantages, it also introduces risk by reducing diversification across provinces. "
            "The data points to geographic concentration that merits monitoring as trade volumes scale."
        )
    },
    {
        "anchor": "prov_map",
        "title": "Strategic Provincial Investment Map",
        "chart_no": "Chart 2.9",
        "chart_title": "Strategic provincial investment map",
        "chart_key": "chart_q7_strategic_provincial_map",
        "placeholder": (
            "The strategic map identifies provinces that combine scale and growth as the most attractive investment opportunities. "
            "Regions positioned in the high volume and high growth quadrant represent the strongest near term potential for expansion. "
            "Provinces with declining growth or smaller trade volumes present lower priority or require reassessment. "
            "This framework supports targeted, data driven allocation of trade and investment efforts."
        )
    },

    {
        "anchor": "prov_growth_polished",
        "title": "Province Growth & Decline",
        "chart_no": "Chart 2.10",
        "chart_title": "Provincial growth and decline by trade value",
        "chart_key": "chart_extra_province_growth_decline_polished",
        "placeholder": (
            "Provincial performance shows a clear divergence, with some regions expanding rapidly while others contract. "
            "This uneven growth pattern indicates that trade momentum is not uniformly distributed across the country. "
            "Leading provinces are capturing the majority of growth, while weaker regions lag behind, highlighting regional imbalances in trade participation and opportunity."
        )
    },
    {
        "anchor": "ontario_spotlight",
        "title": "Ontario Trade Hub Analysis",
        "chart_no": "Chart 2.11a",
        "chart_title": "Ontario trade position and key metrics, 2025",
        "chart_key": "chart_q8_ontario_spotlight",
        "placeholder": (
            "Ontario remains the dominant province in Canada–Brazil trade, contributing the largest share of total volume. "
            "Its performance reflects both scale and consistent participation across key product categories. "
            "However, this dominance also reinforces overall trade concentration risk, as national performance becomes heavily dependent on a single region. "
            "Ontario continues to be the central driver of bilateral trade activity."
        )
    },
    {
        "anchor": "ontario_top5",
        "title": "Ontario Trade Hub Analysis: Top Product Chapters",
        "chart_no": "Chart 2.11b",
        "chart_title": "Ontario top 5 product chapters by trade value, 2025",
        "chart_key": "chart_q8_ontario_top5_chapters",
        "placeholder": (
            "Ontario's trade profile is concentrated in a small group of high-value product chapters. "
            "The expanded view below improves readability and separates the product mix from the province-level performance summary."
        )
    },
    {
        "anchor": "forecast_outlook",
        "title": "Forecast Validation and Baseline Outlook",
        "chart_no": "Chart 4.1",
        "chart_title": "2026 baseline projection: monthly trade volume forecast",
        "chart_key": "chart_q9_forecast_validation",
        "placeholder": (
            "Forecast validation suggests that recent trade trends are stable enough to support a continued expansion scenario. "
            "Under current conditions, total trade is expected to grow further, driven primarily by sustained import activity. "
            "The model does not incorporate structural adjustment scenarios; the projection reflects current trend continuation. "
            "The forecast represents a baseline scenario based on observed trade patterns and should not be interpreted as a prediction of future policy, tariff, or macroeconomic changes. "
            f"The baseline extends observed {YEAR1}\u2013{YEAR2} trade patterns into {YEAR2 + 1}; validation method and error metrics are presented in the Appendix."
        )
    },
    {
        "anchor": "forecast_tables",
        "title": "2026 Monthly Projection Table",
        "chart_no": "Chart 4.2",
        "chart_title": "2026 monthly projection table",
        "chart_key": "chart_q9_forecast_tables",
        "placeholder": (
            "The table below presents the selected monthly baseline forecast with low and high scenario bounds. "
            "Technical validation details are included in the appendix so this page remains focused on the planning outlook."
        )
    },
]

# Final delivery order adjustment: keep core trade/province/product analysis first,
# move Market Intelligence before the Forecast section, and preserve all existing sections.
forecast_section_anchors = {"forecast_outlook", "forecast_tables"}
forecast_sections = [sec for sec in sections if sec.get("anchor") in forecast_section_anchors]
sections = [sec for sec in sections if sec.get("anchor") not in forecast_section_anchors]

print("\nBuilding PDF with the following sections:")
# -----------------------------------------------------------------------------
# LIST OF CHARTS: after sections are defined, before rendering loop
# -----------------------------------------------------------------------------
story.append(make_page_title("List of Charts", anchor="chart_list"))
story.append(Spacer(1, 0.45 * cm))
story.append(Paragraph("The chart numbers and names below correspond to the captions printed under each visual throughout the report.", styles["SubheadingGray"]))
story.append(Spacer(1, 0.16 * cm))
story.append(make_chart_list_table(sections))
story.append(PageBreak())

def render_pdf_sections(sections_to_render):
    for sec in sections_to_render:
        print(f"  - {sec['title']} (chart: {sec['chart_key']})")

        story.append(make_page_title(sec["title"], anchor=sec["anchor"]))

        if sec["placeholder"]:
            if sec["chart_key"] == "chart_extra_monthly_performance_comparative":
                story.append(Spacer(1, 0.24 * cm))
                story.append(Paragraph(sec["placeholder"], styles["ChartLeadText"]))
                story.append(Spacer(1, 0.22 * cm))
            elif sec["chart_key"] == "chart_q7_strategic_provincial_map":
                # Chart 2.9 only: compact vertical spacing so the chart and its caption remain on page 15.
                story.append(Spacer(1, 0.30 * cm))
                story.append(Paragraph(sec["placeholder"], styles["ChartLeadText"]))
                story.append(Spacer(1, 0.26 * cm))
            elif sec["chart_key"] == "chart_q9_forecast_tables":
                # Executive forecast table page: keep technical validation in the appendix.
                story.append(Spacer(1, 0.24 * cm))
                story.append(Paragraph(sec["placeholder"], styles["ChartLeadText"]))
                story.append(Spacer(1, 0.25 * cm))
            else:
                story.append(Spacer(1, 0.42 * cm))
                story.append(Paragraph(sec["placeholder"], styles["ChartLeadText"]))
                story.append(Spacer(1, 0.55 * cm))

        _chart_added = False
        if sec["chart_key"] in saved_chart_paths:
            chart_path = trim_chart_whitespace(saved_chart_paths[sec["chart_key"]])
            if Path(chart_path).exists():
                # Define tall_charts set before using it
                tall_charts = {"chart_q7_strategic_provincial_map"}

                # Properly structured if-elif chain for chart heights
                if sec["chart_key"] == "chart_q9_forecast_validation":
                    max_h = 12.35
                elif sec["chart_key"] == "chart_q9_forecast_tables":
                    max_h = 12.20
                elif sec["chart_key"] == "chart_extra_monthly_performance_comparative":
                                max_h = 11.95
                elif sec["chart_key"] in {"chart_q8_ontario_spotlight", "chart_q8_ontario_top5_chapters"}:
                    max_h = 13.10
                elif sec["chart_key"] == "chart_q7_strategic_provincial_map":
                    max_h = 12.15
                elif sec["chart_key"] == "chart_q3_trade_mix_donut":
                    max_h = 10.70
                elif sec["chart_key"] == "chart_q4_product_performance":
                    max_h = 12.20
                elif sec["chart_key"] == "chart_q5_provincial_trade_volume":
                    max_h = 12.20
                elif sec["chart_key"] == "chart_extra_province_growth_decline_polished":
                    max_h = 12.20
                elif sec["chart_key"] in tall_charts:
                    max_h = 12.10
                else:
                    max_h = 11.90

                max_w = CHART_IMAGE_WIDTH_CM
                img = make_chart_image(chart_path, max_width_cm=max_w, max_height_cm=max_h)
                chart_tbl = Table([[img]], colWidths=[REPORT_CONTENT_WIDTH_CM * cm], hAlign="CENTER")
                chart_tbl.setStyle(TableStyle([
                    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
                    ("VALIGN", (0, 0), (-1, -1), "TOP"),
                    ("LEFTPADDING", (0, 0), (-1, -1), 0),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 0),
                    ("TOPPADDING", (0, 0), (-1, -1), 0),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
                ]))
                story.append(chart_tbl)
                if sec["chart_key"] == "chart_q7_strategic_provincial_map":
                    story.append(Spacer(1, 0.02 * cm))
                else:
                    story.append(Spacer(1, 0.04 * cm))
                story.append(Paragraph(f"<b>{sec['chart_no']}:</b> {sec['chart_title']}", styles["ChartNumber"]))
                story.append(Spacer(1, 0.20 * cm))
                _chart_added = True
                print(f"    ✓ Added chart from {chart_path}")
            else:
                print(f"    ✗ Chart file not found: {chart_path}")
        else:
            print(f"    ✗ Warning: Chart key {sec['chart_key']} not in saved_chart_paths")

        if _chart_added or sec["placeholder"]:
            story.append(PageBreak())







# Render core report sections first; forecast is rendered after Market Intelligence.
render_pdf_sections(sections)

# -----------------------------------------------------------------------------
# 16. MARKET INTELLIGENCE & BUSINESS OPPORTUNITIES: CHART-BASED LAYER
# -----------------------------------------------------------------------------
from textwrap import wrap


def _market_pct_value(value):
    """Return percentage points for plotting from either ratio-style or percent-style inputs."""
    try:
        v = float(value)
        return v * 100 if abs(v) <= 5 else v
    except Exception:
        return 0.0


def _market_cad_b(value):
    """Return CAD billions for plotting."""
    try:
        return float(value) / 1_000_000_000
    except Exception:
        return 0.0


def _market_pct_label(value):
    try:
        return fmt_pct(value)
    except Exception:
        return f"{_market_pct_value(value):.1f}%"


def _market_value_label(value):
    try:
        return fmt_cad(value)
    except Exception:
        return str(value)


def _wrapped_labels(labels, width=26):
    return ["\n".join(wrap(str(label), width=width)) for label in labels]


def _empty_market_chart(title, subtitle):
    fig, ax = plt.subplots(figsize=(13.5, 6.6), dpi=220)
    fig.patch.set_facecolor("white")
    ax.axis("off")
    fig.text(0.5, 0.82, title, ha="center", va="center", fontsize=20, fontweight="bold", color=C_DKGREEN)
    fig.text(0.5, 0.74, subtitle, ha="center", va="center", fontsize=11, color="#555555")
    fig.text(0.5, 0.50, "No sufficient data available", ha="center", va="center", fontsize=16, color="#777777")
    return fig


def chart_market_fastest_growing_sectors():
    data = globals().get("fastest_growing_sectors", pd.DataFrame()).copy()
    if data is None or data.empty:
        return _empty_market_chart(
            "Fastest-Growing Sectors",
            "Sector momentum based on year-over-year trade growth."
        )

    data = data.head(6).copy()
    data["GrowthPct"] = data["YoY Growth"].apply(_market_pct_value)
    data = data.sort_values("GrowthPct", ascending=True)

    fig, ax = plt.subplots(figsize=(16.0, max(7.40, len(data) * 0.72)), dpi=220)
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)
    fig.subplots_adjust(top=0.780, bottom=0.085, left=0.300, right=0.925)

    labels = _wrapped_labels(data["Business_Sector"].tolist(), width=30)
    y = np.arange(len(data))
    vals = data["GrowthPct"].astype(float)
    top_idx = vals.idxmax()
    colors = [C_DKGREEN if idx == top_idx else C_LTGREEN for idx in data.index]
    bars = ax.barh(y, vals, color=colors, edgecolor=C_WHITE, linewidth=1.0, height=0.80)

    max_x = max(float(vals.max()), 1.0)
    ax.set_xlim(0, max_x * 1.32)

    for i, (bar, (_, row)) in enumerate(zip(bars, data.iterrows())):
        x = float(row["GrowthPct"])
        value_label = _market_value_label(row.get(YEAR2, 0))
        is_top = row.name == top_idx
        ax.text(
            x + max_x * 0.025,
            bar.get_y() + bar.get_height() / 2,
            f"+{x:.1f}%  |  {value_label}",
            va="center", ha="left", fontsize=12.3,
            color=C_DKGREEN if is_top else C_DKGRAY,
            fontweight="bold" if is_top else "normal",
            clip_on=False
        )

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=13.2, color=C_DKGRAY)
    for lbl, (_, row) in zip(ax.get_yticklabels(), data.iterrows()):
        if row.name == top_idx:
            lbl.set_fontweight("bold")
            lbl.set_color(C_DKGREEN)

    for s in ["top", "right", "left", "bottom"]:
        ax.spines[s].set_visible(False)
    ax.tick_params(axis="x", bottom=False, labelbottom=False)
    ax.tick_params(axis="y", left=False)
    ax.grid(False)

    top_row = data.loc[top_idx]
    add_polished_chart_header(
        fig,
        "Fastest-Growing Sectors",
        subtitle_parts=[
            ("Business sectors ranked by year-over-year growth; strongest momentum: ", C_DKGRAY, "normal"),
            (str(top_row.get("Business_Sector", "top sector")), C_DKGREEN, "bold"),
            (f" at +{float(top_row['GrowthPct']):.1f}%.", C_DKGRAY, "normal"),
        ]
    )
    return fig


def chart_market_canadian_demand():
    data = globals().get("top_canadian_demand_sectors", pd.DataFrame()).copy()
    if data is None or data.empty:
        return _empty_market_chart(
            "Canadian Demand from Brazil",
            "Import-led demand based on 2025 import value."
        )

    data = data.head(6).copy()
    data["ImportCADB"] = data[YEAR2].apply(_market_cad_b)
    data = data.sort_values("ImportCADB", ascending=True)

    fig, ax = plt.subplots(figsize=(16.0, max(7.40, len(data) * 0.72)), dpi=220)
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)
    fig.subplots_adjust(top=0.780, bottom=0.085, left=0.300, right=0.925)

    labels = _wrapped_labels(data["Business_Sector"].tolist(), width=30)
    y = np.arange(len(data))
    vals = data["ImportCADB"].astype(float)
    top_idx = vals.idxmax()
    colors = [C_DKGREEN if idx == top_idx else C_LTGREEN for idx in data.index]
    bars = ax.barh(y, vals, color=colors, edgecolor=C_WHITE, linewidth=1.0, height=0.80)

    max_x = max(float(vals.max()), 0.1)
    ax.set_xlim(0, max_x * 1.34)

    for bar, (_, row) in zip(bars, data.iterrows()):
        x = float(row["ImportCADB"])
        growth = _market_pct_label(row.get("YoY Growth", 0))
        is_top = row.name == top_idx
        ax.text(
            x + max_x * 0.025,
            bar.get_y() + bar.get_height() / 2,
            f"CAD${x:.2f}B  |  {growth} growth",
            va="center", ha="left", fontsize=12.3,
            color=C_DKGREEN if is_top else C_DKGRAY,
            fontweight="bold" if is_top else "normal",
            clip_on=False
        )

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=13.2, color=C_DKGRAY)
    for lbl, (_, row) in zip(ax.get_yticklabels(), data.iterrows()):
        if row.name == top_idx:
            lbl.set_fontweight("bold")
            lbl.set_color(C_DKGREEN)

    for s in ["top", "right", "left", "bottom"]:
        ax.spines[s].set_visible(False)
    ax.tick_params(axis="x", bottom=False, labelbottom=False)
    ax.tick_params(axis="y", left=False)
    ax.grid(False)

    top_row = data.loc[top_idx]
    add_polished_chart_header(
        fig,
        "Canadian Demand from Brazil",
        subtitle_parts=[
            ("Import sectors ranked by value; largest demand pool: ", C_DKGRAY, "normal"),
            (str(top_row.get("Business_Sector", "top sector")), C_DKGREEN, "bold"),
            (f" at CAD${float(top_row['ImportCADB']):.2f}B.", C_DKGRAY, "normal"),
        ]
    )
    return fig


def chart_market_opportunity_matrix():
    data = globals().get("top_sector_opportunities", pd.DataFrame()).copy()
    if data is None or data.empty:
        return _empty_market_chart(
            "Target Industries & Five-Year Outlook",
            "Opportunity matrix based on trade scale and growth."
        )

    data = data.head(7).copy()
    data["TradeCADB"] = data[YEAR2].apply(_market_cad_b)
    data["GrowthPct"] = data["YoY Growth"].apply(_market_pct_value)
    denom = max(float(data["Absolute Change"].abs().max()), 1.0)
    data["BubbleSize"] = (data["Absolute Change"].abs() / denom) * 1050 + 170

    fig, ax = plt.subplots(figsize=(16.0, 7.55), dpi=220)
    fig.patch.set_facecolor(C_WHITE)
    ax.set_facecolor(C_WHITE)
    fig.subplots_adjust(top=0.780, bottom=0.125, left=0.105, right=0.885)

    ax.scatter(
        data["TradeCADB"], data["GrowthPct"], s=data["BubbleSize"], alpha=0.80,
        color=C_DKGREEN, edgecolor=C_WHITE, linewidth=1.2, zorder=3
    )

    x_max = max(float(data["TradeCADB"].max()), 0.10)
    x_min = min(float(data["TradeCADB"].min()), 0.0)
    y_max = max(float(data["GrowthPct"].max()), 1.0)
    y_min = min(float(data["GrowthPct"].min()), 0.0)
    ax.set_xlim(max(0, x_min - x_max * 0.05), x_max * 1.40)
    ax.set_ylim(y_min - max(abs(y_min), y_max) * 0.14 - 4, y_max * 1.18 + 4)

    # Executive-style quadrant guide, intentionally light to match Chart 2.9 without adding grid clutter.
    x_mid = data["TradeCADB"].median()
    y_mid = 0
    ax.axhline(y_mid, color="#D7D7D7", linewidth=0.9, zorder=1)
    ax.axvline(x_mid, color="#E0E0E0", linewidth=0.8, zorder=1)

    # Keep labels inside the plotting area so the large Precious Metals bubble does not cut off.
    custom_offsets = {
        "Precious Metals and Jewelry": (-16, 12, "right"),
        "Precious Stones, Metals and Jewelry": (-16, 12, "right"),
        "Precious Stones, Metals and Jewellery": (-16, 12, "right"),
    }
    for _, row in data.iterrows():
        name = str(row["Business_Sector"])
        short_name = "\n".join(wrap(name, width=19))
        if name in custom_offsets:
            dx, dy, ha = custom_offsets[name]
        elif row["TradeCADB"] > x_max * 0.70:
            dx, dy, ha = -14, 10, "right"
        else:
            dx, dy, ha = 10, 8, "left"
        ax.annotate(
            short_name,
            (row["TradeCADB"], row["GrowthPct"]),
            xytext=(dx, dy), textcoords="offset points",
            fontsize=10.4, color=C_DKGRAY, ha=ha, va="center",
            clip_on=False, zorder=5,
            bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.82)
        )

    ax.set_xlabel(f"{YEAR2} trade value (CAD$ billions)", fontsize=12.0, color=C_DKGRAY, labelpad=8)
    ax.set_ylabel("Year-over-year growth (%)", fontsize=12.0, color=C_DKGRAY, labelpad=8)
    ax.tick_params(axis="both", labelsize=10.5, colors=C_DKGRAY)
    ax.grid(False)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color("#DADADA")
    ax.spines["bottom"].set_color("#DADADA")

    leader = data.sort_values(["GrowthPct", "TradeCADB"], ascending=[False, False]).iloc[0]
    add_polished_chart_header(
        fig,
        "Target Industries & Five-Year Outlook",
        subtitle_parts=[
            ("Opportunity matrix by trade scale and growth; strongest visible momentum: ", C_DKGRAY, "normal"),
            (str(leader.get("Business_Sector", "top sector")), C_DKGREEN, "bold"),
            (f" at +{float(leader['GrowthPct']):.1f}%.", C_DKGRAY, "normal"),
        ]
    )
    return fig


def _sector_examples_text(sector_name):
    try:
        subset = product_sector_bridge[product_sector_bridge["Business_Sector"].eq(sector_name)]
        col = "Business_Chapter" if "Business_Chapter" in subset.columns else "Chapter_Name"
        examples = subset[col].dropna().astype(str).head(3).tolist()
        return ", ".join(examples) if examples else "selected product chapters"
    except Exception:
        return "selected product chapters"


# Generate and save polished market charts.
market_chart_registry = {
    "chart_market_fastest_growing_sectors": chart_market_fastest_growing_sectors(),
    "chart_market_canadian_demand": chart_market_canadian_demand(),
    "chart_market_opportunity_matrix": chart_market_opportunity_matrix(),
}

for key, fig in market_chart_registry.items():
    saved_chart_paths[key] = save_fig(fig, key)
    plt.close(fig)
    print(f"  ✓ Saved market chart: {key}")

# Overview page
story.append(make_page_title("Market Intelligence & Business Opportunities", anchor="market_intelligence"))
story.append(Spacer(1, 0.42 * cm))
story.append(Paragraph(
    globals().get(
        "market_intelligence_summary",
        "This section translates trade data into practical business opportunities for entrepreneurs, exporters, investors, and ecosystem partners. "
        "It highlights where demand is growing, which sectors show momentum, and where FCBB can focus future market development efforts."
    ),
    styles["BodySummary"]
))
story.append(Spacer(1, 0.35 * cm))
story.append(Paragraph("<b>Business questions addressed</b>", styles["SubheadingGreen"]))
story.append(Spacer(1, 0.14 * cm))
for _q in [
    "Which sectors are growing most rapidly?",
    "Which Canadian sectors are buying more from Brazil?",
    "Which companies or sectors should be looking at these opportunities?",
]:
    story.append(Paragraph(f"• {_q}", styles["BodySummary"]))
story.append(Spacer(1, 0.35 * cm))
story.append(Paragraph(
    "Method note: the market layer uses <b>section_name</b> as the business sector layer and <b>chapter_name</b> as the product opportunity layer. "
    "This keeps the analysis traceable to the cleaned Statistics Canada dataset while making the findings easier for executives and investors to interpret.",
    styles["BodySmall"]
))
story.append(PageBreak())

# Fastest-growing sectors: chart page
story.append(make_page_title("Fastest-Growing Sectors", anchor="fastest_growing_sectors"))
story.append(Spacer(1, 0.20 * cm))
story.append(Paragraph(
    "This chart shows which business sectors are growing fastest, helping entrepreneurs quickly identify where market momentum is strongest.",
    styles["ChartLeadText"]
))
story.append(Spacer(1, 0.10 * cm))
story.append(make_chart_image(saved_chart_paths["chart_market_fastest_growing_sectors"], max_width_cm=25.6, max_height_cm=12.0))
story.append(Spacer(1, 0.04 * cm))
story.append(Paragraph("<b>Chart 3.1:</b> Fastest-growing sectors by year-over-year growth", styles["ChartNumber"]))
story.append(Spacer(1, 0.08 * cm))
story.append(Paragraph(
    "Recent reporting and official trade data point to strong Brazilian exports to Canada in gold, minerals, agricultural products, and industrial goods.",
    styles["BodySummary"]
))
story.append(Spacer(1, 0.05 * cm))
story.append(Paragraph(
    "<b>Sources:</b> "
    "<link href=\"https://www.infomoney.com.br/economia/brasil-bate-recorde-de-exportacoes-ao-canada-em-2025-com-alta-de-15/\" color=\"#1F4E79\">InfoMoney</link>"
    "&nbsp;&nbsp;&#8226;&nbsp;&nbsp;<link href=\"https://exame.com/esferabrasil/commodities-impulsionam-exportacoes-para-o-canada-e-saldo-comercial-brasileiro-cresce-93/\" color=\"#1F4E79\">Exame</link>"
    "&nbsp;&nbsp;&#8226;&nbsp;&nbsp;<link href=\"https://www.gov.br/mre/pt-br/embaixada-ottawa/noticias/brasil-bate-recorde-historico-de-exportacoes-para-o-canada-no-primeiro-semestre-de-2025\" color=\"#1F4E79\">MRE (Gov.br)</link>"
    "&nbsp;&nbsp;&#8226;&nbsp;&nbsp;<link href=\"https://www.gov.br/mme/pt-br/assuntos/noticias/2023-2026/brasil-e-canada-avancam-em-parceria-estrategica-nos-setores-de-energia-e-mineracao\" color=\"#1F4E79\">MME (Gov.br)</link>"
    "&nbsp;&nbsp;&#8226;&nbsp;&nbsp;<link href=\"https://www.ccbc.org.br/en/publicacoes/news-ccbc/brazil-breaks-export-record-to-canada-in-the-first-half-of-2025/\" color=\"#1F4E79\">CCBC</link>",
    styles["SourceLinks"]
))
story.append(PageBreak())

# Canadian demand: chart page
story.append(make_page_title("Canadian Demand from Brazil", anchor="canadian_demand_brazil"))
story.append(Spacer(1, 0.20 * cm))
story.append(Paragraph(
    "This chart focuses on Canadian imports from Brazil, showing where buyer demand is already active and easier to translate into commercial conversations.",
    styles["ChartLeadText"]
))
story.append(Spacer(1, 0.10 * cm))
story.append(make_chart_image(saved_chart_paths["chart_market_canadian_demand"], max_width_cm=25.6, max_height_cm=12.0))
story.append(Spacer(1, 0.04 * cm))
story.append(Paragraph("<b>Chart 3.2:</b> Canadian import demand from Brazil by business sector", styles["ChartNumber"]))
story.append(Spacer(1, 0.08 * cm))
story.append(Paragraph(
    "Executive reading: import-led demand is especially relevant for Brazilian exporters, Canadian distributors, and entrepreneurs looking for proven demand signals.",
    styles["BodySummary"]
))
story.append(Spacer(1, 0.05 * cm))
story.append(Paragraph(
    "<b>Sources:</b> "
    "<link href=\"https://www.international.gc.ca/country-pays/brazil-bresil/relations.aspx?lang=eng\" color=\"#1F4E79\">Global Affairs Canada</link>"
    "&nbsp;&nbsp;&#8226;&nbsp;&nbsp;<link href=\"https://www.infomoney.com.br/mundo/canada-e-mercosul-retomam-negociacoes-para-acordo-de-livre-comercio-ate-2026-diz-ft/\" color=\"#1F4E79\">InfoMoney</link>"
    "&nbsp;&nbsp;&#8226;&nbsp;&nbsp;<link href=\"https://www.gov.br/mdic/pt-br/assuntos/noticias/2025/outubro/mercosul-e-canada-retomam-negociacoes-comerciais\" color=\"#1F4E79\">MDIC (Gov.br)</link>"
    "&nbsp;&nbsp;&#8226;&nbsp;&nbsp;<link href=\"https://g1.globo.com/google/amp/economia/noticia/2026/05/30/mercosul-e-canada-concluem-nova-rodada-de-negociacao-para-acordo-comercial.ghtml\" color=\"#1F4E79\">G1</link>"
    "&nbsp;&nbsp;&#8226;&nbsp;&nbsp;<link href=\"https://noticias.r7.com/economia/mercosul-retoma-negociacoes-com-o-canada-entenda-a-importancia-do-acordo-10102025/\" color=\"#1F4E79\">R7</link>",
    styles["SourceLinks"]
))
story.append(PageBreak())

# Target industries and five-year outlook: chart page
story.append(make_page_title("Target Industries & Five-Year Outlook", anchor="target_industries_outlook"))
story.append(Spacer(1, 0.16 * cm))
story.append(Paragraph(
    "The opportunity matrix separates large markets from fast-growing niches, helping entrepreneurs prioritize where to validate demand, map buyers, and focus outreach over the next five years.",
    styles["ChartLeadText"]
))
story.append(Spacer(1, 0.10 * cm))
story.append(make_chart_image(saved_chart_paths["chart_market_opportunity_matrix"], max_width_cm=25.2, max_height_cm=10.95))
story.append(Spacer(1, 0.04 * cm))
story.append(Paragraph("<b>Chart 3.3:</b> Target industry opportunity matrix", styles["ChartNumber"]))
story.append(Spacer(1, 0.07 * cm))
story.append(Paragraph(
    "<b>Five-year opportunity lens:</b> Priority sectors should move into company mapping, buyer discovery, stakeholder interviews, and external market validation before any investment recommendation is made.",
    styles["BodySummary"]
))
story.append(PageBreak())

# -----------------------------------------------------------------------------
# 16. DATA SOURCE
# -----------------------------------------------------------------------------
# Forecast section rendered last, after Market Intelligence and business opportunity pages.
render_pdf_sections(forecast_sections)


# -----------------------------------------------------------------------------
# 16B. APPENDIX FORECAST VALIDATION METHOD
# -----------------------------------------------------------------------------
story.append(make_page_title("Appendix Forecast Validation Method", anchor="appendix_forecast_method"))
story.append(Spacer(1, 0.32 * cm))
story.append(Paragraph(
    "This appendix provides a simplified validation note for transparency. "
    "The main forecast section presents the business outlook and monthly projection table; this page explains how the baseline method was selected without interrupting the executive narrative.",
    styles["BodySummary"]
))
story.append(Spacer(1, 0.24 * cm))

# Recreate forecast validation metrics for the appendix table.
try:
    def _appendix_smape(y_true, y_pred):
        y_true = np.array(y_true, dtype=float)
        y_pred = np.array(y_pred, dtype=float)
        denom = np.where(np.abs(y_true) + np.abs(y_pred) == 0, 1e-9, np.abs(y_true) + np.abs(y_pred))
        return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)

    _df_fc = monthly.copy().sort_values("Period").reset_index(drop=True)
    _df_fc["Period"] = pd.to_datetime(_df_fc["Period"])
    _df_fc["Year"] = _df_fc["Period"].dt.year
    _df_fc["Month_Num"] = _df_fc["Period"].dt.month
    _df_fc["Value_B"] = _df_fc["total"] / 1e9
    _df_fc = _df_fc[_df_fc["Year"].isin([YEAR1, YEAR2])].copy()
    _pivot = _df_fc.pivot_table(index="Month_Num", columns="Year", values="Value_B", aggfunc="sum").sort_index()
    _months = _pivot.dropna(subset=[YEAR1, YEAR2]).index.tolist()
    _hist_y1 = _pivot.loc[_months, YEAR1].values
    _actual_y2 = _pivot.loc[_months, YEAR2].values
    _annual_growth = _actual_y2.sum() / _hist_y1.sum()
    _pred_seasonal = _hist_y1 * _annual_growth
    _pred_repeat = _hist_y1.copy()
    _ratios = np.where(_hist_y1 > 0, _actual_y2 / _hist_y1, np.nan)
    _robust_growth = np.nanmedian(_ratios)
    _pred_robust = _hist_y1 * _robust_growth

    _candidates = {
        "Seasonally Adjusted YoY Baseline": _pred_seasonal,
        "Robust Monthly Growth Baseline": _pred_robust,
        "Flat Seasonal Repeat": _pred_repeat,
    }
    _metric_rows = []
    for _name, _pred in _candidates.items():
        _metric_rows.append([
            _name,
            f"{mean_absolute_percentage_error(_actual_y2, _pred) * 100:.1f}%",
            f"{_appendix_smape(_actual_y2, _pred):.1f}%",
            f"{mean_absolute_error(_actual_y2, _pred):.2f}",
        ])

    _metric_rows_sorted = sorted(_metric_rows, key=lambda x: (float(x[1].replace("%","")), float(x[2].replace("%",""))))
    _metric_table_rows = [[
        Paragraph("Method", styles["TableHeaderWhite"]),
        Paragraph("MAPE", styles["TableHeaderWhite"]),
        Paragraph("sMAPE", styles["TableHeaderWhite"]),
        Paragraph("MAE (CAD$B)", styles["TableHeaderWhite"]),
        Paragraph("Use", styles["TableHeaderWhite"]),
    ]]
    for _i, _row in enumerate(_metric_rows_sorted):
        _status = "Selected baseline" if _i == 0 else "Reference check"
        _metric_table_rows.append([Paragraph(str(_row[0]), styles["BodySmall"]),
                                   Paragraph(str(_row[1]), styles["BodySmall"]),
                                   Paragraph(str(_row[2]), styles["BodySmall"]),
                                   Paragraph(str(_row[3]), styles["BodySmall"]),
                                   Paragraph(_status, styles["BodySmall"])])

    _tbl = Table(_metric_table_rows, colWidths=[9.5 * cm, 3.5 * cm, 3.5 * cm, 4.0 * cm, 6.2 * cm])
    _tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#004D25")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F7FBF7")]),
        ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#CCDDCC")),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("LEFTPADDING", (0, 0), (-1, -1), 6),
        ("RIGHTPADDING", (0, 0), (-1, -1), 6),
    ]))
    story.append(Paragraph("<b>Forecast Method Summary</b>", styles["SubheadingGreen"]))
    story.append(Spacer(1, 0.14 * cm))
    story.append(_tbl)
except Exception as _e:
    story.append(Paragraph(f"Forecast validation appendix could not be generated automatically: {_e}", styles["BodySmall"]))

story.append(Spacer(1, 0.30 * cm))
story.append(Paragraph("<b>Method Definitions</b>", styles["SubheadingGreen"]))
story.append(Spacer(1, 0.12 * cm))
_def_rows = [
    [Paragraph("<b>Metric</b>", styles["BodySmall"]), Paragraph("<b>Executive meaning</b>", styles["BodySmall"])],
    [Paragraph("<b>MAE</b>", styles["BodySmall"]), Paragraph("Average forecast miss in CAD$ billions. Lower values indicate closer historical fit.", styles["BodySmall"])],
    [Paragraph("<b>MAPE</b>", styles["BodySmall"]), Paragraph("Average forecast miss as a percentage of actual trade volume. Lower values indicate stronger fit.", styles["BodySmall"])],
    [Paragraph("<b>sMAPE</b>", styles["BodySmall"]), Paragraph("Balanced percentage error measure that treats over- and under-estimation consistently. Lower values indicate stronger fit.", styles["BodySmall"])],
]
_def_tbl = Table(_def_rows, colWidths=[4.0 * cm, (REPORT_CONTENT_WIDTH_CM - 4.0) * cm])
_def_tbl.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#EEF4EE")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.HexColor("#004D25")),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#CCDDCC")),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
    ("TOPPADDING", (0, 0), (-1, -1), 5),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ("LEFTPADDING", (0, 0), (-1, -1), 6),
    ("RIGHTPADDING", (0, 0), (-1, -1), 6),
]))
story.append(_def_tbl)
story.append(Spacer(1, 0.22 * cm))
story.append(Paragraph(
    "Business interpretation: this is a planning baseline. It extends observed trade patterns and does not include policy changes, tariff shocks, company-level decisions, or macroeconomic scenarios.",
    styles["BodySummary"]
))
story.append(PageBreak())

story.append(make_page_title("Data Source", anchor="data_source"))
story.append(Spacer(1, 0.45 * cm))

story.append(Paragraph(
    "This report is based on publicly available data from Statistics Canada "
    "(Canadian International Merchandise Trade Database), covering monthly "
    "import and export activity between Canada and Brazil.",
    styles["BodySummary"]
))

story.append(Spacer(1, SECTION_SPACER_CM * cm))

story.append(Paragraph(
    "The dataset includes trade flows from January 2024 to December 2025 and "
    "supports analysis of trade performance, product trends, and provincial distribution.",
    styles["BodySummary"]
))

story.append(Spacer(1, 0.20 * cm))

story.append(Paragraph(
    "Data is sourced from official government records to ensure reliability and consistency.",
    styles["BodySummary"]
))

story.append(Spacer(1, SECTION_SPACER_CM * cm))

story.append(Paragraph(
    "<b>Reference Source</b><br/>"
    "Statistics Canada. <i>Canadian International Merchandise Trade Web Application</i>.<br/>"
    "https://www150.statcan.gc.ca/n1/pub/71-607-x/71-607-x2021004-eng.htm<br/>"
    "Accessed: 2026-04-28",
    styles["BodySmall"]
))

# -----------------------------------------------------------------------------
# 17. CONFIDENTIAL DISCLAIMER
# -----------------------------------------------------------------------------
story.append(Spacer(1, 0.35 * cm))
story.append(make_page_title("Confidentiality & Disclaimer", anchor="confidentiality"))
story.append(Spacer(1, 0.30 * cm))

story.append(Paragraph(
    "<b>Confidential Document</b><br/>"
    "This report is intended for internal use only. "
    "It is based on publicly available data and analytical interpretation.",
    styles["BodySummary"]
))

story.append(Spacer(1, 0.20 * cm))

story.append(Paragraph(
    "All insights and projections are derived from historical trends and should "
    "be interpreted as directional analysis rather than precise forecasts.",
    styles["BodySummary"]
))

story.append(Spacer(1, 0.20 * cm))

story.append(Spacer(1, 0.30 * cm))

story.append(Paragraph(
    "© 2026: Internal Analytical Report",
    styles["SmallGray"]
))

# -----------------------------------------------------------------------------
# 17. Build PDF
# -----------------------------------------------------------------------------
print(f"\nGenerating PDF at: {PDF_OUTPUT}")
doc.build(
    story,
    onFirstPage=draw_footer,
    onLaterPages=draw_footer
)

# Create/update the stable report used by the README.
# Result:
# 1) Dated archive: canada_brazil_trade_report_jun_25_2026.pdf
# 2) Stable latest: canada_brazil_trade_report.pdf
shutil.copy2(PDF_OUTPUT, PDF_OUTPUT_LATEST)

print(f"\n✅ Archived PDF generated successfully: {PDF_OUTPUT}")
print(f"✅ Latest README PDF updated: {PDF_OUTPUT_LATEST}")
print(f"PDF file size: {PDF_OUTPUT.stat().st_size / 1024:.1f} KB")

Generating all charts...
Generated 15 charts successfully
  ✓ Saved: chart_q1_annual_trade
  ✓ Saved: chart_q2_monthly_trade_flow
  ✓ Saved: chart_q3_trade_mix_donut
  ✓ Saved: chart_q4_product_performance
  ✓ Saved: chart_q5_provincial_trade_volume
  ✓ Saved: chart_q6_provincial_growth_decline
  ✓ Saved: chart_q7_strategic_provincial_map
  ✓ Saved: chart_q8_ontario_spotlight
  ✓ Saved: chart_q8_ontario_top5_chapters
  ✓ Saved: chart_q9_forecast_validation
  ✓ Saved: chart_q9_forecast_tables
  ✓ Saved: chart_extra_bilateral_trade_volume
  ✓ Saved: chart_extra_trade_expansion_summary
  ✓ Saved: chart_extra_monthly_performance_comparative
  ✓ Saved: chart_extra_province_growth_decline_polished

Building PDF with the following sections:
  - Trade Expansion Summary (chart: chart_extra_trade_expansion_summary)
    ✓ Added chart from /Users/Julio/Library/Mobile Documents/com~apple~CloudDocs/data-analytics-projects/canada-brazil-trade-report/reports/report_images/chart_extra_trade_expansion_s